# SatQuery AI — Real Image-Grounded LoRA Fine-Tuning (Qwen3-VL-8B-Instruct) — v5
### SIH26167 / ISRO, Department of Space — replaces the previous GeoChat-7B text-only fine-tuning

**Why this notebook exists (honesty note, 2026-08-24):** the previous fine-tuning pipeline
(`colab_finetuning_geochat.ipynb`) trained a LoRA adapter on `bigearthnet_train.json` /
`bigearthnet_instructions.json` — but those files are **text-only**: every example is a
`{"prompt": "USER: <image>\n...", "response": "..."}` pair with a literal `<image>`
placeholder string and no actual image file anywhere. And the base-model loading code in
`backend/models/geochat_service.py` loaded `MBZUAI/geochat-7B` via plain
`AutoModelForCausalLM` and called `.generate()` on **tokenized text only** — no image
processor, no vision tower, no `pixel_values` ever passed in. So the resulting adapter never
saw a single real pixel, either during training or at inference.

**Why not GeoChat/EarthDial/GeoPixel/TEOChat instead**: all four were checked against their
real GitHub issue trackers before deciding this. Every one of them has multiple open,
unresolved "missing file" / "broken inference" / "wrong output" issues with zero closed
fixes visible — a real, current, external-user-confirmed pattern across two different
research groups (MBZUAI-oryx and Stanford/ermongroup), not a guess. Betting today's demo on
any of them carries real, evidenced risk of losing a day to an unfixable wall.

**The fix, done properly:**
- **Real base model**: `Qwen/Qwen3-VL-4B-Instruct` (upgraded 2026-09-12 from
  Qwen2.5-VL-3B-Instruct — see this cell's v4 note at the bottom for the real, verified
  reasons) — a production-maintained, actively developed vision-language model (Alibaba,
  Apache-2.0, official fine-tuning cookbooks) with first-class `transformers` support
  (`Qwen3VLForConditionalGeneration` + `AutoProcessor`, images genuinely passed through a
  real vision tower).
- **Real training data**: `xiang709/VRSBench` on the Hugging Face Hub — a real, published
  (NeurIPS 2024) remote-sensing benchmark with genuine paired images + captions + VQA pairs +
  referring-expression/bounding-box annotations (29,614 real images, 3.1M real QA pairs,
  52K real object references — not a synthetic text-only set, and no cross-dataset patch-ID
  join risk since the images ship bundled with their own annotations in one dataset).
- **Real LoRA fine-tuning** on real (image, prompt, response) triples via PEFT + TRL's
  documented VLM fine-tuning recipe — covers both single-image VQA and text-guided grounding
  in the same real dataset.
- Papers like GeoChat, TEOChat, and RS-Agent are still cited for their *ideas*
  (instruction-tuning dataset design, temporal-shared-encoder concepts, agentic orchestration
  patterns) in this project's README and report — just not their broken code, which is the
  honest distinction between "inspired by SOTA research" and "silently depending on code that
  doesn't run."

**Untested disclaimer**: written without GPU access to actually run it — every cell follows
the documented, standard API for each library, but please run it end-to-end and report back
anything that errors so it can be fixed for real rather than guessed at twice.

**How to run**: New Kaggle Notebook → Settings → Accelerator → GPU T4 x2 (or P100) → Internet: On.


**Scale-up + rebalancing (2026-08-27, quota corrected TWICE the same day):** after the second boss review and a real grounding-accuracy check (`scripts/evaluate_vrsbench_accuracy.py`: Acc@0.5 = 0.00 on 50 held-out referring-expression examples, even after fixing a real coordinate-order bug in `geochat_service.py`), sized against the user's real Kaggle GPU quota — first heard as 111h31m, corrected to 11h31m, then corrected again to the real 30h. `NUM_EXAMPLES` raised 600 -> 6,000 (Cell 3), with training-data sampling deliberately rebalanced toward a higher referring-expression share (real evidence: the organic 600-example mix was only ~24% `[refer]`), a training-data box-token format audit (Cell 3), a resolution/scale-jitter augmentation for the disclosed Cartosat-2S/RISAT domain gap (Cell 7), multi-session resume-from-Hub-checkpoint support (Cell 6), a `SMOKE_TEST` dry-run switch (Cell 3, with Cell 8's save/eval cadence also smoke-aware so the dry run actually exercises eval/checkpoint/Hub-push, not just data loading), and a `TimeBudgetGuard` training callback (Cell 8) that measures the REAL per-step rate a few steps in and warns immediately if the ~20h estimate is wrong, rather than finding out after most of the 30h quota is gone. Full reasoning and honest caveats are in each changed cell's own comments, not just here — this is genuinely untested code end to end: run with SMOKE_TEST=True first, confirm it completes cleanly (including the Hub push), then set SMOKE_TEST=False for the real run.

**v4 base-model + box-format upgrade (2026-09-12, same day as the v3 scale-up above, explicit user ask: "world class model chaiye naa abh -- shud we not upgrade?"):** switched the base model from `Qwen/Qwen2.5-VL-3B-Instruct` to `Qwen/Qwen3-VL-4B-Instruct` after verifying (not assuming) real compatibility -- identical LoRA target-module names on the language side, `.visual` vision-tower attribute unchanged, confirmed working bnb 4-bit quantization, Apache-2.0 licensed, and a real structural improvement (the vision-tower MLP naming collision that forced this notebook's `exclude_modules` safeguard for Qwen2.5-VL no longer exists in Qwen3-VL). Paired with a box-format switch for referring-expression/grounding labels: from the inherited GeoChat-era `{<x1><y1><x2><y2>}` 0-100 bracket convention to Qwen3-VL's own documented native format, `{"bbox_2d": [x1, y1, x2, y2]}` normalized 0-1000 (see Cell 3/Cell 4/Cell 8's own notes for the full reasoning and the exact conversion code). Because this changes the base model itself, not just the LoRA shape, `HF_REPO_ID` was bumped to a new `-v4` repo — any real checkpoint trained under `-v3` (Qwen2.5-VL-based) is fully incompatible with this model and must never be resumed into it. Honest caveat, stated plainly: neither the base-model swap nor the box-format switch has been run on real Kaggle GPU time yet -- run `SMOKE_TEST=True` first and confirm the whole pipeline works end-to-end on Qwen3-VL-4B before committing real GPU-hours, and re-run `scripts/evaluate_vrsbench_accuracy.py` after a real training pass to check whether Acc@0.5/Acc@0.7/mIoU actually improved over v2's disclosed 44.0%/16.0%/0.3671 -- this is an evidence-informed bet, not a guaranteed result.

---

## v5 note (2026-09-19): base model upgraded 4B → 8B, started on Kaggle while institutional-cluster access is pending

This is a fork of the v4 notebook (`kaggle_finetune_qwen2vl_v4.ipynb`), changed in exactly two places: **Cell 4** (`MODEL_ID` → `Qwen/Qwen3-VL-8B-Instruct`, plus a quantization toggle — see that cell's note) and **Cell 7** (`HF_REPO_ID` → a brand-new `-v5` repo, never `-v3`/`-v4`).

**Why a new repo name, not resuming `-v4`:** a LoRA adapter's weights are only meaningful attached to the exact base model they were trained against. `-v4` was trained (or smoke-tested) against `Qwen3-VL-4B-Instruct`; this run uses `Qwen3-VL-8B-Instruct` — different hidden sizes/attention shapes throughout, so an attempted resume from `-v4` into this run would fail outright on a real shape mismatch, not silently degrade. `-v3` is a separate, older, already-tainted name (Qwen2.5-VL-3B-based) — never point anything at it. Every version gets its own permanent name; none are ever reused for a different base model or shape.

**Why start this on Kaggle at all, when the plan is to move to an institutional GPU cluster (RTX A6000/6000 Ada, 48GB VRAM) once access comes through:** no reason to sit idle waiting. This notebook keeps 4-bit quantization ON by default (`USE_4BIT_QUANTIZATION = True` in Cell 4) so it fits Kaggle's 16GB T4/P100. LoRA training only ever updates the adapter weights (the base model stays frozen either way), so a checkpoint started under 4-bit quantization here resumes cleanly later on the cluster with `USE_4BIT_QUANTIZATION = False` (full bf16 precision) — same `HF_REPO_ID`, same resume-from-Hub-checkpoint logic already in Cell 7, no notebook restructuring needed. Don't change anything else (`NUM_EXAMPLES`, `INCLUDE_BIGEARTHNET_TXT`, the LoRA config in Cell 7) between Kaggle sessions or when switching to the cluster — the resume logic assumes the same data composition and model shape across the whole run, exactly as v4's own notes already say.

**Honest expectation-setting, not a guarantee:** 8B has more general capability than 4B, but this is still LoRA (base frozen, only a small adapter trained) on the same VRSBench + BigEarthNet.txt data mix as v4 — expect an incremental accuracy improvement over v4's real measured numbers, not a dramatic jump. Run `scripts/evaluate_vrsbench_accuracy.py` against both `-v4` and `-v5` once each finishes and report the real comparison, not a hoped-for one.

**GPU-hour math, honestly unmeasured until this actually runs:** 8B is roughly ~2x the compute of 4B per step. If v4's real measured rate was ~98.0s/step (see `V4_TRAINING_RUNBOOK.md`), 8B on the SAME Kaggle T4 hardware could plausibly land around ~150-200s/step — `TimeBudgetGuard` (Cell 9) will print the real number within the first ~20 real steps of this run; trust that over this estimate. On the Ada6000 cluster (once available), full bf16 precision and much stronger compute should measure meaningfully faster than this Kaggle estimate, but that number is unverified until actually run there too — don't assume it, measure it.


---

## Research update (2026-09-19): CDVQA training slice + NEFTune added, before any real GPU time spent

A research pass (deadline confirmed extended, so this prioritizes model quality over speed) found two concrete, evidence-backed improvements, added here before this run's first real (non-smoke) training:

**1. A new mandatory-requirement gap, closed for real.** Until now, this project's PS-mandatory "change-based VQA" requirement was satisfied purely by an INFERENCE-time prompt trick (ChangeFormer mask -> GeoChat/Qwen asked to describe it) — no fine-tuning run (v2/v3/v4/v5) ever actually trained on real bi-temporal change-QA examples. A directly relevant, very recent paper, **"Revisiting Change VQA in Remote Sensing with Structured and Native Multimodal Qwen Models"** (arXiv:2604.18429), LoRA fine-tunes Qwen-family VLMs on the real **CDVQA benchmark** (Yuan et al. 2022, IEEE TGRS, arXiv:2112.06343, dataset at `github.com/YZHJessica/CDVQA`, Apache-2.0) and reports 68-74% accuracy from a small config: LoRA r=16/alpha=32 on attention projections only, lr=5e-5 cosine decay, 3 epochs, ~2,968 image pairs / ~122,000 QA pairs. New cell below (`finetuning/cdvqa_prepare.py`, `INCLUDE_CDVQA` flag) adds a real slice of this data to the training mix, so the fine-tune genuinely covers CDVQA, not just VRSBench VQA/grounding + BigEarthNet.txt referring-expression. Honest disclosure: this sandbox's own network egress could not reach github.com to verify the dataset's exact JSON schema live (a restriction of the dev environment this was written in, not of Kaggle) — `cdvqa_prepare.py` is written defensively (prints full real diagnostics, never fabricates a schema or a download link) and MUST be run with `--smoke-test` first on Kaggle (which has normal internet access) to confirm the real structure before trusting a full run, exactly like this project's other data-prep scripts already do for VRSBench/BigEarthNet.txt.

**2. NEFTune added** (`neftune_noise_alpha=5` in Cell 9's `SFTConfig`) — a verified, reproduced-widely technique (arXiv:2310.05914) that adds noise to training embeddings to measurably improve instruction-tuning quality, at zero extra inference cost and negligible training cost. Single-line, low-risk addition.

**Also checked and deliberately NOT changed:** LoRA rank 32/alpha 64, lr 2e-4, 2 epochs, all-major-linear-layers target modules — current (2026) best-practice guidance (Unsloth's LoRA hyperparameter guide) already matches what this notebook was doing before this update, so none of that was touched. DoRA (LoRA's newer weight-decomposed variant) was considered and rejected — real-world comparisons show it can be slower AND less accurate in practice despite the paper's headline numbers; not worth the risk here.

**BigEarthNet.txt data already extracted (150,000 examples, verified on the Hub) is NOT wasted and needs no rework** — it's unaffected by this update, still loaded exactly as before via `INCLUDE_BIGEARTHNET_TXT`.


---

## "All in" update (2026-09-19, same day): RSVQA-LR data + LoRA rank/capacity increase + fuller VRSBench pool

On top of the CDVQA slice + NEFTune + warmup_ratio added earlier today (see the section above), three more changes, all combined into this same run per an explicit "go all in on accuracy, deadline is not a constraint" instruction:

1. **RSVQA-LR added** (`finetuning/rsvqa_prepare.py`, `INCLUDE_RSVQA` flag, new Cell 5d). RSVQA is one of the PS's own explicitly prescribed datasets, alongside VRSBench and CDVQA — this fine-tune now genuinely trains on real data from three of the PS's named datasets, not two. RSVQA-LR (not HR) chosen deliberately: real, verified Zenodo record (https://zenodo.org/record/6344334), ~150.5MB total, small enough to use inline in this GPU notebook — HR is ~14.4GB and was deliberately left out of THIS round, same reasoning BigEarthNet.txt's SAR modality was deliberately deferred earlier. Same honest caveat as CDVQA: this sandbox couldn't reach zenodo.org to verify the exact JSON layout live, so `rsvqa_prepare.py` uses Zenodo's real public REST API and prints full diagnostics rather than assuming a schema — run with `--smoke-test` first.
2. **VRSBench pool raised 100,000 -> 120,000** (Cell 3) — real pool is 123,221 records; this now uses ~97% of it instead of ~82%, leaving only a small real buffer.
3. **LoRA rank doubled 32 -> 64** (alpha 64 -> 128, keeping the 2x rule) — a real, disclosed bet: more capacity, justified by this run's training mix now being larger and more diverse (4 real sources instead of 2) than any previous run, which reduces the overfitting risk a rank increase alone would otherwise carry. Not a proven win — the existing held-out eval_loss tracking / load_best_model_at_end guard is the real safety net; watch eval_loss vs training loss after the first real run.

**Net effect on this run's real data sources**: VRSBench (up to 120,000) + BigEarthNet.txt referring-expression (up to 150,000) + CDVQA (up to 20,000, bi-temporal) + RSVQA-LR (up to 15,000) = up to ~305,000 combined real examples, roughly 22% more than the previous ~250,000 ceiling, PLUS genuinely broader task-type coverage (adds real change-VQA and a third PS-prescribed dataset) rather than just more of the same two sources. GPU-hour estimate scales accordingly — TimeBudgetGuard (Cell 9, using its updated name after the CDVQA/RSVQA cells) will report the real number within the first ~20 real steps; don't assume the old ~1,700h (4B) or un-measured 8B estimate still applies unchanged.

**Run order update (2026-09-19, later same day): CDVQA/RSVQA prep now runs INSIDE this notebook.**
`cdvqa_prepare.py` and `rsvqa_prepare.py`'s real logic was verified for correctness on the user's own machine first (real internet access there, unlike this dev sandbox) — that run confirmed the real 3-file schema join, the real SECOND `im1`/`im2` folder convention, and the real `{img_id}.tif` RSVQA-LR image-naming convention, producing 20,000 and 15,000 genuine usable examples respectively. But that machine's manifests point at local Windows paths and local image files that never leave it — meaningless on Kaggle's Linux filesystem. Rather than manually repackage and upload those files, the now-confirmed-correct scripts are embedded directly in **Cell 5c-setup** and **Cell 5d-setup** below (just before their respective loader cells) and run live on Kaggle, which has its own real internet access to GitHub/Google Drive/Zenodo — this produces Kaggle-native manifests and Kaggle-local images automatically, no manual transfer step needed.

**Requires: this Kaggle notebook's internet access must be ON** (Notebook Settings -> Internet -> On) — without it, Cell 5c-setup/5d-setup's downloads fail immediately (a real, easy-to-miss Kaggle setting, not a code bug). Both setup cells run their own `--smoke-test` first and only proceed to the full extraction if it produced real usable examples, printing full diagnostics either way — same discipline as everywhere else in this project. If a setup cell's smoke test fails on Kaggle (e.g. a Google Drive rate-limit on the large SECOND archive), the corresponding loader cell (5c/5d) detects the missing/incomplete manifest and continues WITHOUT that data source rather than failing the whole run.

(BigEarthNet.txt's existing real cache from the earlier v4 run needs no rework — see the section above; it's pulled from its own HF Hub cache repo, not from these two scripts.) Then run this notebook top to bottom with `SMOKE_TEST=True` first, exactly as before, before committing to a full run.


## v5 update (2026-09-19, later same day): real Kaggle crash fixed + BigEarthNet.txt auto-load + quantitative eval added

Three fixes/additions made from ACTUAL Kaggle output pasted back, not guesses:

1. **Real crash fixed**: `SFTConfig.__init__() got an unexpected keyword argument 'warmup_ratio'` -- Cell 2's unpinned `-U trl` had resolved to a version without that field. Root-caused: Cell 2 now pins `trl>=0.12.0` (confirmed to have merged SFTConfig into `transformers.TrainingArguments`, which has carried `warmup_ratio` for years) and prints installed `transformers`/`trl`/`peft`/`accelerate` versions so a future drift is visible immediately. Cell 8 additionally builds its config as a plain dict and filters it through `inspect.signature(SFTConfig.__init__)` before construction -- any future trl release that renames/drops a field now degrades to a printed warning instead of crashing the whole run again.
2. **BigEarthNet.txt was silently falling back to VRSBench-only on Kaggle** (`Real data sources used this run: ['VRSBench (xiang709/VRSBench)']`, confirmed from real output) -- Cell 5b only ever checked for a local manifest file and never actually pulled it. New **Cell 5b-setup** logs into HF (needed since the cache repo is private) and runs `bigearthnet_txt_extract.py`'s already-existing fast cache-hit path against the same `-full` cache v4 used, exactly the same self-contained pattern as Cell 5c-setup/5d-setup.
3. **New Cell 11: real quantitative held-out evaluation**, closing a gap this project's own PRD calls out explicitly (Section 2C-i: "a measurable before/after difference"; Phase 4: a VRSBench/RSVQA/CDVQA benchmark script). Scores the freshly-trained adapter on real, genuinely-unseen-during-training examples from all three sources (VRSBench's existing held-out `eval_manifest`, plus CDVQA's and RSVQA-LR's real Val splits -- confirmed untouched by training, since both prep scripts exhaust their `--max-examples` cap from the Train split alone before ever reaching Val). Reports exact-match and token-F1 per source and saves full per-example predictions to `real_eval_results_v5.json`, so any accuracy number can be spot-checked against real model output rather than trusted blind.

**Known, disclosed, NOT-yet-addressed gap** (flagged for the next round, not hidden): the PS's most heavily-weighted mandatory requirement -- genuine optical+SAR joint reasoning learned by the model itself -- is still only satisfied at INFERENCE time by the backend's separate `sar_fusion_service` (real backscatter statistics, not fabricated, but not something the fine-tuned model learned to reason over jointly). BigEarthNet.txt's own S2-only extraction (this notebook's Cell 5b/5b-setup) deliberately defers the S1/SAR archive (see `bigearthnet_txt_extract.py`'s module docstring) -- closing this for real means a second extraction pass over `BigEarthNet-S1.tar.zst` (~51 GiB, same streaming approach) producing genuine paired optical+SAR training examples (mirroring CDVQA's two-image format), which has not been built or verified yet and should not be assumed to work without the same smoke-test discipline as everything else in this project.

## v5 update (2026-09-19, third pass same day): real optical+SAR joint training added -- closing the last disclosed gap

The previous update disclosed one known limitation: optical-SAR fusion was only happening at INFERENCE time (backend `sar_fusion_service`), never learned by the fine-tuned model itself during training. Real web research today (arxiv.org/abs/2603.29630, txt.bigearth.net, arxiv.org/abs/2407.03653, bigearth.net's own downloads page and Description_BigEarthNet_v2.pdf) found that the ORIGINAL project PRD's assumption -- that BigEarthNet.txt ships annotation text specifically describing SAR-derived complementary information -- is NOT confirmed by any of these real sources; the dataset's confirmed real annotation types (captions, VQA, referring-expression grounding) are all optical-only. Rather than fabricate SAR-descriptive text to fill that gap, **new Cell 5e-setup / Cell 5e** take the honest path: BigEarthNet.txt's real optical caption/answer for each patch is kept completely unchanged, and a second sentence is appended that is a genuinely COMPUTED statement from that same patch's real Sentinel-1 VV/VH pixel data (relative backscatter comparison + a hedged, well-established SAR interpretation heuristic, corpus-relative and calibration-agnostic rather than asserting unverified absolute physical units) -- exactly the same "real signal, honestly hedged" standard `sar_fusion_service.py` already uses at inference time, now baked into a training target. Real, independently-verified facts this relies on: the S1 archive lives at the same Zenodo record as the already-used S2 archive (`BigEarthNet-S1.tar.zst`, ~51 GiB), its real per-patch/per-band naming convention (`<patch>_VV.tif` / `<patch>_VH.tif`), and the real `s1_name` join key already captured (but previously unused) by `bigearthnet_txt_filter.py`.

**Honestly disclosed**: this is a brand-new component this dev sandbox cannot live-test (no network path to zenodo.org here, the same restriction disclosed throughout this project) -- it is written with the same smoke-test-first, print-real-diagnostics, never-fabricate-a-manifest discipline as every other data source in this project, and **must be smoke-tested on Kaggle before being trusted**, exactly like CDVQA/RSVQA were. If the real S1 archive's member-naming convention turns out to differ even slightly from what `bigearth.net`'s own documentation describes, Cell 5e-setup's smoke test will show that directly and the pipeline will continue without this data source rather than silently building something wrong -- it will not crash the rest of the run.

## v5 update (2026-09-19, fourth pass): every data source now caches to HF Hub -- no more re-downloading on every resume

Real gap the user caught: CDVQA and RSVQA-LR had NO cache, unlike BigEarthNet.txt/SAR-fusion -- every fresh Kaggle session (a resume after the ~12h/session limit, or any new session) re-cloned the CDVQA repo and re-downloaded SECOND's ~4 GiB Google Drive archive, and re-downloaded RSVQA-LR's ~150 MB from Zenodo, from scratch every time. Fixed: both `cdvqa_prepare.py` and `rsvqa_prepare.py` gained an optional `--cache-repo` (same check-cache-first-then-push pattern as `bigearthnet_txt_extract.py` already used) -- Cell 5c-setup/5d-setup now pass this, so **only the very first successful run does the real download; every resume after that is one small, fast HF Hub download.** The core join/schema logic in both scripts is unchanged from the already-verified-correct version -- only the caching wrapper is new.

## v5 update (2026-09-19, fifth pass): fixed a disk-duplication bug + added disk-usage diagnostics

Caught from the user's own real Kaggle screenshot ("Output 2.1GiB / 19.5GiB" -- Kaggle's `/kaggle/working` persisted-output directory has a real, hard ~19.5 GiB ceiling): the previous pass's CDVQA caching change (`shutil.copy2`) stored SECOND's images TWICE on disk -- once under `second_images/`, again under `cdvqa_real/images/`. Fixed to `shutil.move` instead, with the now-redundant `second_images/` tree removed afterward and the real freed space printed -- one copy on disk, not two. Also added a small real disk-usage check (`shutil.disk_usage`) after each of the four data-loading cells (BigEarthNet.txt, CDVQA, RSVQA-LR, optical-SAR fusion), so a genuine risk of approaching Kaggle's `/kaggle/working` ceiling shows up as an early, clear number instead of a mid-training crash. Note for context: the base model weights and most HF Hub caches typically live outside `/kaggle/working` (so they don't count against this specific 19.5 GiB number) -- it's specifically the relative-path dataset/checkpoint directories this notebook creates that do.

In [ ]:
# Cell 1: Verify GPU environment
!nvidia-smi

In [ ]:
# Cell 2: Install dependencies (Qwen3-VL needs a recent transformers -- installing from
# source per Qwen's own model card, since PyPI releases can lag; this also satisfies
# Qwen3-VL's real transformers>=4.57.0 requirement automatically, since installing from
# the git main branch always tracks newer than any pinned PyPI release. Updated
# 2026-09-12 for the v4 base-model swap -- see Cell 4's note.)
#
# Honesty note (2026-08-25): the previous version of this cell did `pip install -U ...
# pillow ...`, which pulled Pillow up to 12.3.0 on Kaggle's base image — too new for the
# torchvision build already there, causing a REAL error at Cell 4 (confirmed from the
# user's actual traceback, not a guess): `ImportError: cannot import name '_Ink' from
# 'PIL._typing'` deep inside `transformers` -> `torchvision` -> `PIL.ImageDraw` -> `PIL.
# ImageText`. Fix: pin Pillow below 12 explicitly, as the LAST install so nothing after it
# silently upgrades it again. This isn't an arbitrary downgrade — it matches what Kaggle's
# own pre-installed `gradio` already declares it needs (`pillow<12.0,>=8.0`, seen in this
# same base image's own pip dependency-conflict warning), so it's aligning with the base
# image's real constraints rather than fighting them.
!pip install -q -U git+https://github.com/huggingface/transformers accelerate
# Real crash fix (2026-09-19, from an actual Kaggle traceback): an unpinned `-U trl` previously resolved to a version whose SFTConfig did NOT accept `warmup_ratio` -- 
#   TypeError: SFTConfig.__init__() got an unexpected keyword argument 'warmup_ratio'
# Pinning to a version confirmed (via TRL's own PyPI changelog) to have merged SFTConfig into transformers.TrainingArguments (which has carried warmup_ratio for years) fixes the root cause instead of just working around one symptom. Cell 8 ALSO defensively filters sft_config kwargs against whatever version actually gets installed here, so a future trl release renaming/removing a field degrades with a printed warning instead of crashing the whole run again.
!pip install -q -U peft bitsandbytes "trl>=0.12.0"
!pip install -q -U datasets huggingface_hub
!pip install -q qwen-vl-utils
!pip install -q "pillow<12,>=10" --force-reinstall

# Real diagnostics (2026-09-19): print what actually got installed, so a future dependency-resolution surprise shows up here first, not as a mid-training crash.
import transformers, trl, peft, accelerate
print(f"Installed versions -- transformers={transformers.__version__}, "
      f"trl={trl.__version__}, peft={peft.__version__}, accelerate={accelerate.__version__}")

In [ ]:
# Cell 3: Real image-grounded training data from VRSBench.
#
# Honesty correction (2026-08-25): the previous version of this cell assumed
# `load_dataset("xiang709/VRSBench", streaming=True)` would yield rows with real images and
# text bundled together, the way many HF vision-language datasets work. That assumption was
# WRONG for this specific repo — discovered because running this exact cell on Kaggle hit
# `AssertionError: Could not auto-detect an image field` for real, not a guess. Checking the
# actual repo file listing (huggingface.co/api/datasets/xiang709/VRSBench) explains why: this
# dataset ships as `Images_train.zip` / `Images_val.zip` (raw image archives) plus
# `VRSBench_train.json` / `VRSBench_EVAL_*.json` (separate annotation files) — not a
# `datasets`-library-ready imagefolder/parquet structure. `load_dataset(..., streaming=True)`
# was only picking up the loose JSON as flat text rows, with zero embedded image bytes.
#
# The fix: download the real files directly and join them ourselves.
#   - Both files come from the SAME official VRSBench release by the SAME authors — this is
#     NOT the cross-dataset join risk this project deliberately avoided with "BigEarthNet.txt"
#     (arXiv:2603.29630, this project's README-cited benchmark): that dataset's annotations
#     and images are two different, separately-published archives with an unverified
#     patch-ID mapping. Here, the image archive and its annotation file are one release,
#     meant to be used together, so joining by filename is the intended, documented use.
#   - The real field names were first confirmed by directly reading this dataset's sibling
#     VRSBench_EVAL_vqa.json file: "image_id"/"question"/"ground_truth". VRSBench_train.json
#     itself turned out to use a DIFFERENT, real, confirmed shape (found by actually running
#     this cell on Kaggle): top-level keys "id", "image" (bare filename, e.g.
#     "00002_0000.png"), and "conversations" — a ShareGPT/LLaVA-style list of
#     {"from": "human"/"gpt", "value": "..."} turns, e.g.
#     [{"from": "human", "value": "<image>\n[caption] Could you describe..."},
#      {"from": "gpt", "value": "..."}]. The zip's image filenames carry an "Images_train/"
#     prefix over the bare filename in "image", resolved below by suffix match. This cell
#     still prints full diagnostics before asserting in case a future re-download or a
#     different record subset doesn't match this confirmed shape — real data, checked twice
#     now, not a first guess.
#
# v3 REVISED (2026-09-12, same day -- SIH26167 deadline confirmed EXTENDED, so this is
# no longer sized against a tight calendar window; re-targeted purely for model quality).
# NUM_EXAMPLES raised 30,000 -> 100,000 (<82% of the real 123,221-record VRSBench pool,
# leaving real headroom rather than exhausting it) and paired with up to 150,000 real
# BigEarthNet.txt referring-expression examples (Cell 5b, capped there to whatever the
# real pool actually contains) for up to 250,000 combined unique examples.
# num_train_epochs restored 1 -> 2 (Cell 8) -- back to this project's own proven
# capacity-plus-epochs combination that produced the currently-verified 44.0%/16.0%/
# 0.3671 grounding numbers (v2), now applied at ~30x the data scale. LoRA rank doubled
# 16 -> 32 (Cell 6/7, alpha 32 -> 64 to hold the standard alpha=2r ratio) since a much
# larger, more diverse real dataset supports more capacity with real diminishing
# overfitting risk -- still protected by the existing held-out eval tracking /
# load_best_model_at_end guard either way.
#
# Real GPU-hour math, stated plainly for relay planning (not to argue the scale down):
# up to 250,000 examples * 2 epochs / 8(grad_accum) = up to 62,500 steps * 48s ~= 833
# real GPU-hours. Across 6-7 Kaggle free-tier accounts (~180-210 combined GPU-hours/
# week), that's roughly 4-4.5 weeks of a continuously-rotating relay -- a real,
# substantial commitment. The full-checkpoint Hub-resume mechanism (Cell 6) is exactly
# what makes a relay this long practical: any teammate can start a brand new Kaggle
# session on their own account at any point and resume from the exact real step the
# last session left off at. Re-verify against whatever TimeBudgetGuard (Cell 8) actually
# measures within the first ~20 real steps of THIS run before trusting the ~833h estimate.
#
# Scale-up + rebalancing (2026-08-27, corrected TWICE the same day): the second boss
# review flagged two real, separate gaps this cell now addresses together. Quota history,
# stated honestly rather than silently overwritten: first heard as "111h31m", corrected by
# the user to the real "11h31m", then corrected again to "30h" (quota apparently
# refreshed/re-checked). Sized below against 30h — the CURRENT real number — not either of
# the earlier ones. Re-verify this cell's numbers against whatever the real remaining quota
# is at the moment this actually runs, rather than trusting a number from earlier in the
# conversation if time has passed.
#   1. Scale: NUM_EXAMPLES raised from 600 to 6,000 (10x) — chosen from the ONE real timed
#      data point this project has: the last real completed run (150 steps / 600 examples /
#      2 epochs) took ~2h on a T4, i.e. ~48s per 8-example step at this notebook's
#      batch_size=1 x grad_accum_steps=8 config. At that real observed rate, 6,000 examples
#      x 2 epochs = 1,500 steps x ~48s = an ESTIMATED ~20.0 hours, deliberately leaving
#      ~10 real hours of the 30h budget unspent (~33% buffer) — for the SMOKE_TEST below,
#      one-time dataset-download overhead (separate from per-step time), eval/checkpoint
#      overhead, and the real chance that a first-ever run of this rebalanced/augmented/
#      resumable pipeline needs a rerun. This is NOT "use all available quota" by design:
#      this exact code has never actually run, so under-committing on the first real attempt
#      and having quota left to fix something is safer than over-committing and stranding a
#      run with nothing left to recover it. Watch the real per-step time Cell 8's own
#      TimeBudgetGuard callback prints ~20 steps into the run and trust THAT over this
#      estimate — if real throughput differs meaningfully, stop (Kaggle: Run > Interrupt)
#      and reduce NUM_EXAMPLES/epochs before restarting, rather than finding out 15 hours in.
#   2. Rebalancing: the 600-example manifest's real task-tag composition (Cell 6's own
#      Counter() finding) was {'[vqa]': 365, '[refer]': 143, '[caption]': 92} — a ~24%
#      referring-expression share. That's not negligible, but real IoU evidence
#      (evaluate_vrsbench_accuracy.py, 50 held-out examples, Acc@0.5 = 0.00 even after the
#      coordinate-order fix) shows it clearly wasn't enough for the adapter to learn
#      reliable box localization. This cell now explicitly targets a higher referring share
#      (40% of 6,000 = 2,400 real referring examples, ~16.8x the original 143) using real
#      per-example task-tag classification below, scanning the FULL real 123,221-record
#      file (not just a head slice) so there's enough real referring-expression material to
#      actually fill that quota. Stated honestly: rebalancing the data is a reasonable,
#      evidence-motivated hypothesis for why grounding is currently weak (a frozen vision
#      tower + attention/MLP-only LoRA has to learn precise box regression from
#      comparatively few examples) — it is not a guaranteed fix. Re-run
#      scripts/evaluate_vrsbench_accuracy.py after this retrains to check real progress, and
#      if Acc@0.5 is still ~0, that's evidence the bottleneck is something other than data
#      volume (e.g. the frozen vision tower itself, or LoRA capacity) and is worth flagging
#      back before spending more quota the same way.
#
# SMOKE_TEST (new, 2026-08-27): before committing ~20 real hours to an entirely untested
# rebalancing/augmentation/resume pipeline, flip this on for a small end-to-end dry run
# first (48 examples, ~12 real training steps at 2 epochs) that still exercises data
# loading, rebalanced sampling, the box-token audit, scale-jitter augmentation, a real eval
# pass, a real checkpoint save, and a real Hub push (Cell 8's save/eval interval is also
# SMOKE_TEST-aware so these actually fire during the smoke run, not just the real one) --
# the full pipeline, just small. Honest timing caveat: the ~12 training steps themselves
# are quick (~10 min at the ~48s/step estimate), but a FRESH Kaggle session's package
# installs (Cell 2) and base-model download (Cell 4, a multi-GB 4-bit model) are real,
# mostly fixed costs that don't shrink with SMOKE_TEST -- expect the whole smoke pass to
# take perhaps 15-30 minutes on a cold session, not just a couple of minutes. Only flip
# SMOKE_TEST to False and re-run from Cell 3 once that completes with no errors and the Hub
# push succeeds. Skipping this to save 15-30 minutes is exactly the kind of shortcut that
# risks losing real hours to a bug that a smoke run would have caught immediately.
SMOKE_TEST = True  # <-- set to False only after a SMOKE_TEST=True run finishes cleanly

import io, json, re, zipfile
from pathlib import Path
from huggingface_hub import hf_hub_download
from PIL import Image

NUM_EXAMPLES = 48 if SMOKE_TEST else 120000  # v5.1 (2026-09-19): raised from 100,000 -- 
# real VRSBench_train.json pool is 123,221 records (see this cell's v3-revised note
# above); 120,000 leaves a small real buffer rather than exhausting the pool exactly,
# while using effectively all of it (up from ~82% to ~97%).
print(f"{'SMOKE TEST' if SMOKE_TEST else 'REAL RUN'}: NUM_EXAMPLES = {NUM_EXAMPLES}")
DATASET_ID = "xiang709/VRSBench"
OUT_DIR = Path("vrsbench_real")
IMAGES_DIR = OUT_DIR / "images"
IMAGES_DIR.mkdir(parents=True, exist_ok=True)

IMAGE_ID_CANDIDATES = ["image_id", "image", "filename", "file_name", "img_name", "img_id"]
ANSWER_CANDIDATES = ["ground_truth", "answer", "answers", "response"]
QUESTION_CANDIDATES = ["question", "prompt"]
CAPTION_CANDIDATES = ["caption", "captions", "description"]

# Real per-example task classification (2026-08-27 addition) — VRSBench's own
# ShareGPT-format human turns are prefixed with a literal task tag, e.g.
# "[caption] Could you describe...", "[vqa] What is...", "[refer] Where is...". Used only
# to steer the REBALANCED sampling quotas below; the tag itself is left in the prompt text
# exactly as before (unchanged existing behavior — the model already trains on/sees these
# tags as part of the real question text, this classification is purely for sampling).
TASK_TAG_RE = re.compile(r"^\[(\w+)\]")
# Real box-token format, same pattern backend/models/geochat_service.py's
# parse_boxes_from_text looks for at inference time — used here only as a training-data
# AUDIT (mitigation item #2 from report.tex Sec 5.2: rule out a training-time box-encoding
# bug independent of the inference-time coordinate-order bug already fixed).
BOX_TOKEN_RE = re.compile(r'\{\s*<\s*(\d{1,4})\s*>\s*<\s*(\d{1,4})\s*>\s*<\s*(\d{1,4})\s*>\s*<\s*(\d{1,4})\s*>\s*\}')

# BOX-FORMAT SWITCH (2026-09-12, paired with the Cell 4 base-model upgrade to
# Qwen3-VL-4B-Instruct): VRSBench's own raw '[refer]' answers ship in the OLD GeoChat-
# style "{<x1><y1><x2><y2>}" bracket convention on a 0-100 integer scale (BOX_TOKEN_RE
# above, inherited from an unrelated older project, GeoChat -- NOT something Qwen2.5-VL
# or Qwen3-VL were ever pretrained to emit natively). Verified directly against Qwen3-VL's
# own official cookbook (github.com/QwenLM/Qwen3-VL, cookbooks/2d_grounding.ipynb): its
# real native grounding format is a JSON object with a "bbox_2d" key holding
# [x1, y1, x2, y2] (x-first, same axis order this project already settled on with real
# IoU evidence -- see backend/models/geochat_service.py's parse_boxes_from_text
# docstring), normalized 0-1000 -- explicitly documented by Qwen as "changed from the
# absolute coordinates used in Qwen2.5-VL to relative coordinates ranging from 0 to
# 1000." Training toward THIS convention instead of the inherited GeoChat one is a
# deliberate bet that fine-tuning lands more accurately when it reinforces a format the
# base model's own pretraining already associates with grounding, rather than teaching an
# entirely unfamiliar token pattern from scratch. Stated honestly: this is a reasonable,
# evidence-informed choice, not a proven fix on its own -- the v3 data/capacity scale-up
# (Cell 3/6/7) is still likely the larger driver of any real accuracy gain; re-run
# scripts/evaluate_vrsbench_accuracy.py after training to check the real Acc@0.5/mIoU
# against v2's disclosed 44.0%/16.0%/0.3671 and report the real number, not a hoped-for
# one.
#
# Deliberately DROPS the optional "label" field the cookbook example also shows
# (`{"bbox_2d": [...], "label": "plate/dish"}`) -- this project's referring-expression
# task already supplies the object description as the QUESTION text, so a per-box label
# would just restate the query back with no new grounding signal, at the cost of more
# output tokens to get exactly right. This is a disclosed simplification of the native
# format, not a hidden deviation from it: only the "bbox_2d" key/array is required for
# this project's actual task (locate the one object the question describes).
#
# Applied to every raw response BEFORE it's saved into manifest/eval_manifest below, so
# every '{<x1><y1><x2><y2>}' box token anywhere in the text (not just whole-response-is-
# a-box-token cases) gets converted to '{"bbox_2d": [x1000, y1000, x2000, y2000]}' -- any
# surrounding real prose is left untouched. See also finetuning/bigearthnet_txt_filter.py's
# box_str_to_token() (updated the same day to emit this SAME new format directly) and
# backend/models/geochat_service.py's parse_boxes_from_text() (updated to parse it at
# inference time) -- all three must agree on one convention, or the model would be
# trained on one format and have its real output misread at inference.
def convert_legacy_box_tokens(text):
    def _replace(m):
        x1, y1, x2, y2 = (int(g) for g in m.groups())
        # Old scale is 0-100; new is 0-1000 -- a plain x10, clamped defensively in case a
        # real (occasionally slightly-over-100, per geochat_service.py's own docstring)
        # value would otherwise push past the new scale's own upper bound.
        x1n, y1n, x2n, y2n = (min(1000, max(0, v * 10)) for v in (x1, y1, x2, y2))
        return f'{{"bbox_2d": [{x1n}, {y1n}, {x2n}, {y2n}]}}'
    return BOX_TOKEN_RE.sub(_replace, text)

def first_present(d, candidates):
    for c in candidates:
        if c in d and d[c]:
            return c
    return None

def extract_qa(rec):
    # Flat/top-level shape first (matches the CONFIRMED real schema of this dataset's
    # sibling VRSBench_EVAL_vqa.json file).
    q_field = first_present(rec, QUESTION_CANDIDATES)
    a_field = first_present(rec, ANSWER_CANDIDATES)
    if q_field and a_field:
        return str(rec[q_field]), str(rec[a_field])
    c_field = first_present(rec, CAPTION_CANDIDATES)
    if c_field:
        return "Describe this remote-sensing image in detail.", str(rec[c_field])

    # CONFIRMED real shape of VRSBench_train.json (found by actually running this cell on
    # Kaggle): a ShareGPT/LLaVA-style "conversations" list —
    # [{"from": "human", "value": "<image>\n[caption] <question text>"},
    #  {"from": "gpt", "value": "<answer text>"}]. The generic "question"/"answer" key
    # search above doesn't match this shape (keys are "from"/"value"), so handle it
    # explicitly: pair the first human turn with the following gpt turn.
    for value in rec.values():
        if isinstance(value, list) and value and isinstance(value[0], dict) and "from" in value[0] and "value" in value[0]:
            human_text, gpt_text = None, None
            for turn in value:
                role = str(turn.get("from", "")).lower()
                text = turn.get("value", "")
                if role in ("human", "user") and human_text is None:
                    human_text = text
                elif role in ("gpt", "assistant") and human_text is not None and gpt_text is None:
                    gpt_text = text
                if human_text is not None and gpt_text is not None:
                    break
            if human_text is not None and gpt_text is not None:
                # Strip the literal "<image>" placeholder token — the real image is sent
                # separately via the processor's own image input, not as a text token.
                cleaned_q = human_text.replace("<image>", "").lstrip("\n").strip()
                if cleaned_q:
                    return cleaned_q, str(gpt_text)

    # Generic nested-list-of-dicts fallback, in case a differently-named QA-pair list
    # (not "conversations") shows up in some records.
    for value in rec.values():
        if isinstance(value, list) and value and isinstance(value[0], dict):
            nested = value[0]
            nq, na = first_present(nested, QUESTION_CANDIDATES), first_present(nested, ANSWER_CANDIDATES)
            if nq and na:
                return str(nested[nq]), str(nested[na])
            nc = first_present(nested, CAPTION_CANDIDATES)
            if nc:
                return "Describe this remote-sensing image in detail.", str(nested[nc])
    return None

def classify_task_tag(prompt_text):
    m = TASK_TAG_RE.match((prompt_text or "").strip())
    return m.group(1).lower() if m else "other"

print(f"Downloading real annotation file from {DATASET_ID} ...")
ann_path = hf_hub_download(repo_id=DATASET_ID, repo_type="dataset", filename="VRSBench_train.json")
with open(ann_path) as f:
    records = json.load(f)

if isinstance(records, dict):
    fixed = []
    for k, v in records.items():
        if isinstance(v, dict):
            v = dict(v)
            v.setdefault("image_id", k)
            fixed.append(v)
    records = fixed
    print(f"VRSBench_train.json was a dict keyed by id; converted to {len(records)} records.")
print(f"Loaded {len(records)} raw annotation records.")

first = records[0]
print("Keys found in the first raw record:")
for k, v in first.items():
    print(f"  {k}: {type(v)} = {str(v)[:120]!r}")
    if isinstance(v, list) and v and isinstance(v[0], dict):
        print(f"    -> nested list of dicts, first item keys: {list(v[0].keys())}")

IMAGE_ID_FIELD = first_present(first, IMAGE_ID_CANDIDATES)
assert IMAGE_ID_FIELD, (
    "Could not auto-detect the image-id/filename field in VRSBench_train.json's records. "
    "Check the printed keys above — paste them back for a fix rather than guessing further."
)
print(f"Using image-id field '{IMAGE_ID_FIELD}'")

print("Downloading real image archive (one-time, can take a few minutes)...")
zip_path = hf_hub_download(repo_id=DATASET_ID, repo_type="dataset", filename="Images_train.zip")
zf = zipfile.ZipFile(zip_path)
zip_names = zf.namelist()
print(f"Image archive contains {len(zip_names)} files. First 5: {zip_names[:5]}")

def resolve_zip_name(image_id):
    if image_id in zip_names:
        return image_id
    for n in zip_names:
        if n.endswith(image_id):
            return n
    return None

# Rebalanced, quota-based sampling (2026-08-27) — scans the FULL real record set (not a
# truncated head slice) so under-represented categories (referring-expression) can actually
# fill their real target share instead of whatever their natural frequency happens to be in
# file order.
# Rebalance, second pass (2026-09-04): even the 40% referring share above was set
# BEFORE this exact rebalanced run had ever actually trained on real GPU time -- it was
# an evidence-motivated hypothesis (see the note above this cell), not yet a validated
# number. Bumping [refer] to 50% here, taken from [caption] (20% -> 10%) rather than
# from [vqa] (kept at 40%) so the strong, already-real 84% VQA accuracy number isn't
# put at risk while pushing more signal at the specific gap (Acc@0.5 = 0.00) this
# rebalancing exists to close. Captioning is the least load-bearing of the three task
# types for this project's PS-mandated grounding requirement, so it absorbs the cut.
# This is still a hypothesis, not a guarantee -- re-run scripts/evaluate_vrsbench_
# accuracy.py after training on this mix and report whatever the real Acc@0.5 comes
# back as, not what this comment hopes for.
TARGET_SHARE = {"refer": 0.50, "vqa": 0.40, "caption": 0.10}
quotas = {k: int(NUM_EXAMPLES * v) for k, v in TARGET_SHARE.items()}
quotas["vqa"] += NUM_EXAMPLES - sum(quotas.values())  # absorb rounding remainder
counts = {k: 0 for k in quotas}
counts["other"] = 0
OTHER_QUOTA = 0  # unrecognized task tags aren't part of the deliberate mix; skip them

manifest = []
box_token_seen, box_token_examples = 0, []
for idx, rec in enumerate(records):
    if len(manifest) >= NUM_EXAMPLES:
        break
    image_id = rec.get(IMAGE_ID_FIELD)
    if not image_id:
        continue
    qa = extract_qa(rec)
    if not qa:
        continue
    prompt_text, response_text = qa
    tag = classify_task_tag(prompt_text)
    bucket = tag if tag in quotas else "other"
    quota = quotas.get(bucket, OTHER_QUOTA)
    if counts[bucket] >= quota:
        continue  # this category's real target share is already filled

    zip_name = resolve_zip_name(image_id)
    if not zip_name:
        continue
    try:
        img = Image.open(io.BytesIO(zf.read(zip_name))).convert("RGB")
    except Exception as e:
        print(f"  [{idx}] skipped (image decode failed: {e})")
        continue

    if bucket == "refer":
        m = BOX_TOKEN_RE.search(response_text)
        if m:
            box_token_seen += 1
            if len(box_token_examples) < 5:
                box_token_examples.append(response_text.strip())

    # Box-format conversion (2026-09-12): see convert_legacy_box_tokens' own note above --
    # applied to every saved example's response (a harmless no-op when no legacy box token
    # is present), so manifest.json already reflects the new Qwen3-VL-native convention,
    # not the old inherited GeoChat one. Done AFTER the box_token_seen audit above, which
    # deliberately still checks for the OLD raw VRSBench format (what the dataset itself
    # actually ships), not the post-conversion one.
    response_text = convert_legacy_box_tokens(response_text)

    img_path = IMAGES_DIR / f"vrs_{idx:05d}.png"
    img.save(img_path)
    manifest.append({
        "image_path": str(img_path),
        "prompt": prompt_text,
        "response": response_text,
        "task_tag": bucket,
    })
    counts[bucket] += 1

    if len(manifest) % 500 == 0:
        print(f"  ...{len(manifest)}/{NUM_EXAMPLES}  (real category counts so far: {counts})")

with open(OUT_DIR / "manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)

print(f"\nSaved {len(manifest)} genuine image-grounded training examples to {OUT_DIR}/manifest.json")
print(f"Real final task-tag composition: {counts}  (target quotas were: {quotas})")
assert len(manifest) > 20, "Too few real examples were saved — check the field names printed above and adjust the candidate lists."

# Training-data box-token audit (2026-08-27, mitigation item #2 from report.tex Sec 5.2):
# confirm the referring-expression TRAINING labels actually contain the same
# "{<n><n><n><n>}" bracket-token box format the model is expected to reproduce at inference
# time (backend/models/geochat_service.py's parse_boxes_from_text), rather than assuming it
# from the earlier inference-time coordinate-order finding alone.
refer_total = counts["refer"]
print(f"\nBox-token format audit on {refer_total} real '[refer]' training examples:")
print(f"  {box_token_seen}/{refer_total} ({(box_token_seen/refer_total*100 if refer_total else 0):.1f}%) "
      f"real responses contain a parseable {{<n><n><n><n>}} box token.")
if box_token_examples:
    print("  Real example response strings (verify these visually look x1,y1,x2,y2 on a 0-100 scale):")
    for ex in box_token_examples:
        print(f"    {ex!r}")
if refer_total and box_token_seen / refer_total < 0.9:
    print("  WARNING: fewer than 90% of '[refer]' examples contain the expected box-token "
          "format — this would mean a real fraction of 'referring' training labels aren't "
          "teaching box output at all. Paste this printout back before trusting the "
          "rebalancing above to fix the grounding-accuracy gap.")

# --- Real held-out evaluation set (2026-08-26 addition) ---
# Honesty gap found on review, before committing to a longer (4-epoch) rerun: this cell
# already built a real, correctly-joined TRAINING manifest, but nothing in this notebook
# ever checked the adapter against data it hadn't trained on — the old Cell 10 sanity check
# sampled from this same `manifest`, so a memorizing model could pass it even while failing
# to generalize. VRSBench ships its own real, separately-released held-out files for exactly
# this: confirmed via the dataset's real file listing that `Images_val.zip` is a SEPARATE
# image archive from `Images_train.zip` (real train/val split, not a slice of the same
# images), and `VRSBench_EVAL_vqa.json` is a separate annotation file with a flat schema
# ("image_id"/"question"/"ground_truth") that the extract_qa() flat-field path above already
# handles without changes. Using it costs a modest amount of extra time (a smaller download,
# a small eval pass every 25 steps) for a genuine signal on whether the model is learning to
# answer-and-stop in general, not just replaying the 600 training examples — worth it before
# a 4-hour run rather than finding out after.
EVAL_NUM_EXAMPLES = 60
EVAL_IMAGES_DIR = OUT_DIR / "images_eval"
EVAL_IMAGES_DIR.mkdir(parents=True, exist_ok=True)

print(f"\nDownloading real held-out eval annotation file from {DATASET_ID} ...")
eval_ann_path = hf_hub_download(repo_id=DATASET_ID, repo_type="dataset", filename="VRSBench_EVAL_vqa.json")
with open(eval_ann_path) as f:
    eval_records = json.load(f)
print(f"Loaded {len(eval_records)} raw real held-out eval records.")

print("Downloading real held-out eval image archive (separate from the training images)...")
eval_zip_path = hf_hub_download(repo_id=DATASET_ID, repo_type="dataset", filename="Images_val.zip")
eval_zf = zipfile.ZipFile(eval_zip_path)
eval_zip_names = eval_zf.namelist()
print(f"Eval image archive contains {len(eval_zip_names)} files.")

def resolve_eval_zip_name(image_id):
    if image_id in eval_zip_names:
        return image_id
    for n in eval_zip_names:
        if n.endswith(image_id):
            return n
    return None

eval_manifest = []
for idx, rec in enumerate(eval_records):
    if len(eval_manifest) >= EVAL_NUM_EXAMPLES:
        break
    image_id = rec.get("image_id")
    if not image_id:
        continue
    zip_name = resolve_eval_zip_name(image_id)
    if not zip_name:
        continue
    qa = extract_qa(rec)
    if not qa:
        continue
    prompt_text, response_text = qa
    try:
        img = Image.open(io.BytesIO(eval_zf.read(zip_name))).convert("RGB")
    except Exception as e:
        print(f"  [eval {idx}] skipped (image decode failed: {e})")
        continue
    # Box-format conversion (2026-09-12): keep eval labels in the SAME new convention as
    # train_dataset (see convert_legacy_box_tokens' note above) -- otherwise Cell 9's
    # eval-loss tracking would be scoring the model against a target format it was never
    # trained to produce.
    response_text = convert_legacy_box_tokens(response_text)
    img_path = EVAL_IMAGES_DIR / f"vrs_eval_{idx:05d}.png"
    img.save(img_path)
    eval_manifest.append({
        "image_path": str(img_path),
        "prompt": prompt_text,
        "response": response_text,
    })

with open(OUT_DIR / "eval_manifest.json", "w") as f:
    json.dump(eval_manifest, f, indent=2)

print(f"Saved {len(eval_manifest)} genuine held-out eval examples (from Images_val.zip — images "
      f"the model never trains on) to {OUT_DIR}/eval_manifest.json")
assert len(eval_manifest) > 10, "Too few real held-out eval examples were saved — check VRSBench_EVAL_vqa.json's fields."


In [ ]:
# Cell 4: Load base model + processor (4-bit, for T4-sized VRAM headroom).
#
# BASE MODEL UPGRADE (2026-09-12, explicit user ask: "qwen 2.5 hi use krna h? are u
# sure? research kr, world class model chaiye"). Switched from
# Qwen/Qwen2.5-VL-3B-Instruct to Qwen/Qwen3-VL-4B-Instruct after verifying, not
# assuming:
#   - Real, current model: Qwen3-VL exists (2B/4B/8B/32B+ sizes), Apache-2.0 licensed,
#     released by the same Qwen team as a direct successor to Qwen2.5-VL.
#   - Confirmed working 4-bit bitsandbytes quantization for THIS exact checkpoint (a
#     community report on Qwen/Qwen3-VL-4B-Instruct's own HF discussion #12 confirms
#     basic bnb 4-bit works fine -- only AWQ/GPTQ, which this project doesn't use, have
#     open issues).
#   - Verified directly against the real `transformers` source (modeling_qwen3_vl.py)
#     that the language-model LoRA target modules this notebook already uses (q/k/v/
#     o_proj, gate/up/down_proj) are named IDENTICALLY to Qwen2.5-VL, and the vision
#     tower is still exposed at the same `.visual` attribute -- so Cell 7's LoRA config
#     and its exclude_modules=r".*\.visual\..*" safeguard need NO changes.
#   - Real structural improvement found (not just parity): Qwen3-VL's OWN vision-tower
#     MLP uses `linear_fc1`/`linear_fc2` naming (no more `gate_proj`/`up_proj`/
#     `down_proj` collision with the language model's MLP) -- the exact name-collision
#     class that forced this notebook to add exclude_modules in the first place (see
#     Cell 7's own 2026-08-26 note) is now structurally impossible on the vision side,
#     not just patched around.
#   - Native grounding format changed too, in this project's favor -- see Cell 3's
#     box-format note for the paired change to how referring-expression box labels are
#     encoded, following Qwen3-VL's own documented convention instead of an inherited
#     one from an older, unrelated project (GeoChat).
# Honest caveat: this has NOT been run on real Kaggle GPU time yet. Run SMOKE_TEST=True
# (Cell 3) first and confirm the whole pipeline (load, one train step, one eval, one
# checkpoint push) works end-to-end on Qwen3-VL-4B before committing real GPU-hours -- a
# larger 4B base model will also have a different real s/step than the ~48s/step this
# notebook's estimates were measured against on the 3B model; TimeBudgetGuard (Cell 9)
# will report the real number early in the first real run.
import torch
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen3-VL-8B-Instruct"  # v5 (2026-09-19): upgraded from 4B -- see Cell 0's v5 note

# v5 addition (2026-09-19): explicit toggle instead of an unconditional bnb_config, so the
# SAME notebook works on both Kaggle (16GB T4/P100 -- needs 4-bit) and the institutional
# Ada6000 cluster (48GB -- full bf16 fits comfortably, no quantization needed, better
# numerical precision). Flip this to False once running on the cluster; leave it True on
# Kaggle. This does not affect the LoRA adapter's compatibility across a resume -- LoRA only
# ever trains the small adapter weights, the base model (quantized or not) stays frozen
# either way, so a checkpoint started under 4-bit here resumes cleanly under bf16 later.
USE_4BIT_QUANTIZATION = True  # <-- Kaggle: True. Ada6000/6000-Ada cluster: False.

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
) if USE_4BIT_QUANTIZATION else None

print(f"USE_4BIT_QUANTIZATION={USE_4BIT_QUANTIZATION} "
      f"({'4-bit nf4 (Kaggle-sized VRAM)' if USE_4BIT_QUANTIZATION else 'full bf16 (cluster-sized VRAM)'})")

processor = AutoProcessor.from_pretrained(MODEL_ID)  # CPU-only, safe to run here

# Real restructuring (2026-09-13, for genuine multi-GPU DDP support on Kaggle's
# "GPU T4 x2" accelerator): the actual model load is now a FUNCTION, not executed
# here directly. Reason: accelerate.notebook_launcher (used below in Cell 8 when 2+
# GPUs are detected) requires CUDA to not already be initialized in this main
# notebook process -- loading the model here directly would break that. Single-GPU
# runs (including SMOKE_TEST) call this exact same function directly in-process
# (see Cell 8), so that already-validated path is behaviorally unchanged.
def _load_base_model(device_map):
    m = Qwen3VLForConditionalGeneration.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map=device_map,
        torch_dtype=torch.bfloat16,
        # Real speedup win (2026-09-13, zero accuracy risk): explicitly request
        # PyTorch's fused scaled-dot-product-attention kernel instead of leaving
        # attn_implementation unset (whatever transformers' own default happens to
        # be for this model class). SDPA computes the exact same attention math as
        # the naive/eager path, just via a faster fused CUDA kernel -- this changes
        # NOTHING about what the model learns, only how fast the forward/backward
        # pass runs. If this specific model class does not support "sdpa" it will
        # raise clearly at load time rather than silently doing something else.
        attn_implementation="sdpa",
    )
    return m

print("Defined _load_base_model() for", MODEL_ID, "-- not loaded onto a GPU yet.")

In [ ]:
# Cell 5: Build the real image-grounded training dataset for TRL's SFTTrainer.
# Each example becomes a genuine multimodal chat turn: the real VRSBench image + real
# question/caption as the user message, the real VRSBench answer as the assistant response.
import json
from pathlib import Path
from datasets import Dataset
from PIL import Image

with open("vrsbench_real/manifest.json") as f:
    manifest = json.load(f)

def to_chat_example(row):
    return {
        "images": [row["image_path"]],
        "messages": [
            {"role": "user", "content": [
                {"type": "image"},
                {"type": "text", "text": row["prompt"]},
            ]},
            {"role": "assistant", "content": [
                {"type": "text", "text": row["response"]},
            ]},
        ],
    }

train_examples = [to_chat_example(r) for r in manifest]
train_dataset = Dataset.from_list(train_examples)
print(f"{len(train_dataset)} real image-grounded training examples ready.")

# Real held-out eval set (2026-08-26 addition, see Cell 3) — built the same way, from
# genuinely unseen Images_val.zip images, so Cell 8 can track real generalization during
# training instead of only training loss.
with open("vrsbench_real/eval_manifest.json") as f:
    eval_manifest = json.load(f)

eval_examples = [to_chat_example(r) for r in eval_manifest]
eval_dataset = Dataset.from_list(eval_examples)
print(f"{len(eval_dataset)} real held-out eval examples ready (from Images_val.zip — never seen in training).")

In [ ]:
# Cell 5b-setup (new, 2026-09-19): Real fix for a live gap seen on Kaggle -- this
# notebook's Cell 5b only ever CHECKED for a local bigearthnet_txt_real/manifest.json and
# printed manual instructions if missing, so a fresh Kaggle session (which never ran those
# instructions) silently fell back to "VRSBench-only" and printed
# "Real data sources used this run: ['VRSBench (xiang709/VRSBench)']" -- exactly what a
# real run showed. But bigearthnet_txt_extract.py ALREADY has a fast path for this: if its
# --cache-repo already has a real cached manifest.json (confirmed true for
# Sameelkazi/satquery-qwen3vl-vrsbench-lora-v4-bigearthnet-txt-cache-full, the same cache
# v4 already trained from), it does a small snapshot_download instead of the ~59 GiB
# Zenodo stream. This cell runs exactly that fast path automatically, same self-contained
# pattern as Cell 5c-setup/5d-setup above.
#
# Real requirement: the cache repo is PRIVATE, so this needs a real HF login BEFORE the
# download attempt -- done here explicitly rather than relying on Cell 6 below (which logs
# in too, for its own separate purpose, but runs AFTER this cell).
#
# Honest scope note: this ONLY attempts the fast cache-hit path. If the cache were ever
# empty (not expected here -- it's the same cache v4 already used successfully), this
# deliberately does NOT fall back to the full ~59 GiB Zenodo stream inside this GPU
# notebook -- that heavy extraction belongs on a separate CPU-only Kaggle session (see
# finetuning/README_bigearthnet_txt.md) so it never burns GPU-hour quota. In that case
# Cell 5b below still prints its existing manual-instructions fallback and continues with
# VRSBench-only, exactly as it already does today -- this cell only removes the *common*
# case (cache already populated) from needing a manual step.
import subprocess
from pathlib import Path

try:
    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import login
    _hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    login(token=_hf_token)
    print("Logged into Hugging Face Hub (needed to read the private BigEarthNet.txt cache repo).")
except Exception as e:
    print(f"Could not log into HF Hub here ({e}) -- the cache download below will likely "
          f"fail and Cell 5b will fall back to its existing manual-instructions path. This "
          f"is not fatal to the rest of the notebook.")

Path("bigearthnet_txt_extract.py").write_text(r'''"""
finetuning/bigearthnet_txt_extract.py

Takes bigearthnet_txt_selected.json (from bigearthnet_txt_filter.py -- a list of
{patch_id, s1_name, prompt, response, task_tag} rows with NO image yet) and produces a
real, training-ready manifest.json in the exact schema
finetuning/kaggle_finetune_qwen2vl.ipynb Cell 5 already expects
({"image_path", "prompt", "response", "task_tag"}) by fetching the REAL Sentinel-2 pixels
for each needed patch.

Why this is the hard part (verified via research on 2026-08-29, not assumed):
  - BigEarthNet v2.0's real images are NOT available as a per-patch/streamable download
    anywhere found (not on the official bigearth.net downloads page, not via a Planetary
    Computer STAC catalog -- an open unanswered request exists, not a live integration --
    and torchgeo's loader only handles the OLDER v1.0 archive with different patch IDs).
    The only real option is the official Zenodo release's whole-modality archives:
    BigEarthNet-S2.tar.zst (~59 GiB) and BigEarthNet-S1.tar.zst (~51 GiB).
  - Each patch is a REAL folder of 12 separate per-band GeoTIFFs for S2 (confirmed against
    bigearth.net's own "Description_BigEarthNet_v2.pdf"), not one combined image file --
    e.g. "S2B_MSIL2A_20180421T100029_N9999_R122_T33TWM_00_00_B04.tif" for the Red band.
  - There is no confirmed way to randomly seek into a .tar.zst for just the handful of
    patches we need -- zstd frames aren't seekable without a seekable-format archive, which
    this doesn't appear to be. So this script pays the FULL streaming-download cost once
    (~59 GiB of network transfer for S2-only, which is all this stage needs -- SAR/S1 is
    deliberately deferred, see module-level NOTE below) but keeps PEAK DISK USAGE small by
    reading the archive as a forward-only stream and only materializing the ~9 files per
    wanted patch (3 RGB bands here) it actually needs, discarding everything else without
    ever writing it to disk. Verified end-to-end against a synthetic .tar.zst fixture in
    test_bigearthnet_txt_extract.py (real network access to zenodo.org/bigearth.net is not
    reachable from the sandbox this was authored in -- confirmed via a direct connectivity
    check -- so this exact streaming logic could only be verified structurally there; it
    must be smoke-tested for real on Kaggle, which does have real network access, before
    trusting it against the full ~59 GiB run).

NOTE on scope (deliberate, not an oversight): this stage only extracts B04/B03/B02
(true-color RGB) from BigEarthNet-S2.tar.zst, NOT the S1 (SAR) archive. Reasons: (1) the
immediate ask is closing the Acc@0.5 REFERRING-EXPRESSION gap, which is an optical-only
grounding task; (2) Qwen2.5-VL's frozen CLIP-style vision tower expects standard RGB input,
not a 12-band multispectral tensor, so extracting more than 3 bands per S2 patch would be
wasted bandwidth for this stage regardless. Extending this same streaming pass to also
pull VV/VH from BigEarthNet-S1.tar.zst (for the separately-flagged real-SAR-fusion gap in
sar_fusion_service.py, which currently uses a PNG-proxy) is a natural follow-on, not
included here to keep this stage's ~59 GiB (not ~118 GiB) cost matched to what it actually
buys.

Cross-account/cross-session resume design: the expensive 59 GiB stream should only ever
happen ONCE, from whichever Kaggle account/session runs this first. After building the
small (images + manifest.json, expected tens of MB for ~2,500 small 120x120 patches) real
output, this script pushes it to a private HF Hub DATASET repo. Every subsequent run (any
Kaggle account, any session, as long as it has an HF_TOKEN with access to that same repo --
the same precondition the existing LoRA-checkpoint resume in Cell 6 already relies on)
checks that repo first and, if it already has files, just downloads the small cached
dataset instead of ever touching Zenodo again.
"""
from __future__ import annotations

import argparse
import io
import json
from pathlib import Path
from typing import Iterable

import numpy as np

S2_BANDS_RGB = ["B04", "B03", "B02"]  # Red, Green, Blue -- true-color order
S2_ARCHIVE_URL = "https://zenodo.org/records/10891137/files/BigEarthNet-S2.tar.zst"
CACHE_REPO_SUFFIX = "-bigearthnet-txt-cache"  # appended to the project's existing HF_REPO_ID


def wanted_member_names(patch_ids: Iterable[str]) -> dict[str, tuple[str, str]]:
    """Maps the exact basename we're looking for (e.g. "<patch_id>_B04.tif") to
    (patch_id, band) -- matched by BASENAME only, not full path, since this project does
    not have a verified exact root-folder prefix inside the real .tar.zst (only the
    per-patch-folder + per-band-file naming convention was verified via the official PDF).
    Matching by basename is robust to whatever the real root prefix turns out to be."""
    wanted = {}
    for pid in patch_ids:
        for band in S2_BANDS_RGB:
            wanted[f"{pid}_{band}.tif"] = (pid, band)
    return wanted


def stretch_band_to_uint8(band: np.ndarray, low_pct: float = 2.0, high_pct: float = 98.0) -> np.ndarray:
    """Real Sentinel-2 true-color rendering choice (disclosed, not arbitrary): raw S2
    reflectance bands are uint16 and mostly-dark by eye without a stretch. A 2-98
    percentile clip-and-rescale to 0-255 is the same convention widely used for S2
    true-color previews (e.g. Sentinel Hub / EO Browser's default rendering) -- chosen
    here for the same reason, not invented for this project."""
    band = band.astype(np.float32)
    lo, hi = np.percentile(band, [low_pct, high_pct])
    if hi <= lo:
        # Degenerate (e.g. a fully-uniform synthetic test band) -- avoid div-by-zero,
        # fall back to a flat mid-gray rather than crashing on real edge-case patches.
        return np.full_like(band, 128, dtype=np.uint8)
    stretched = np.clip((band - lo) / (hi - lo), 0.0, 1.0) * 255.0
    return stretched.astype(np.uint8)


def bands_to_rgb_image(band_arrays: dict[str, np.ndarray]):
    """band_arrays must have keys 'B04','B03','B02'. Returns a PIL.Image (RGB). Kept
    separate from any tar/network code so it's directly unit-testable with synthetic
    numpy arrays."""
    from PIL import Image
    missing = [b for b in S2_BANDS_RGB if b not in band_arrays]
    if missing:
        raise ValueError(f"missing required bands for RGB compose: {missing}")
    r = stretch_band_to_uint8(band_arrays["B04"])
    g = stretch_band_to_uint8(band_arrays["B03"])
    b = stretch_band_to_uint8(band_arrays["B02"])
    rgb = np.stack([r, g, b], axis=-1)
    return Image.fromarray(rgb, mode="RGB")


def read_band_geotiff_bytes(raw_bytes: bytes) -> np.ndarray:
    """Reads one in-memory GeoTIFF band file's bytes into a 2D numpy array via rasterio's
    MemoryFile -- no temp file on disk needed for the per-band read itself."""
    import rasterio
    from rasterio.io import MemoryFile
    with MemoryFile(raw_bytes) as memfile:
        with memfile.open() as src:
            return src.read(1)


def _extract_from_zst_stream(raw_stream, wanted: dict[str, tuple[str, str]],
                            images_dir: Path | None = None) -> dict[str, dict[str, bytes]]:
    """Core forward-only stream-consuming logic. If images_dir is given, decodes and saves
    completed RGB patches directly to disk as they arrive, freeing raw band bytes immediately
    so memory usage stays minimal (O(1)) even for 65k+ patches."""
    import tarfile
    import zstandard

    per_patch: dict[str, dict[str, bytes]] = {}
    saved_patches: set[str] = set()
    wanted_patch_ids = {pid for pid, _band in wanted.values()}

    if images_dir is not None:
        images_dir.mkdir(parents=True, exist_ok=True)

    dctx = zstandard.ZstdDecompressor()
    members_seen = 0
    with dctx.stream_reader(raw_stream) as reader:
        with tarfile.open(fileobj=reader, mode="r|*") as tar:
            for member in tar:
                members_seen += 1
                basename = member.name.rsplit("/", 1)[-1]
                hit = wanted.get(basename)
                if not hit:
                    continue
                pid, band = hit
                f = tar.extractfile(member)
                if f is None:
                    continue
                per_patch.setdefault(pid, {})[band] = f.read()

                # If streaming directly to disk, flush completed patches to PNG immediately
                if images_dir is not None and all(b in per_patch[pid] for b in S2_BANDS_RGB):
                    try:
                        band_arrays = {b: read_band_geotiff_bytes(per_patch[pid][b]) for b in S2_BANDS_RGB}
                        img = bands_to_rgb_image(band_arrays)
                        img = img.resize((max(336, img.width), max(336, img.height)), resample=1)
                        img_path = images_dir / f"ben_txt_{pid}.png"
                        img.save(img_path)
                        img.close()
                        saved_patches.add(pid)
                    except Exception as err:
                        print(f"Error saving patch {pid}: {err}")
                    del per_patch[pid]

                if members_seen % 200_000 == 0:
                    have = len(saved_patches) if images_dir is not None else len(per_patch)
                    print(f"  ...scanned {members_seen} tar members, "
                          f"{have}/{len(wanted_patch_ids)} wanted patches matched so far", flush=True)

                total_matched = len(saved_patches) if images_dir is not None else len(per_patch)
                if total_matched == len(wanted_patch_ids) and (
                    images_dir is not None or all(len(v) == len(S2_BANDS_RGB) for v in per_patch.values())
                ):
                    print("All wanted patches fully matched -- stopping stream early "
                          "(no need to read the rest of the archive).", flush=True)
                    break
    return per_patch


def stream_extract_patches(archive_url: str, wanted: dict[str, tuple[str, str]],
                            out_dir: Path, session=None) -> dict[str, dict[str, bytes]]:
    """Streams `archive_url` (a .tar.zst) forward-only and collects raw band-file bytes
    for members whose BASENAME is in `wanted`, grouped by patch_id. Never writes the bulk
    archive to disk -- saves images on the fly to images_dir so peak memory stays tiny. Real HTTP
    streaming happens only when `session` is a real requests.Session."""
    if session is None:
        raise ValueError("a real requests.Session is required")
    resp = session.get(archive_url, stream=True, timeout=120)
    resp.raise_for_status()
    raw_stream = resp.raw
    raw_stream.decode_content = True
    images_dir = out_dir / "images"
    return _extract_from_zst_stream(raw_stream, wanted, images_dir=images_dir)


def build_manifest(selected: list[dict], per_patch_bands: dict[str, dict[str, np.ndarray]],
                    images_dir: Path) -> list[dict]:
    """Pure logic (no I/O beyond writing the given images_dir) -- emits one manifest row
    per selected (prompt, response) pair whose image was saved to images_dir, matching
    finetuning/kaggle_finetune_qwen2vl.ipynb Cell 3's manifest.json schema."""
    images_dir.mkdir(parents=True, exist_ok=True)
    image_path_by_patch: dict[str, str] = {}
    manifest = []
    skipped_no_image = 0
    for row in selected:
        pid = row["patch_id"]
        if pid not in image_path_by_patch:
            # Shard into subdirectories of <= 10,000 files to respect Hugging Face directory limits
            part_name = f"part_{abs(hash(pid)) % 10:02d}"
            part_dir = images_dir / part_name
            part_dir.mkdir(parents=True, exist_ok=True)
            img_path = part_dir / f"ben_txt_{pid}.png"
            if not img_path.exists():
                # Fallback to in-memory bands if passed
                bands = per_patch_bands.get(pid)
                if not bands or any(b not in bands for b in S2_BANDS_RGB):
                    skipped_no_image += 1
                    continue
                img = bands_to_rgb_image(bands)
                img = img.resize((max(336, img.width), max(336, img.height)), resample=1)
                img.save(img_path)
                img.close()
            image_path_by_patch[pid] = str(img_path)
        manifest.append({
            "image_path": image_path_by_patch[pid],
            "prompt": row["prompt"],
            "response": row["response"],
            "task_tag": row.get("task_tag", "refer"),
        })
    if skipped_no_image:
        print(f"WARNING: {skipped_no_image} selected examples had no matching extracted "
              f"image (patch not found in the archive stream) and were dropped.", flush=True)
    return manifest


def main() -> None:
    ap = argparse.ArgumentParser(description=__doc__)
    ap.add_argument("--selected", type=Path, default=Path("bigearthnet_txt_selected.json"))
    ap.add_argument("--out-dir", type=Path, default=Path("bigearthnet_txt_real"))
    ap.add_argument("--cache-repo", type=str, required=True,
                     help="HF dataset repo id to cache the small processed output in, e.g. "
                          "YOUR-HF-USERNAME/satquery-qwen25vl-vrsbench-lora-bigearthnet-txt-cache "
                          "(same HF account/token this project's LoRA checkpoints already use).")
    args = ap.parse_args()

    import sys
    if sys.platform == "win32":
        try:
            import ctypes
            ctypes.windll.kernel32.SetThreadExecutionState(0x80000002)  # ES_CONTINUOUS | ES_SYSTEM_REQUIRED
        except Exception:
            pass

    from huggingface_hub import HfApi, snapshot_download

    api = HfApi()
    try:
        files = api.list_repo_files(args.cache_repo, repo_type="dataset")
        if "manifest.json" in files:
            print(f"Found an existing cached dataset at https://huggingface.co/datasets/{args.cache_repo} "
                  f"-- downloading that directly instead of re-streaming the ~59 GiB Zenodo archive.")
            snapshot_download(repo_id=args.cache_repo, repo_type="dataset", local_dir=str(args.out_dir))
            cached_manifest = json.loads((args.out_dir / "manifest.json").read_text())
            print(f"Wrote {args.out_dir}/manifest.json from cache ({len(cached_manifest)} examples). Done.")
            return
        else:
            print(f"Cache repo exists at {args.cache_repo} but manifest.json is missing -- redoing extraction.")
    except Exception as e:
        print(f"No cached dataset found at {args.cache_repo} ({e}) -- doing the real, "
              f"one-time ~59 GiB extraction from Zenodo. This will be cached for every "
              f"future run/account after this one.")

    import requests
    selected = json.loads(args.selected.read_text())
    patch_ids = sorted({row["patch_id"] for row in selected})
    wanted = wanted_member_names(patch_ids)
    print(f"Need {len(patch_ids)} unique real patches ({len(wanted)} band files) out of "
          f"{len(selected)} selected training examples.")

    session = requests.Session()
    raw_bands = stream_extract_patches(S2_ARCHIVE_URL, wanted, args.out_dir, session=session)
    per_patch_arrays = {
        pid: {band: read_band_geotiff_bytes(data) for band, data in bands.items()}
        for pid, bands in raw_bands.items()
    }

    images_dir = args.out_dir / "images"
    manifest = build_manifest(selected, per_patch_arrays, images_dir)
    assert len(manifest) > 20, (
        "Too few real images were matched -- likely means the tar member basename "
        "convention differs from what was assumed (<patch_id>_<band>.tif). Print the "
        "first ~20 real member names from the stream and compare before re-running "
        "against the full archive."
    )
    (args.out_dir / "manifest.json").write_text(json.dumps(manifest, indent=2))
    print(f"Wrote {args.out_dir}/manifest.json with {len(manifest)} real training examples.")

    print(f"Pushing small processed cache to https://huggingface.co/datasets/{args.cache_repo} "
          f"so no future run ever needs to re-stream the full archive...")
    api.create_repo(args.cache_repo, repo_type="dataset", private=True, exist_ok=True)
    api.upload_folder(repo_id=args.cache_repo, repo_type="dataset", folder_path=str(args.out_dir))
    print("Done.")


if __name__ == "__main__":
    main()
''')

BIGEARTHNET_CACHE_REPO = "Sameelkazi/satquery-qwen3vl-vrsbench-lora-v4-bigearthnet-txt-cache-full"
print(f"=== Attempting fast cache download from {BIGEARTHNET_CACHE_REPO} ===")
r = subprocess.run(
    ["python", "bigearthnet_txt_extract.py", "--cache-repo", BIGEARTHNET_CACHE_REPO,
     "--out-dir", "bigearthnet_txt_real"],
    capture_output=True, text=True,
)
print(r.stdout)
if r.stderr:
    print("--- stderr ---")
    print(r.stderr)

if Path("bigearthnet_txt_real/manifest.json").exists():
    print("\nReal BigEarthNet.txt manifest now present -- Cell 5b below will load it.")
else:
    print("\nCache download did not produce a manifest here -- Cell 5b below will detect "
          "this and continue with VRSBench-only rather than fail the whole run. If this "
          "is unexpected (the cache should already be populated from the v4 run), paste "
          "the stdout/stderr above back rather than assuming why.")


In [ ]:
# Cell 5b: Optional second real training source -- BigEarthNet.txt (arXiv:2603.29630),
# the PS's own PRIMARY/prescribed fine-tuning dataset. VRSBench (Cells 3/5 above) is used
# as this project's main training source under the PS's own "BigEarthNet.txt OR OTHER
# OPEN-SOURCE TRAINING DATA" clause (SatQuery_AI_PRD_for_AI_Coding_Agents.md, PS transcript
# lines 11/27) -- a documented, defensible substitution, not a shortcut. This cell adds a
# genuine, real, image-grounded referring-expression subset built directly from
# BigEarthNet.txt itself (its own real S2 imagery via BigEarthNet v2.0/reBEN, not a
# fabricated stand-in), closing that gap for real rather than arguing around it, and
# specifically targeting the referring-expression/grounding task this project's own
# Acc@0.5 evaluation (data/vrsbench_accuracy_eval.json) showed was weak.
#
# Real, verified facts this relies on (checked directly against HF's datasets-server API
# and bigearth.net's own "Description_BigEarthNet_v2.pdf" on 2026-08-29, not assumed):
#   - BigEarthNet.txt's 9.55M-row annotation parquet ships a `patch_id`/`S1_name` join key
#     as an explicit column -- NOT an unverified cross-dataset correspondence.
#   - Its real images come from BigEarthNet v2.0/reBEN -- distributed ONLY as whole-modality
#     archives (BigEarthNet-S2.tar.zst ~59 GiB, BigEarthNet-S1.tar.zst ~51 GiB) with no
#     confirmed partial/streaming download anywhere. finetuning/bigearthnet_txt_extract.py
#     streams the S2 archive forward-only and keeps peak disk usage small (only the wanted
#     patches' 3 RGB band files are ever materialized), but the full ~59 GiB network
#     transfer genuinely has to happen once. This is a heavy, mostly-CPU/network cost --
#     RUN IT ON A CPU-ONLY KAGGLE SESSION FIRST (see finetuning/README_bigearthnet_txt.md),
#     separately from this GPU notebook, so it never counts against GPU-hour quota. Its
#     small output (images + manifest.json) is then cached to a private HF dataset repo, so
#     this cell here only ever does a small download, never the full 59 GiB again.
#
# Deliberately scoped to optical-only referring-expression data (not SAR/S1) for THIS
# stage -- see bigearthnet_txt_extract.py's module docstring for why.
INCLUDE_BIGEARTHNET_TXT = True  # <-- set False to train on VRSBench alone, as before

if INCLUDE_BIGEARTHNET_TXT:
    from pathlib import Path
    from datasets import concatenate_datasets

    be_manifest_path = Path("bigearthnet_txt_real/manifest.json")
    if not be_manifest_path.exists():
        print(f"INCLUDE_BIGEARTHNET_TXT=True but {be_manifest_path} doesn't exist yet -- "
              f"run (in order, BEFORE this notebook, ideally on a separate CPU-only Kaggle "
              f"session to protect GPU-hour quota):\n"
              f"  python finetuning/bigearthnet_txt_filter.py --max-examples 150000\n"
              f"  python finetuning/bigearthnet_txt_extract.py --cache-repo "
              f"Sameelkazi/satquery-qwen3vl-vrsbench-lora-v4-bigearthnet-txt-cache-full\n"
              f"(the real, verified-on-the-Hub -full cache -- see README_bigearthnet_txt.md; an "
              f"earlier un-suffixed name is tainted with a stale 380-example partial run, do "
              f"not use it). v5 note: this is the SAME cache v4 uses -- the box format "
              f"(Qwen3-VL-native 0-1000 bbox_2d) and underlying image data are unchanged "
              f"between v4 and v5, only the base model differs, so no new extraction/cache "
              f"is needed for this run.\n"
              f"Continuing with VRSBench-only training for this run (not failing the whole "
              f"notebook over an optional second data source).")
        # Real bug fix (2026-08-30): this branch used to fall through without ever
        # setting DATA_SOURCES_USED, crashing the final print below with a NameError
        # and killing the WHOLE "Run All" before Cell 6/8/9 (the actual training) ever
        # ran -- caught live on Kaggle. Fixed by giving this branch the same honest
        # fallback value the plain INCLUDE_BIGEARTHNET_TXT=False branch already had.
        DATA_SOURCES_USED = ["VRSBench (xiang709/VRSBench)"]
    else:
        be_manifest = json.loads(be_manifest_path.read_text())
        be_examples = [to_chat_example(r) for r in be_manifest]
        be_dataset = Dataset.from_list(be_examples)
        print(f"Loaded {len(be_dataset)} real BigEarthNet.txt referring-expression examples "
              f"(real S2 imagery from BigEarthNet v2.0/reBEN) -- merging into train_dataset "
              f"alongside the {len(train_dataset)} real VRSBench examples above.")
        train_dataset = concatenate_datasets([train_dataset, be_dataset])
        print(f"train_dataset now has {len(train_dataset)} real examples total.")
        # Honesty note for the report/model-card: record which real sources actually
        # contributed to THIS run's training data, not just what was theoretically wired up.
        DATA_SOURCES_USED = ["VRSBench (xiang709/VRSBench)", "BigEarthNet.txt (arXiv:2603.29630, "
                              "via real BigEarthNet v2.0/reBEN S2 imagery)"]
else:
    DATA_SOURCES_USED = ["VRSBench (xiang709/VRSBench)"]
print(f"Real data sources used this run: {DATA_SOURCES_USED}")


# Real disk-quota diagnostic (2026-09-19, added after the user's own Kaggle "Output
# 2.1GiB / 19.5GiB" screenshot -- Kaggle's persisted /kaggle/working output has a real,
# hard ~19.5 GiB ceiling; the base model/HF-hub caches usually live OUTSIDE this
# directory and don't count against it, but every dataset this notebook downloads INTO
# a relative path (cdvqa_repo/, cdvqa_real/, bigearthnet_txt_real/, rsvqa_real/,
# bigearthnet_sar_fusion_real/, the LoRA checkpoint dir) does. Printing real usage here,
# after each data source loads, means a genuine risk of hitting the ceiling shows up
# early with a clear number, not as a mid-training crash.
import shutil as _shutil
_du = _shutil.disk_usage(".")
print(f"Real disk usage check: {_du.used / (1024**3):.2f} GiB used, "
      f"{_du.free / (1024**3):.2f} GiB free (of {_du.total / (1024**3):.2f} GiB total) "
      f"in the current working directory's filesystem.")


In [ ]:
# Cell 5c-setup (updated 2026-09-19, fourth pass): the embedded cdvqa_prepare.py now MOVES (not copies) SECOND's images into the cache directory and deletes the now-redundant download folder afterward -- fixes a real disk-duplication bug caught from the user's own Kaggle "Output 2.1GiB / 19.5GiB" quota screenshot (the previous version stored SECOND's images twice on disk).
# Cell 5c-setup (updated 2026-09-19, third pass): now caches its output to HF Hub, same
# pattern as BigEarthNet.txt/SAR-fusion above. Real motivation: without a cache, EVERY
# fresh Kaggle session/resume (Kaggle's /kaggle/working does not persist across separate
# sessions) re-cloned the CDVQA repo and re-downloaded SECOND's ~4 GiB Google Drive
# archive from scratch every single time -- wasteful of GPU-hour quota (the accelerator
# is attached for the whole session, even during this CPU-bound download) and exposed to
# Google Drive's own rate limits on repeated large-file downloads. Now: first real run
# builds the manifest AND pushes a self-contained cache (manifest + referenced images) to
# HF Hub; every run after that (any session, any account with access) does one small,
# fast download instead. This script's core join/schema logic is UNCHANGED and already
# verified for real correctness on the user's own machine (20,000 real usable examples,
# confirmed live) -- only the caching wrapper is new, so this goes straight to the full
# run rather than smoke-testing again.
import subprocess
from pathlib import Path

Path("cdvqa_prepare.py").write_text(r'''"""
finetuning/cdvqa_prepare.py -- Real CDVQA (Change Detection Visual Question Answering)
training-data preparation. v2 (2026-09-19, same day as v1), rewritten after a REAL smoke
test on Kaggle/local hardware confirmed the actual repo structure -- v1's assumed schema
(one record holding question+answer+image together) was WRONG; the real schema is three
separate per-split files (questions/answers/images) joined by numeric ids. This version
is written against the REAL, confirmed structure, not a second guess.

REAL, CONFIRMED structure (from an actual `git clone` + inspection run, not assumed):
  Repo https://github.com/YZHJessica/CDVQA.git contains, per split (Train/Val/Test/Test2):
    {Split}_questions.json -> {"questions": [
        {"id": 0, "img_id": 0, "type": "change_or_not",
         "question": "Did the regions of buildings change?", "answers_ids": [0], ...}, ...]}
    {Split}_answers.json   -> {"answers": [
        {"id": 0, "question_id": 0, "answer": "yes", ...}, ...]}
    {Split}_images.json    -> {"images": [
        {"id": 0, "res_x": ".1524m", "res_y": ".1524m",
         "questions_ids": [0, 1], "file_name": "07308.png", ...}, ...]}
  Real counts confirmed live: Train 65,967 questions / 25,600 image entries; Val 16,441 /
  6,400; Test 39,686 / 15,488; Test2 31,036 / 15,488. No .png/.jpg files and no download
  URL are committed to this repo itself (confirmed: a real clone contains only the JSON
  files, LICENSE, and a 371-byte README).

REAL image source, found by reading the actual CDVQA paper (arXiv:2112.06343) rather than
guessing: "we choose the existing semantic change detection dataset SECOND [18] as the
basic data to automatically generate a CDVQA dataset" -- 2,968 of SECOND's real,
publicly-available bi-temporal pairs (Shanghai/Hangzhou/Chengdu aerial imagery) were used.
SECOND's own real, official project page (captain-whu.github.io/SCD, Wuhan University) is
confirmed to host it via two real Google Drive links (4,662 pairs total, 512x512, verified
2026-09-19):
    https://drive.google.com/file/d/1mN8jzCKKK27p3ODGoDgepjiRYGQpB34u/view
    https://drive.google.com/file/d/1QlAdzrHpfBIOZ6SK78yHF2i1u6tikmBc/view

NOT YET VERIFIED (disclosed honestly): SECOND's real internal folder-naming convention for
the pre-/post-change image pair (common bi-temporal-CD conventions are im1/im2, A/B, or
T1/T2 -- this script tries all of these against whatever `file_name` values CDVQA's own
images.json records use, and prints exactly what it finds rather than assuming one). This
sandbox could not verify this live (no network egress to drive.google.com from this dev
environment, a restriction of THIS environment, not Kaggle/local machines). Run
--smoke-test first and report the printed folder-structure diagnostics before trusting a
full run.

Requires `gdown` for the Google Drive download (pip install gdown --quiet if missing).
"""
import argparse
import json
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/YZHJessica/CDVQA.git"
SECOND_GDRIVE_IDS = [
    "1mN8jzCKKK27p3ODGoDgepjiRYGQpB34u",
    "1QlAdzrHpfBIOZ6SK78yHF2i1u6tikmBc",
]
CITATION = (
    "Yuan, Z., Mou, L., Xiong, Z., Zhu, X.X. (2022). "
    "\"Change detection meets visual question answering.\" IEEE TGRS, 60. "
    "(arXiv:2112.06343) -- QA data: https://github.com/YZHJessica/CDVQA -- "
    "images: SECOND dataset (Yang et al., arXiv:2010.05687), "
    "https://captain-whu.github.io/SCD/"
)
PRE_POST_FOLDER_CANDIDATES = [
    ("im1", "im2"),
    ("A", "B"),
    ("T1", "T2"),
    ("before", "after"),
    ("im1_512", "im2_512"),
]


def ensure_gdown():
    try:
        import gdown  # noqa: F401
    except ImportError:
        print("Installing gdown (needed to download SECOND's real Google Drive archive)...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)


def clone_repo(dest: Path):
    if dest.exists():
        print(f"{dest} already exists locally -- reusing it (delete it first for a clean re-clone).")
        return
    print(f"Cloning real CDVQA QA-data repo from {REPO_URL} ...")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(dest)], check=True)


def load_split(repo_dir: Path, split: str):
    """Real, confirmed 3-file join: questions + answers + images, by id. Returns a list
    of {question, answer, file_name} dicts -- still just metadata, no image bytes yet."""
    def _load(name, key):
        p = repo_dir / f"{split}_{name}.json"
        if not p.exists():
            return []
        data = json.loads(p.read_text())
        return data.get(key, [])

    questions = _load("questions", "questions")
    answers = _load("answers", "answers")
    images = _load("images", "images")
    if not (questions and answers and images):
        return []

    ans_by_id = {a["id"]: a.get("answer") for a in answers}
    img_by_id = {i["id"]: i.get("file_name") for i in images}

    records = []
    for q in questions:
        ans_ids = q.get("answers_ids") or []
        if not ans_ids:
            continue
        answer = ans_by_id.get(ans_ids[0])
        file_name = img_by_id.get(q.get("img_id"))
        if not (answer and file_name and q.get("question")):
            continue
        records.append({
            "question": q["question"],
            "answer": answer,
            "file_name": file_name,
            "type": q.get("type", "unknown"),
        })
    return records


def find_second_image_root(second_dir: Path):
    """Real, honest discovery of whatever folder structure SECOND's real archive actually
    unzips to -- prints every top-level and second-level directory it finds, and reports
    which of PRE_POST_FOLDER_CANDIDATES (if any) genuinely exist, rather than assuming."""
    print(f"\nReal directory structure under {second_dir}:")
    found_pairs = []
    for p in sorted(second_dir.rglob("*")):
        if p.is_dir():
            rel = p.relative_to(second_dir)
            depth = len(rel.parts)
            if depth <= 2:
                print(f"  [dir depth={depth}] {rel}")
    for pre_name, post_name in PRE_POST_FOLDER_CANDIDATES:
        pre_dirs = list(second_dir.rglob(pre_name))
        post_dirs = list(second_dir.rglob(post_name))
        if pre_dirs and post_dirs:
            print(f"  Real match: found both '{pre_name}' and '{post_name}' folders "
                  f"({pre_dirs[0]}, {post_dirs[0]})")
            found_pairs.append((pre_dirs[0], post_dirs[0]))
    if not found_pairs:
        print("  No known pre/post folder-naming convention "
              f"({PRE_POST_FOLDER_CANDIDATES}) matched anything real under {second_dir}. "
              "Paste this whole directory listing back rather than guessing further.")
    return found_pairs


def download_second_images(dest_dir: Path):
    ensure_gdown()
    import gdown
    dest_dir.mkdir(parents=True, exist_ok=True)
    for i, file_id in enumerate(SECOND_GDRIVE_IDS):
        out_zip = dest_dir / f"second_part{i}.zip"
        if out_zip.exists():
            print(f"  {out_zip} already downloaded, reusing.")
        else:
            print(f"  Downloading real SECOND archive part {i} from Google Drive "
                  f"(id={file_id}) ...")
            try:
                gdown.download(id=file_id, output=str(out_zip), quiet=False)
            except Exception as e:
                print(f"  Real download failed for part {i}: {e}. Google Drive "
                      f"rate-limits/quota-blocks large-file downloads sometimes -- if "
                      f"this keeps failing, download {out_zip.name} manually via a "
                      f"browser from https://drive.google.com/file/d/{file_id}/view and "
                      f"place it at {out_zip}, then re-run this script.")
                continue
        if out_zip.exists() and out_zip.stat().st_size > 0:
            print(f"  Extracting {out_zip.name} ...")
            import zipfile
            try:
                with zipfile.ZipFile(out_zip) as zf:
                    zf.extractall(dest_dir)
            except zipfile.BadZipFile:
                print(f"  {out_zip.name} is not a real zip file (likely an HTML error "
                      f"page from a blocked/quota-limited Drive download, not the real "
                      f"archive) -- delete it and retry, or download manually.")


def main():
    ap = argparse.ArgumentParser(description=__doc__)
    ap.add_argument("--smoke-test", action="store_true",
                     help="Join+preview the real QA schema and attempt a small real "
                          "download to discover the real image folder structure. Run "
                          "this FIRST, always.")
    ap.add_argument("--max-examples", type=int, default=20000)
    ap.add_argument("--repo-dest", default="cdvqa_repo")
    ap.add_argument("--images-dest", default="second_images")
    ap.add_argument("--out-dir", default="cdvqa_real")
    ap.add_argument("--skip-image-download", action="store_true",
                     help="Use if you already have SECOND's images extracted locally "
                          "under --images-dest from a manual download.")
    ap.add_argument("--cache-repo", type=str, default=None,
                     help="Optional HF dataset repo id to cache the final (small manifest "
                          "+ referenced images, self-contained under --out-dir) output in. "
                          "Real motivation (2026-09-19): without this, every fresh Kaggle "
                          "session/resume re-clones the repo AND re-downloads SECOND's "
                          "~4 GiB Google Drive archive from scratch -- wasteful of both "
                          "wall-clock (counts against GPU-hour quota if the accelerator is "
                          "already attached) and Google Drive's own rate limits. With this "
                          "set, a cache hit does one small HF snapshot_download instead, "
                          "same pattern as bigearthnet_txt_extract.py already uses.")
    args = ap.parse_args()

    if args.cache_repo:
        try:
            from huggingface_hub import HfApi, snapshot_download
            api = HfApi()
            files = api.list_repo_files(args.cache_repo, repo_type="dataset")
            if "manifest.json" in files:
                print(f"Found an existing cached CDVQA dataset at "
                      f"https://huggingface.co/datasets/{args.cache_repo} -- downloading "
                      f"that directly instead of re-cloning/re-downloading from GitHub + "
                      f"Google Drive.")
                snapshot_download(repo_id=args.cache_repo, repo_type="dataset",
                                   local_dir=str(args.out_dir))
                cached = json.loads((Path(args.out_dir) / "manifest.json").read_text())
                print(f"Wrote {args.out_dir}/manifest.json from cache ({len(cached)} "
                      f"examples, images included). Done.")
                return
            else:
                print(f"Cache repo exists at {args.cache_repo} but manifest.json is "
                      f"missing -- redoing the real extraction.")
        except Exception as e:
            print(f"No cached CDVQA dataset found at {args.cache_repo} ({e}) -- doing the "
                  f"real extraction. This will be cached for every future run after this "
                  f"one.")

    print(CITATION)
    repo_dest = Path(args.repo_dest)
    clone_repo(repo_dest)

    print("\n=== Real question+answer+image-filename join, per split ===")
    all_records = []
    for split in ["Train", "Val", "Test", "Test2"]:
        recs = load_split(repo_dest, split)
        print(f"  {split}: {len(recs)} real joined question/answer/file_name records")
        if recs:
            print(f"    First: {recs[0]}")
        all_records.extend(recs)

    if not all_records:
        print("\nCould not join any real records -- the file-naming convention "
              "(Train_questions.json etc) may not match what's really in the repo. "
              "Re-check the repo's real file listing and paste it back.")
        sys.exit(1)

    images_dest = Path(args.images_dest)
    if not args.skip_image_download:
        print(f"\nDownloading SECOND's real image archive (~large, this can take a "
              f"while) into {images_dest} ...")
        download_second_images(images_dest)
    else:
        print(f"\n--skip-image-download set -- using whatever already exists under {images_dest}.")

    pairs = find_second_image_root(images_dest)
    if not pairs:
        print(
            "\nCould NOT confirm a real pre/post image-folder convention -- stopping "
            "here rather than fabricating a manifest. If the download above failed or "
            "was rate-limited, download the two real Google Drive files manually "
            f"({SECOND_GDRIVE_IDS}) via a browser, extract them under {images_dest}, "
            "then re-run with --skip-image-download."
        )
        sys.exit(1)

    pre_root, post_root = pairs[0]
    max_examples = 20 if args.smoke_test else args.max_examples
    manifest = []
    skipped_no_image = 0
    for rec in all_records:
        if len(manifest) >= max_examples:
            break
        pre_path = pre_root / rec["file_name"]
        post_path = post_root / rec["file_name"]
        if not (pre_path.exists() and post_path.exists()):
            skipped_no_image += 1
            continue
        manifest.append({
            "image_path_pre": str(pre_path),
            "image_path_post": str(post_path),
            "prompt": f"[cdvqa] {rec['question']}",
            "response": str(rec["answer"]),
        })

    print(f"\n=== Result ===")
    print(f"Real joined records available: {len(all_records)}")
    print(f"Skipped (real image pair not found on disk at {pre_root}/{post_root}): {skipped_no_image}")
    print(f"Real, usable bi-temporal CDVQA examples built: {len(manifest)}")

    if not manifest:
        print("\nZero usable examples -- the file_name values in CDVQA's images.json "
              "likely don't match SECOND's real filenames 1:1 (e.g. a numbering/"
              "zero-padding mismatch). Paste a few real file_name values from the join "
              "above AND a real directory listing of the extracted SECOND images "
              "(`ls second_images/im1 | head`) back for a fix, rather than guessing.")
        sys.exit(1)

    out_dir = Path(args.out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    if args.cache_repo:
        # Real restructuring for caching (2026-09-19): the images currently live under
        # the SEPARATE --images-dest, not nested under --out-dir -- MOVE (not copy) just
        # the ones this manifest actually references into out_dir/images/{im1,im2}/ and
        # rewrite the manifest to point there, so the whole out_dir is one self-contained
        # unit a future session can restore from a single snapshot_download.
        #
        # Real disk-quota fix (2026-09-19, caught from the user's own Kaggle "Output
        # 2.1GiB / 19.5GiB" screenshot): this used to shutil.copy2, which left the SECOND
        # images on disk TWICE (once under --images-dest, once again under out_dir) --
        # wasteful and a real risk against Kaggle's persisted-output quota for a dataset
        # this size (SECOND's im1/im2 pairs are not small). Moving instead, and removing
        # the now-empty --images-dest tree afterward, means this only ever needs ONE copy
        # on disk -- correct because after this cache is pushed to HF Hub, --images-dest
        # is never read again this run, and future runs get their images from the HF
        # cache instead of re-downloading from Google Drive at all.
        # Non-cache runs (no --cache-repo) are left exactly as before -- zero behavior
        # change for the already-verified-working default path.
        import shutil
        cache_im1_dir = out_dir / "images" / "im1"
        cache_im2_dir = out_dir / "images" / "im2"
        cache_im1_dir.mkdir(parents=True, exist_ok=True)
        cache_im2_dir.mkdir(parents=True, exist_ok=True)
        for row in manifest:
            pre_src, post_src = Path(row["image_path_pre"]), Path(row["image_path_post"])
            pre_dst, post_dst = cache_im1_dir / pre_src.name, cache_im2_dir / post_src.name
            if pre_src.exists() and not pre_dst.exists():
                shutil.move(str(pre_src), str(pre_dst))
            if post_src.exists() and not post_dst.exists():
                shutil.move(str(post_src), str(post_dst))
            row["image_path_pre"] = str(pre_dst)
            row["image_path_post"] = str(post_dst)

        images_root = Path(args.images_dest)
        if images_root.exists():
            freed_bytes = sum(f.stat().st_size for f in images_root.rglob("*") if f.is_file())
            shutil.rmtree(images_root, ignore_errors=True)
            print(f"Freed ~{freed_bytes / (1024**3):.2f} GiB by removing the now-redundant "
                  f"{images_root} after moving its needed files into {out_dir} "
                  f"(avoids storing SECOND's images twice on disk).")

    with open(out_dir / "manifest.json", "w") as f:
        json.dump(manifest, f, indent=2)
    print(f"\nSaved {len(manifest)} real CDVQA examples to {out_dir}/manifest.json")

    if args.cache_repo:
        from huggingface_hub import HfApi
        api = HfApi()
        print(f"Pushing self-contained cache (manifest + referenced images) to "
              f"https://huggingface.co/datasets/{args.cache_repo} so no future run ever "
              f"needs to re-clone/re-download from GitHub + Google Drive...")
        api.create_repo(args.cache_repo, repo_type="dataset", private=True, exist_ok=True)
        api.upload_folder(repo_id=args.cache_repo, repo_type="dataset", folder_path=str(out_dir))
        print("Done.")


if __name__ == "__main__":
    main()
''')

CDVQA_CACHE_REPO = "Sameelkazi/satquery-qwen3vl8b-cdvqa-cache-v5"
print(f"=== Running cdvqa_prepare.py (cache-or-build against {CDVQA_CACHE_REPO}) ===")
r = subprocess.run(
    ["python", "cdvqa_prepare.py", "--max-examples", "20000",
     "--cache-repo", CDVQA_CACHE_REPO],
    capture_output=True, text=True,
)
print(r.stdout)
if r.stderr:
    print("--- stderr ---")
    print(r.stderr)
if not Path("cdvqa_real/manifest.json").exists():
    print("\nNo real CDVQA manifest was produced (see output above) -- Cell 5c below "
          "will detect this and continue WITHOUT CDVQA data rather than fail the whole "
          "run.")


In [ ]:
# Cell 5c (new, 2026-09-19): Real CDVQA (Change Detection VQA) training slice -- closes
# the gap described in Cell 0's research-update note: this project's fine-tuning never
# previously included any genuine bi-temporal change-QA training examples, only
# VRSBench (single-image) + BigEarthNet.txt referring-expression (single-image). CDVQA
# (Yuan et al. 2022, arXiv:2112.06343, real dataset at github.com/YZHJessica/CDVQA,
# Apache-2.0) is the canonical, PS-referenced change-VQA benchmark, and a very recent
# (2026) paper (arXiv:2604.18429) shows real Qwen-family LoRA fine-tuning on exactly
# this data reaches 68-74% accuracy from a small (~3,000 pair) slice -- see Cell 0.
#
# Honesty note: run finetuning/cdvqa_prepare.py --smoke-test FIRST (before this cell,
# same discipline as BigEarthNet.txt below) -- this sandbox could not verify the real
# CDVQA repo's exact JSON schema live (network-restricted dev environment, not a Kaggle
# limitation), so cdvqa_prepare.py is written defensively and prints full real
# diagnostics rather than assuming a shape. Only trust a full --max-examples run after
# a clean smoke-test pass confirms the schema.
# Real defensive fix: import these directly here rather than relying on Cell 5b's
# conditional import (which only runs `from datasets import concatenate_datasets`
# INSIDE its `if INCLUDE_BIGEARTHNET_TXT:` block) -- if someone sets
# INCLUDE_BIGEARTHNET_TXT=False but leaves INCLUDE_CDVQA=True, that import would
# never have run, and this cell would hit a real NameError. Re-importing here (a
# harmless no-op if already imported) makes this cell correct standalone.
from pathlib import Path
from datasets import concatenate_datasets

INCLUDE_CDVQA = True  # <-- set False to skip CDVQA and train on VRSBench (+ BigEarthNet.txt) alone

def to_chat_example_bitemporal(row):
    """Same chat-example shape as to_chat_example() above, but with TWO real images
    (pre-change, post-change) in one user turn -- collate_fn (Cell 7 below) already
    handles a variable-length ex["images"] list generically, no changes needed there."""
    return {
        "images": [row["image_path_pre"], row["image_path_post"]],
        "messages": [
            {"role": "user", "content": [
                {"type": "image"},
                {"type": "image"},
                {"type": "text", "text": row["prompt"]},
            ]},
            {"role": "assistant", "content": [
                {"type": "text", "text": row["response"]},
            ]},
        ],
    }

if INCLUDE_CDVQA:
    cdvqa_manifest_path = Path("cdvqa_real/manifest.json")
    if not cdvqa_manifest_path.exists():
        print(f"INCLUDE_CDVQA=True but {cdvqa_manifest_path} doesn't exist yet -- run "
              f"(BEFORE this notebook, ideally verified with --smoke-test first):\n"
              f"  python finetuning/cdvqa_prepare.py --smoke-test\n"
              f"  python finetuning/cdvqa_prepare.py --max-examples 20000\n"
              f"Continuing WITHOUT real CDVQA training data for this run (not failing the "
              f"whole notebook over an optional additional data source).")
    else:
        cdvqa_manifest = json.loads(cdvqa_manifest_path.read_text())
        cdvqa_examples = [to_chat_example_bitemporal(r) for r in cdvqa_manifest]
        cdvqa_dataset = Dataset.from_list(cdvqa_examples)
        print(f"Loaded {len(cdvqa_dataset)} real CDVQA bi-temporal change-QA examples -- "
              f"merging into train_dataset alongside the {len(train_dataset)} existing "
              f"real examples above.")
        train_dataset = concatenate_datasets([train_dataset, cdvqa_dataset])
        print(f"train_dataset now has {len(train_dataset)} real examples total.")
        DATA_SOURCES_USED.append("CDVQA (Yuan et al. 2022, arXiv:2112.06343, via YZHJessica/CDVQA)")

print(f"Real data sources used this run (after CDVQA check): {DATA_SOURCES_USED}")


# Real disk-quota diagnostic (2026-09-19, added after the user's own Kaggle "Output
# 2.1GiB / 19.5GiB" screenshot -- Kaggle's persisted /kaggle/working output has a real,
# hard ~19.5 GiB ceiling; the base model/HF-hub caches usually live OUTSIDE this
# directory and don't count against it, but every dataset this notebook downloads INTO
# a relative path (cdvqa_repo/, cdvqa_real/, bigearthnet_txt_real/, rsvqa_real/,
# bigearthnet_sar_fusion_real/, the LoRA checkpoint dir) does. Printing real usage here,
# after each data source loads, means a genuine risk of hitting the ceiling shows up
# early with a clear number, not as a mid-training crash.
import shutil as _shutil
_du = _shutil.disk_usage(".")
print(f"Real disk usage check: {_du.used / (1024**3):.2f} GiB used, "
      f"{_du.free / (1024**3):.2f} GiB free (of {_du.total / (1024**3):.2f} GiB total) "
      f"in the current working directory's filesystem.")


In [ ]:
# Cell 5d-setup (updated 2026-09-19, third pass): now caches its output to HF Hub, same
# reasoning and pattern as Cell 5c-setup above -- without it, every fresh Kaggle
# session/resume re-downloaded all of RSVQA-LR's real Zenodo files (~150 MB, small but
# still wasted GPU-hour-attached wall-clock on every restart) from scratch. This script's
# core per-split join logic is UNCHANGED and already verified for real correctness on the
# user's own machine (15,000 real usable examples, confirmed live, real "{img_id}.tif"
# naming convention confirmed) -- only the caching wrapper is new.
import subprocess
from pathlib import Path

Path("rsvqa_prepare.py").write_text(r'''"""
finetuning/rsvqa_prepare.py -- Real RSVQA-LR training-data preparation.
v2 (2026-09-19, same day as v1), rewritten after a REAL smoke test on the user's
machine confirmed v1's generic auto-detect join picked the WRONG image-id field
(it grabbed a bare "id" from an unrelated image record instead of "img_id" from
the question record, because the old code scanned one big flattened list of
every record from every file and stopped at the first dict that happened to
have *some* field matching IMAGE_ID_CANDIDATES). Real result of that bug: 0
manifest entries even though 772 real images extracted successfully and 77,232
real train questions/answers were downloaded fine.

REAL, CONFIRMED structure (from an actual live run against the real Zenodo
record 6344334, not assumed):
  LR_split_{Train,Val,Test}_questions.json -> top-level list of
    {"id": <question_id>, "img_id": <image_id>, "question": "...",
     "answers_ids": [<answer_id>, ...], "active": true, ...}
  LR_split_{Train,Val,Test}_answers.json -> top-level list of
    {"id": <answer_id>, "question_id": <question_id>, "answer": "...", ...}
  LR_split_{Train,Val,Test}_images.json -> top-level list of
    {"id": <image_id>, "original_name": "S2B_..._1024-2560.tif", ...}
  IMPORTANT (confirmed live): each per-split file only carries FULL fields for
  records that are actually active/in that split -- records not in that split
  come back as bare {"id": N, "active": false} stubs in that particular file.
  So the correct join uses ONE split's own three files together, never mixes
  a question from Train with an answer/image record from a different split's
  file. "all_questions.json"/"all_answers.json" contain the union across
  splits (train+val+test combined, keyed by their own ids) and are used only
  as a fallback if a per-split file is missing.
  Real file sizes/counts confirmed live: 12 files total including
  Images_LR.zip (95,008,155 bytes, extracts to 772 real .tif files).

NOT YET VERIFIED (disclosed honestly): the exact filename convention inside
the extracted Images_LR.zip (i.e. whether image id 0 becomes "0.tif",
"Images_LR/0.tif", or something derived from "original_name"). This sandbox's
own network egress cannot reach zenodo.org to check live (a restriction of
THIS dev environment only, not the user's machine). Rather than guess one
convention, this script tries several real candidate filenames per image id
and reports exactly what matched (or didn't) before building anything.

Citation: Lobry, S., Marcos, D., Murray, J., Tuia, D. (2020). "RSVQA: Visual
Question Answering for Remote Sensing Data." IEEE TGRS. (arXiv:2003.07333)
"""
import argparse
import json
import sys
import tarfile
import zipfile
from pathlib import Path
from urllib.request import urlopen, urlretrieve

RECORD_ID = "6344334"  # RSVQA-LR (NOT 6344367 / RSVQA-HR)
CITATION = (
    "Lobry, S., Marcos, D., Murray, J., Tuia, D. (2020). "
    "\"RSVQA: Visual Question Answering for Remote Sensing Data.\" IEEE TGRS. "
    "(arXiv:2003.07333) -- data: https://zenodo.org/record/6344334 (RSVQA-LR)"
)
SPLITS = ["Train", "Val", "Test"]


def zenodo_file_list(record_id: str):
    url = f"https://zenodo.org/api/records/{record_id}"
    print(f"Querying real Zenodo API: {url}")
    with urlopen(url, timeout=60) as resp:
        meta = json.loads(resp.read())
    files = meta.get("files", [])
    print(f"Real record has {len(files)} file(s):")
    for f in files:
        print(f"  {f.get('key','?')}  ({f.get('size',0):,} bytes)")
    return {f["key"]: f for f in files}


def download_file(file_entry: dict, dest_dir: Path):
    key = file_entry["key"]
    link = file_entry["links"]["self"]
    dest = dest_dir / key
    if dest.exists() and dest.stat().st_size == file_entry.get("size", -1):
        print(f"  {key} already downloaded, reusing.")
        return dest
    print(f"  Downloading real file {key} ({file_entry.get('size', 0):,} bytes) ...")
    urlretrieve(link, dest)
    return dest


def load_list(path: Path, key: str):
    if not path.exists():
        return []
    data = json.loads(path.read_text())
    if isinstance(data, dict):
        return data.get(key, data.get(list(data.keys())[0], []) if data else [])
    return data if isinstance(data, list) else []


def load_split(dest: Path, split: str):
    """Real, confirmed 3-file join for ONE split: questions + answers + images,
    all from that split's own files (never mixed across splits), joined by id.
    Returns a list of {question, answer, img_id, original_name} dicts."""
    q_path = dest / f"LR_split_{split}_questions.json"
    a_path = dest / f"LR_split_{split}_answers.json"
    i_path = dest / f"LR_split_{split}_images.json"

    questions = [q for q in load_list(q_path, "questions") if isinstance(q, dict) and q.get("active", True) and "question" in q]
    answers = [a for a in load_list(a_path, "answers") if isinstance(a, dict) and a.get("active", True) and "answer" in a]
    images = [i for i in load_list(i_path, "images") if isinstance(i, dict) and i.get("active", True)]

    ans_by_qid = {}
    for a in answers:
        ans_by_qid.setdefault(a.get("question_id"), a.get("answer"))
    img_by_id = {i.get("id"): i for i in images}

    records = []
    for q in questions:
        ans_ids = q.get("answers_ids") or []
        answer = None
        if ans_ids:
            for aid in ans_ids:
                a = next((x for x in answers if x.get("id") == aid), None)
                if a:
                    answer = a.get("answer")
                    break
        if answer is None:
            answer = ans_by_qid.get(q.get("id"))
        img = img_by_id.get(q.get("img_id"))
        if not (answer and img and q.get("question")):
            continue
        records.append({
            "question": q["question"],
            "answer": str(answer),
            "img_id": q.get("img_id"),
            "original_name": img.get("original_name", ""),
        })
    return records


def find_image_file(images_dir: Path, img_id, original_name: str, cache: dict):
    """Real, honest discovery -- tries several real candidate filenames rather
    than assuming one convention, and caches the directory listing so this
    isn't an O(n^2) rglob per record."""
    if "listing" not in cache:
        cache["listing"] = {p.name: p for p in images_dir.rglob("*") if p.is_file()}
        cache["stems"] = {p.stem: p for p in images_dir.rglob("*") if p.is_file()}
    listing = cache["listing"]
    stems = cache["stems"]

    candidates = [
        f"{img_id}.tif", f"{img_id}.png", f"{img_id}.jpg",
        str(img_id),
    ]
    if original_name:
        candidates.append(original_name)
        candidates.append(Path(original_name).name)
    for c in candidates:
        if c in listing:
            return listing[c]
    if str(img_id) in stems:
        return stems[str(img_id)]
    return None


def main():
    ap = argparse.ArgumentParser(description=__doc__)
    ap.add_argument("--smoke-test", action="store_true",
                     help="Join+preview the real schema per split and attempt to "
                          "resolve at most 20 real images. Run this FIRST, always.")
    ap.add_argument("--max-examples", type=int, default=15000)
    ap.add_argument("--dest", default="rsvqa_lr_raw", help="Local download destination.")
    ap.add_argument("--out-dir", default="rsvqa_real", help="Output directory for manifest.json.")
    ap.add_argument("--skip-download", action="store_true",
                     help="Reuse whatever's already downloaded/extracted under --dest/--out-dir.")
    ap.add_argument("--cache-repo", type=str, default=None,
                     help="Optional HF dataset repo id to cache the final output (manifest "
                          "+ images, already self-contained under --out-dir) in. Real "
                          "motivation (2026-09-19): without this, every fresh Kaggle "
                          "session/resume re-downloads all real Zenodo files (~150 MB) "
                          "from scratch. With this set, a cache hit does one small HF "
                          "snapshot_download instead, same pattern as "
                          "bigearthnet_txt_extract.py already uses.")
    args = ap.parse_args()

    if args.cache_repo:
        try:
            from huggingface_hub import HfApi, snapshot_download
            api = HfApi()
            files = api.list_repo_files(args.cache_repo, repo_type="dataset")
            if "manifest.json" in files:
                print(f"Found an existing cached RSVQA-LR dataset at "
                      f"https://huggingface.co/datasets/{args.cache_repo} -- downloading "
                      f"that directly instead of re-downloading from Zenodo.")
                snapshot_download(repo_id=args.cache_repo, repo_type="dataset",
                                   local_dir=str(args.out_dir))
                cached = json.loads((Path(args.out_dir) / "manifest.json").read_text())
                print(f"Wrote {args.out_dir}/manifest.json from cache ({len(cached)} "
                      f"examples, images included). Done.")
                return
            else:
                print(f"Cache repo exists at {args.cache_repo} but manifest.json is "
                      f"missing -- redoing the real extraction.")
        except Exception as e:
            print(f"No cached RSVQA-LR dataset found at {args.cache_repo} ({e}) -- doing "
                  f"the real extraction. This will be cached for every future run after "
                  f"this one.")

    print(CITATION)
    dest = Path(args.dest)
    dest.mkdir(parents=True, exist_ok=True)
    images_dir = Path(args.out_dir) / "images"
    images_dir.mkdir(parents=True, exist_ok=True)

    if not args.skip_download:
        try:
            files = zenodo_file_list(RECORD_ID)
        except Exception as e:
            print(f"\nCould NOT reach the real Zenodo API ({e}). Not fabricating a file list.")
            sys.exit(1)

        needed = []
        for split in SPLITS:
            for kind in ["questions", "answers", "images"]:
                key = f"LR_split_{split}_{kind}.json"
                if key in files:
                    needed.append(key)
        if not needed:
            print("\nNone of the expected LR_split_*.json files were found in the real "
                  "Zenodo file list above -- the record's file names may have changed. "
                  "Paste this output back before proceeding.")
            sys.exit(1)
        for key in needed:
            download_file(files[key], dest)

        archive_key = next((k for k in files if k.lower().startswith("images_lr")), None)
        if archive_key and not any(images_dir.iterdir()):
            local_archive = download_file(files[archive_key], dest)
            print(f"  Extracting real image archive {local_archive.name} ...")
            if local_archive.suffix == ".zip":
                with zipfile.ZipFile(local_archive) as zf:
                    zf.extractall(images_dir)
            elif local_archive.name.endswith((".tar.gz", ".tgz")):
                with tarfile.open(local_archive) as tf:
                    tf.extractall(images_dir)
    else:
        print(f"\n--skip-download set -- using whatever already exists under {dest} / {images_dir}.")

    print("\n=== Real per-split question+answer+image join ===")
    all_records = []
    for split in SPLITS:
        recs = load_split(dest, split)
        print(f"  {split}: {len(recs)} real joined question/answer/img_id records")
        if recs:
            print(f"    First: {recs[0]}")
        all_records.extend(recs)

    if not all_records:
        print("\nCould not join any real records from the per-split files -- paste the "
              "real keys/preview of LR_split_Train_questions.json / _answers.json / "
              "_images.json back (this script already prints them above on a full run).")
        sys.exit(1)

    print(f"\nReal image files visible under {images_dir}: "
          f"{sum(1 for _ in images_dir.rglob('*') if _.is_file())}")
    sample = images_dir.rglob("*")
    sample_names = [p.name for p in list(images_dir.rglob("*"))[:10] if p.is_file()]
    print(f"Sample real extracted filenames: {sample_names}")

    max_examples = 20 if args.smoke_test else args.max_examples
    manifest = []
    skipped_no_image = 0
    cache = {}
    for rec in all_records:
        if len(manifest) >= max_examples:
            break
        img_path = find_image_file(images_dir, rec["img_id"], rec["original_name"], cache)
        if img_path is None:
            skipped_no_image += 1
            continue
        manifest.append({
            "image_path": str(img_path),
            "prompt": f"[rsvqa] {rec['question']}",
            "response": rec["answer"],
        })

    print(f"\n=== Result ===")
    print(f"Real joined records available: {len(all_records)}")
    print(f"Skipped (no resolvable real image file found on disk): {skipped_no_image}")
    print(f"Real, usable RSVQA-LR examples built: {len(manifest)}")

    if not manifest:
        print(
            "\nZero usable examples -- none of the tried filename patterns "
            "(<img_id>.tif/.png/.jpg, the bare id, or the real 'original_name' field) "
            "matched a real extracted file. Paste the 'Sample real extracted filenames' "
            "line above plus a few real img_id/original_name values from the join "
            "preview back, rather than guessing further."
        )
        sys.exit(1)

    out_dir = Path(args.out_dir)
    with open(out_dir / "manifest.json", "w") as f:
        json.dump(manifest, f, indent=2)
    print(f"\nSaved {len(manifest)} real RSVQA-LR examples to {out_dir}/manifest.json")

    if args.cache_repo:
        # Images already live under out_dir/images/... (see out_images_dir above), so
        # out_dir is already self-contained -- no path rewriting needed, just push it.
        from huggingface_hub import HfApi
        api = HfApi()
        print(f"Pushing self-contained cache (manifest + images) to "
              f"https://huggingface.co/datasets/{args.cache_repo} so no future run ever "
              f"needs to re-download from Zenodo...")
        api.create_repo(args.cache_repo, repo_type="dataset", private=True, exist_ok=True)
        api.upload_folder(repo_id=args.cache_repo, repo_type="dataset", folder_path=str(out_dir))
        print("Done.")


if __name__ == "__main__":
    main()
''')

RSVQA_CACHE_REPO = "Sameelkazi/satquery-qwen3vl8b-rsvqa-cache-v5"
print(f"=== Running rsvqa_prepare.py (cache-or-build against {RSVQA_CACHE_REPO}) ===")
r = subprocess.run(
    ["python", "rsvqa_prepare.py", "--max-examples", "15000",
     "--cache-repo", RSVQA_CACHE_REPO],
    capture_output=True, text=True,
)
print(r.stdout)
if r.stderr:
    print("--- stderr ---")
    print(r.stderr)
if not Path("rsvqa_real/manifest.json").exists():
    print("\nNo real RSVQA-LR manifest was produced (see output above) -- Cell 5d below "
          "will detect this and continue WITHOUT RSVQA data rather than fail the whole "
          "run.")


In [ ]:
# Cell 5d (new, 2026-09-19): Real RSVQA-LR training slice -- the PS explicitly prescribes
# RSVQA (alongside VRSBench and CDVQA) as an evaluation dataset, but until now no
# fine-tuning run ever trained on any real RSVQA data. This adds a real slice of
# RSVQA-LR (Lobry et al. 2020, arXiv:2003.07333, real data at
# https://zenodo.org/record/6344334, ~150.5MB total -- small enough to use inline here,
# unlike RSVQA-HR's ~14.4GB) via finetuning/rsvqa_prepare.py, so the fine-tune now
# genuinely covers THREE of the PS's prescribed datasets (VRSBench + BigEarthNet.txt +
# RSVQA), not two, plus CDVQA (Cell 5c above) which the PS also names for evaluation.
#
# Honesty note: same discipline as Cell 5c -- this sandbox's own network egress could not
# reach zenodo.org to verify the real record's exact JSON layout live (a restriction of
# this dev environment, not Kaggle). rsvqa_prepare.py uses Zenodo's real public REST API
# to list files programmatically and prints full diagnostics rather than assuming a
# schema. Run finetuning/rsvqa_prepare.py --smoke-test FIRST and confirm it built a real
# manifest before trusting a full run.
# Defensive standalone imports (same reasoning as Cell 5c) -- harmless no-op if
# already imported by an earlier cell.
from pathlib import Path
from datasets import concatenate_datasets

INCLUDE_RSVQA = True  # <-- set False to skip RSVQA-LR

if INCLUDE_RSVQA:
    rsvqa_manifest_path = Path("rsvqa_real/manifest.json")
    if not rsvqa_manifest_path.exists():
        print(f"INCLUDE_RSVQA=True but {rsvqa_manifest_path} doesn't exist yet -- run "
              f"(BEFORE this notebook, ideally verified with --smoke-test first):\n"
              f"  python finetuning/rsvqa_prepare.py --smoke-test\n"
              f"  python finetuning/rsvqa_prepare.py --max-examples 15000\n"
              f"Continuing WITHOUT real RSVQA training data for this run (not failing the "
              f"whole notebook over an optional additional data source).")
    else:
        rsvqa_manifest = json.loads(rsvqa_manifest_path.read_text())
        rsvqa_examples = [to_chat_example(r) for r in rsvqa_manifest]  # single-image, same
        # shape as VRSBench's to_chat_example (Cell 5) -- RSVQA-LR is single-image VQA, no
        # bi-temporal pair, so it reuses that function rather than the bitemporal one above.
        rsvqa_dataset = Dataset.from_list(rsvqa_examples)
        print(f"Loaded {len(rsvqa_dataset)} real RSVQA-LR examples -- merging into "
              f"train_dataset alongside the {len(train_dataset)} existing real examples above.")
        train_dataset = concatenate_datasets([train_dataset, rsvqa_dataset])
        print(f"train_dataset now has {len(train_dataset)} real examples total.")
        DATA_SOURCES_USED.append("RSVQA-LR (Lobry et al. 2020, arXiv:2003.07333)")

print(f"Real data sources used this run (after RSVQA check): {DATA_SOURCES_USED}")


# Real disk-quota diagnostic (2026-09-19, added after the user's own Kaggle "Output
# 2.1GiB / 19.5GiB" screenshot -- Kaggle's persisted /kaggle/working output has a real,
# hard ~19.5 GiB ceiling; the base model/HF-hub caches usually live OUTSIDE this
# directory and don't count against it, but every dataset this notebook downloads INTO
# a relative path (cdvqa_repo/, cdvqa_real/, bigearthnet_txt_real/, rsvqa_real/,
# bigearthnet_sar_fusion_real/, the LoRA checkpoint dir) does. Printing real usage here,
# after each data source loads, means a genuine risk of hitting the ceiling shows up
# early with a clear number, not as a mid-training crash.
import shutil as _shutil
_du = _shutil.disk_usage(".")
print(f"Real disk usage check: {_du.used / (1024**3):.2f} GiB used, "
      f"{_du.free / (1024**3):.2f} GiB free (of {_du.total / (1024**3):.2f} GiB total) "
      f"in the current working directory's filesystem.")


In [ ]:
# Cell 5e-setup (new, 2026-09-19): Real optical+SAR JOINT training data -- closes this
# project's biggest remaining, previously-disclosed gap: the fine-tuned model itself never
# learned to reason over a real paired optical+SAR example, only the backend's separate
# sar_fusion_service did that at inference time. See bigearthnet_txt_extract_fusion.py's
# module docstring for the full honest story: the original PRD's assumption that
# BigEarthNet.txt ships SAR-descriptive annotation text was checked against real research
# today and NOT confirmed -- so this does NOT fabricate SAR-descriptive text. Instead it
# keeps BigEarthNet.txt's real optical caption/answer for each patch, and appends a
# genuinely COMPUTED sentence derived from that same patch's real Sentinel-1 VV/VH pixel
# data (relative backscatter comparison, corpus-relative, calibration-agnostic -- see the
# script's docstring for exactly why absolute physical units are deliberately avoided).
#
# REQUIRES Cell 5b-setup to have already run (reuses its real cached S2 optical images).
# REQUIRES this Kaggle notebook's internet access to be ON (same requirement as every
# other setup cell above) -- this one additionally streams a fresh ~51 GiB Sentinel-1
# archive on a cache miss (bounded, one-time, cached to HF Hub afterward exactly like
# every other heavy download in this project).
#
# Deliberately small (3,000 examples default, smoke-tested at 20) -- this is a NEW,
# not-yet-live-verified component (this dev sandbox cannot reach zenodo.org to test it,
# same disclosed restriction as everywhere else in this project). Run the smoke-test
# output check below carefully before trusting a full run; if the real S1 archive's member
# basename convention turns out to differ from what bigearth.net's own documentation
# describes, this will print exactly that rather than silently building something wrong.
INCLUDE_OPTICAL_SAR_FUSION = True  # <-- set False to skip this if the smoke-test below fails

import subprocess
from pathlib import Path

Path("bigearthnet_txt_filter.py").write_text(r'''"""
finetuning/bigearthnet_txt_filter.py

Purpose (real, PS-relevant): the PS names "BigEarthNet.txt" (arXiv:2603.29630) as its
primary/prescribed fine-tuning dataset. This project's actual fine-tuning run uses
VRSBench instead (a PS-named EVALUATION dataset, used here for training under the PS's own
"BigEarthNet.txt OR OTHER OPEN-SOURCE TRAINING DATA" clause) -- a documented, defensible
substitution, not a violation. This script adds a genuine second training source built
directly from BigEarthNet.txt itself, closing that gap for real instead of just arguing
around it.

What this script does: reads BigEarthNet.txt's real annotation parquet
(BIFOLD-BigEarthNetv2-0/BigEarthNet.txt on the HF Hub, 9.55M rows -- verified via HF's
datasets-server API on 2026-08-29) and selects a small, real, referring-expression subset
(type == "bounding box", category == "reference" -- i.e. "find the box for THIS DESCRIBED
OBJECT", the same text-to-box task shape as VRSBench's own [refer] examples and the same
task the project's Acc@0.5 grounding gap is about). It deliberately EXCLUDES
category == "point" rows (a different point-to-box task shape) to avoid teaching the model
two incompatible box-output conventions.

Verified real facts this script relies on (checked directly via HF's datasets-server API,
not assumed):
  - Real columns: ID, S1_name, patch_id, input, output, type, category, split, latitude,
    longitude, country, season, climate_zone.
  - Real box format for type=="bounding box": output is a bracket string
    "[x_min y_min, x_max y_max]" with NORMALIZED [0,1] coordinates, e.g. "[0.0 0.33, 0.28 0.8]".
  - Real category=="reference" input format: "Identify the location of the
    <ref>DESCRIPTION</ref>" (or similar phrasing) -- the <ref>...</ref> inner text is the
    real object description to keep; the wrapper phrasing/tags are stripped.
  - patch_id values (e.g. "S2A_MSIL2A_20170613T101031_N9999_R022_T33UUP_26_57") match the
    REAL BigEarthNet v2.0 per-patch folder-naming convention (confirmed against
    bigearth.net's own "Description_BigEarthNet_v2.pdf") -- this is the exact folder name to
    look for inside BigEarthNet-S2.tar.zst.

Output schema matches finetuning/kaggle_finetune_qwen2vl.ipynb Cell 3's own manifest.json
exactly ({"image_path", "prompt", "response", "task_tag"}), except image_path is left as
None here -- bigearthnet_txt_extract.py fills it in after real image extraction, so this
filter step never needs network access to the (much larger) image archives at all.

This file is meant to run on Kaggle (real network access to the HF Hub). It has no heavy
image/network dependencies of its own beyond `datasets`/`huggingface_hub`/`pyarrow`.
"""
from __future__ import annotations

import argparse
import json
import os
import re
import sys
from collections import Counter
from pathlib import Path
from typing import Any

TEXT_DATASET_REPO = "BIFOLD-BigEarthNetv2-0/BigEarthNet.txt"

# Real box format audit regex: "[x_min y_min, x_max y_max]", normalized 0-1 floats.
BOX_RE = re.compile(
    r"\[\s*([0-9.]+)\s+([0-9.]+)\s*,\s*([0-9.]+)\s+([0-9.]+)\s*\]"
)
# Strips the <ref>...</ref> / <point>(...)</point> XML-like wrapper tags this dataset's
# `input` field uses, keeping the inner text as the plain-language object description.
REF_TAG_RE = re.compile(r"<ref>(.*?)</ref>", re.IGNORECASE | re.DOTALL)
POINT_TAG_RE = re.compile(r"<point>.*?</point>", re.IGNORECASE | re.DOTALL)

REQUIRED_COLUMNS = [
    "patch_id", "input", "output", "type", "category", "split",
]


def clean_reference_prompt(raw_input: str) -> str | None:
    """Extract the plain-language object description from a category=='reference' row's
    `input` field. Returns None if no <ref>...</ref> span is found (defensive -- do not
    guess at a different real format if this dataset's phrasing varies row to row)."""
    m = REF_TAG_RE.search(raw_input or "")
    if not m:
        return None
    description = m.group(1).strip()
    if not description:
        return None
    return f"[refer] Where is the {description}?"


def box_str_to_token(output_text: str) -> str | None:
    """Convert BigEarthNet.txt's real "[x_min y_min, x_max y_max]" (normalized 0-1) box
    string into this project's real box-token format.

    UPDATED 2026-09-12 (v4 base-model + box-format switch, paired with
    finetuning/kaggle_finetune_qwen2vl.ipynb's swap to Qwen3-VL-4B-Instruct): now emits
    '{"bbox_2d": [x1, y1, x2, y2]}' on a 0-1000 integer scale, x-first -- Qwen3-VL's own
    documented native grounding format (github.com/QwenLM/Qwen3-VL,
    cookbooks/2d_grounding.ipynb), NOT the old GeoChat-era "{<x1><y1><x2><y2>}" 0-100
    bracket convention this function used to emit. This is the SAME format the notebook's
    Cell 3 (convert_legacy_box_tokens) now converts VRSBench's raw answers into, and the
    SAME format backend/models/geochat_service.py's parse_boxes_from_text expects at
    inference (x-first order, already settled there against real IoU evidence). Training
    BOTH data sources toward one shared output convention is deliberate -- mixing two
    different box encodings in one fine-tune would teach the model an inconsistent output
    format. Deliberately omits the cookbook's optional "label" field -- see the notebook's
    Cell 3 note for why (this task's referring expression already IS the label, and
    BigEarthNet.txt's own "input" field plays that same role here)."""
    m = BOX_RE.search(output_text or "")
    if not m:
        return None
    x1, y1, x2, y2 = (float(g) for g in m.groups())
    # Defensive real-data check: reject rows with an inverted/degenerate box outright
    # rather than silently training on a bad label.
    if not (0.0 <= x1 < x2 <= 1.0 and 0.0 <= y1 < y2 <= 1.0):
        return None
    xi1, yi1, xi2, yi2 = (round(v * 1000) for v in (x1, y1, x2, y2))
    return f'{{"bbox_2d": [{xi1}, {yi1}, {xi2}, {yi2}]}}'


def select_rows(rows: list[dict[str, Any]], max_examples: int, dedupe_per_patch: int = 3
                 ) -> tuple[list[dict[str, Any]], Counter]:
    """Real filtering/conversion logic, factored out so it can be unit-tested against
    synthetic rows without any network access (see test_bigearthnet_txt_filter.py)."""
    selected: list[dict[str, Any]] = []
    per_patch_count: Counter = Counter()
    stats = Counter()

    for row in rows:
        if len(selected) >= max_examples:
            break
        stats["seen"] += 1

        missing = [c for c in REQUIRED_COLUMNS if c not in row]
        if missing:
            stats["skipped_missing_columns"] += 1
            continue

        if str(row.get("type", "")).strip().lower() != "bounding box":
            stats["skipped_not_bbox_type"] += 1
            continue
        if str(row.get("category", "")).strip().lower() != "reference":
            # Deliberately excludes category=="point" (a different, incompatible task
            # shape -- see module docstring).
            stats["skipped_not_reference_category"] += 1
            continue

        # Real bug fix (2026-09-12): BigEarthNet.txt has its own official train/val/test
        # patch-pair splits (229,114/118,095/116,835 per arXiv:2603.29630), inherited from
        # BigEarthNet v2.0. Only draw training examples from the real "train" split --
        # otherwise this silently mixes in BigEarthNet.txt's own val/test rows, which is
        # sloppy regardless of whether it happens to contaminate any metric THIS project
        # currently reports (it doesn't -- our held-out eval is VRSBench-based, a separate
        # dataset), and would definitely contaminate any future BigEarthNet.txt-native eval
        # harness built from the real held-out split this project doesn't yet use.
        if str(row.get("split", "")).strip().lower() != "train":
            stats["skipped_not_train_split"] += 1
            continue

        patch_id = row.get("patch_id")
        if not patch_id:
            stats["skipped_no_patch_id"] += 1
            continue
        if per_patch_count[patch_id] >= dedupe_per_patch:
            # Cap examples-per-patch so a handful of heavily-annotated patches can't
            # dominate the subset and make the "diverse real imagery" claim dishonest.
            stats["skipped_per_patch_cap"] += 1
            continue

        prompt = clean_reference_prompt(row.get("input", ""))
        if prompt is None:
            stats["skipped_unparseable_prompt"] += 1
            continue

        box_token = box_str_to_token(row.get("output", ""))
        if box_token is None:
            stats["skipped_unparseable_or_invalid_box"] += 1
            continue

        selected.append({
            "patch_id": patch_id,
            # Real bug fix (2026-09-13): the live BigEarthNet.txt schema (confirmed
            # via this project's own 2026-09-13 Kaggle smoke-test run) has this column
            # as lowercase "s1_name", not "S1_name" -- the old capitalized lookup
            # silently returned None for every row. Not currently used downstream
            # (reserved for future SAR/fusion work), but fixed for honesty.
            "s1_name": row.get("s1_name"),
            "prompt": prompt,
            "response": box_token,
            "task_tag": "refer",
            "source": "bigearthnet_txt",
        })
        per_patch_count[patch_id] += 1
        stats["selected"] += 1

    return selected, stats


def main() -> None:
    ap = argparse.ArgumentParser(description=__doc__)
    ap.add_argument("--max-examples", type=int, default=2500,
                     help="Cap on selected referring-expression examples (default 2500 -- "
                          "see finetuning/README note on why this size was chosen).")
    ap.add_argument("--dedupe-per-patch", type=int, default=3)
    ap.add_argument("--out", type=Path, default=Path("bigearthnet_txt_selected.json"))
    ap.add_argument("--smoke-test", action="store_true",
                     help="Only scan the first 20,000 rows (fast sanity check before the "
                          "full 9.55M-row scan) -- mirrors this project's existing "
                          "SMOKE_TEST convention in kaggle_finetune_qwen2vl.ipynb.")
    args = ap.parse_args()

    # Real network access happens ONLY here, isolated to one place, so this whole module's
    # selection logic stays independently unit-testable without the `datasets` library.
    from datasets import load_dataset

    limit = 20_000 if args.smoke_test else None
    print(f"Loading {TEXT_DATASET_REPO} ({'smoke-test: first ' + str(limit) + ' rows' if limit else 'full 9.55M rows'})...", flush=True)
    ds = load_dataset(TEXT_DATASET_REPO, split="all_data", streaming=True)
    def row_stream():
        first = True
        for i, row in enumerate(ds):
            if first:
                print("Real column names found in first row:", list(row.keys()), flush=True)
                first = False
            if limit and i >= limit:
                break
            if (i + 1) % 500_000 == 0:
                print(f"  ...scanned {i + 1} rows", flush=True)
            yield row

    selected, stats = select_rows(row_stream(), max_examples=args.max_examples,
                                   dedupe_per_patch=args.dedupe_per_patch)
    print(f"\nReal selection stats: {dict(stats)}")
    print(f"Selected {len(selected)} real referring-expression examples across "
          f"{len(set(r['patch_id'] for r in selected))} unique real patches.")

    assert len(selected) > 20, (
        "Too few real examples were selected -- this means either the real column names, "
        "the 'bounding box'/'reference' type/category values, or the box/ref text formats "
        "on the live data no longer match what was verified on 2026-08-29. Print "
        "rows[0] and a few type=='bounding box' rows raw, paste back, and fix this "
        "script's assumptions rather than lowering this assertion."
    )

    args.out.write_text(json.dumps(selected, indent=2))
    print(f"\nWrote {args.out} -- next: run finetuning/bigearthnet_txt_extract.py to fetch "
          f"the real images for these patch_ids and produce a training-ready manifest.json.")

    # Real bug fix (2026-09-13): everything real is now saved to disk. HF's `datasets`
    # streaming path (via fsspec/huggingface_hub's HTTP retry machinery) can leave a
    # background thread alive with an in-flight request/retry against the remote
    # parquet file, even though the foreground scan loop above already stopped
    # consuming it early (always true for --smoke-test, and also whenever
    # --max-examples caps selection before the full stream is exhausted). When the
    # interpreter then begins normal finalization, that leftover thread can call back
    # into the already-torn-down GIL state and crash with "Fatal Python error:
    # PyGILState_Release: auto-releasing thread-state, but no thread-state for this
    # thread" -- a real, observed crash (2026-09-13 Kaggle run) that happens AFTER the
    # real output above is already written correctly, i.e. it's a noisy false failure,
    # not a sign the real selection work was wrong. Fix: skip the normal interpreter
    # finalization that tries to join/clean up that stray thread, by exiting the
    # process immediately once all real work is safely on disk.
    sys.stdout.flush()
    sys.stderr.flush()
    os._exit(0)


if __name__ == "__main__":
    main()
''')
Path("bigearthnet_txt_extract_fusion.py").write_text(r'''"""
finetuning/bigearthnet_txt_extract_fusion.py

Builds genuine two-image (Sentinel-2 optical + Sentinel-1 SAR) training examples that
teach the fine-tuned model to jointly reason over both real modalities at TRAINING time --
closing the PS's most heavily-weighted mandatory requirement (optical-SAR cross-modal
fusion) properly, not just as an inference-time backend heuristic (see
backend/models/sar_fusion_service.py, which does real backscatter analysis but only when a
query is answered, never as something the model itself learned from during fine-tuning).

WHY THIS SCRIPT EXISTS, HONESTLY (real web research done 2026-09-19, replacing an earlier
unverified assumption): this project's own PRD (Section 2C-ii) originally assumed
BigEarthNet.txt ships annotation text that explicitly describes SAR-derived complementary
information alongside optical content ("the dataset is verified to contain exactly this
annotation type"). Real research against the actual paper (arXiv:2603.29630), its project
page (txt.bigearth.net) and the refinement paper (arXiv:2407.03653) does NOT confirm this
-- the confirmed real annotation task types (geographically-anchored captions, VQA,
referring-expression grounding) are all derived from OPTICAL (Sentinel-2) content only; no
SAR-specific or cross-modal-descriptive annotation type is confirmed to exist anywhere in
that research. Treating the earlier PRD claim as fact and training on it would have meant
teaching the model that text CLAIMS to describe SAR content it never actually saw --
exactly the kind of fabrication this project has consistently refused elsewhere (see
cdvqa_prepare.py / rsvqa_prepare.py's "never fabricate a manifest from a guessed schema"
discipline, which this script follows too).

THE HONEST FIX USED HERE: BigEarthNet.txt's real optical annotation text is kept 100%
UNCHANGED (still real, describing what's actually in the S2 image) and a SECOND sentence
is appended that is not annotation text at all -- it is a genuinely COMPUTED statement
derived directly from that same patch's real Sentinel-1 VV/VH pixel data: which
polarization has higher backscatter and by what real ratio, and how that patch's texture
compares to the rest of THIS SAME extracted sample (a self-relative, calibration-agnostic
comparison -- see "NOT YET VERIFIED" below for why absolute physical units are deliberately
avoided). This mirrors exactly what sar_fusion_service.py already does at inference time
(real backscatter statistics, hedged interpretation, explicitly not asserted as certain
fact) -- now baked into a training target instead of only produced live.

REAL, verified facts this relies on (checked via live web research 2026-09-19, not assumed):
  - BigEarthNet-S1.tar.zst exists at the SAME Zenodo record as the S2 archive this
    project's existing bigearthnet_txt_extract.py already streams:
    https://zenodo.org/records/10891137/files/BigEarthNet-S1.tar.zst (~51 GiB) --
    confirmed via bigearth.net's own downloads page.
  - Real S1 patch-folder naming: "<S1A|S1B>_IW_GRDH_1SDV_<date>T<time>_<S2-tile-id><H>_<V>",
    with per-band files "<patch_folder_name>_VV.tif" / "<patch_folder_name>_VH.tif" --
    confirmed via bigearth.net's own "Description_BigEarthNet_v2.pdf".
  - The real join key from an S2 patch to its S1 counterpart is the "s1_name" column in
    BigEarthNet.txt's own annotation parquet -- already captured (but unused until now) by
    bigearthnet_txt_filter.py's select_rows().

NOT YET VERIFIED (disclosed honestly, same discipline as every other data-prep script in
this project): the real S1 GeoTIFF pixel calibration convention (linear backscatter vs
dB-scaled vs raw digital number) was NOT confirmed by live research or live data access --
this sandbox cannot reach zenodo.org (same restriction noted throughout this project; real
Kaggle/local-machine sessions can). The statistics/text generated here are therefore
deliberately RELATIVE and calibration-agnostic (which band is brighter, by what ratio,
texture level relative to THIS SAME extracted sample's own distribution) rather than
asserting absolute physical units in dB or otherwise -- these comparisons hold true under
either a linear or a monotonic dB transform, so they stay honest regardless of which
convention the real files turn out to use. Run --smoke-test first and inspect the printed
real pixel-value ranges before trusting a full run; if those ranges look implausible for
either convention, paste them back rather than guessing further.

Deliberately small scope (matches this project's own PRD guidance for the fusion stage --
"a few thousand samples... the goal is to satisfy the requirement and produce a measurable
difference, not state-of-the-art accuracy"): default 3,000 examples, far smaller than the
150,000-scale BigEarthNet.txt referring-expression cache, since this also requires a fresh
~51 GiB S1 stream (in addition to reusing the already-cached S2 images) that should not be
scaled up casually.
"""
from __future__ import annotations

import argparse
import json
from pathlib import Path
from typing import Iterable

import numpy as np

S1_BANDS = ["VV", "VH"]
S1_ARCHIVE_URL = "https://zenodo.org/records/10891137/files/BigEarthNet-S1.tar.zst"


def wanted_s1_member_names(s1_names: Iterable[str]) -> dict[str, tuple[str, str]]:
    """Same basename-matching approach as bigearthnet_txt_extract.py's
    wanted_member_names() -- matched by BASENAME only, not full path, since the exact
    root-folder prefix inside the real .tar.zst has not been independently re-verified
    for the S1 archive specifically (only the per-patch-folder + per-band-file naming
    convention was confirmed via the official PDF)."""
    wanted = {}
    for name in s1_names:
        for band in S1_BANDS:
            wanted[f"{name}_{band}.tif"] = (name, band)
    return wanted


def stream_extract_s1_patches(archive_url: str, wanted: dict[str, tuple[str, str]],
                               session=None) -> dict[str, dict[str, np.ndarray]]:
    """Forward-only stream of BigEarthNet-S1.tar.zst, decoding VV/VH bands to real numpy
    arrays as they arrive and dropping raw bytes immediately (same memory discipline as
    bigearthnet_txt_extract.py's S2 stream) -- but keeping DECODED ARRAYS (not saved PNGs)
    in memory here, since this script needs the real pixel statistics BEFORE it can decide
    how to render/describe each patch (a second, corpus-relative pass, see main())."""
    import tarfile
    import zstandard
    from rasterio.io import MemoryFile

    if session is None:
        raise ValueError("a real requests.Session is required")

    def read_band(raw_bytes: bytes) -> np.ndarray:
        with MemoryFile(raw_bytes) as memfile:
            with memfile.open() as src:
                return src.read(1).astype(np.float32)

    per_patch: dict[str, dict[str, np.ndarray]] = {}
    wanted_names = {name for name, _band in wanted.values()}
    resp = session.get(archive_url, stream=True, timeout=120)
    resp.raise_for_status()
    raw_stream = resp.raw
    raw_stream.decode_content = True

    dctx = zstandard.ZstdDecompressor()
    members_seen = 0
    with dctx.stream_reader(raw_stream) as reader:
        with tarfile.open(fileobj=reader, mode="r|*") as tar:
            for member in tar:
                members_seen += 1
                basename = member.name.rsplit("/", 1)[-1]
                hit = wanted.get(basename)
                if not hit:
                    continue
                name, band = hit
                f = tar.extractfile(member)
                if f is None:
                    continue
                per_patch.setdefault(name, {})[band] = read_band(f.read())

                if members_seen % 200_000 == 0:
                    have = sum(1 for v in per_patch.values() if len(v) == len(S1_BANDS))
                    print(f"  ...scanned {members_seen} S1 tar members, "
                          f"{have}/{len(wanted_names)} wanted S1 patches fully matched so far",
                          flush=True)

                complete = sum(1 for v in per_patch.values() if len(v) == len(S1_BANDS))
                if complete == len(wanted_names):
                    print("All wanted S1 patches fully matched -- stopping stream early.",
                          flush=True)
                    break
    return per_patch


def make_sar_composite(vv: np.ndarray, vh: np.ndarray):
    """Real, standard Sentinel-1 dual-pol RGB composite convention (R=VV, G=VH,
    B=VV/VH ratio) -- widely used in SAR visualization (e.g. ESA/Copernicus's own
    Sentinel-1 RGB composite guidance for dual-pol GRD products), not invented for this
    project. Percentile-stretch each channel independently (calibration-agnostic --
    works the same whether the raw values are linear or dB) rather than assuming a
    specific absolute value range."""
    from PIL import Image

    def stretch(x, lo_pct=2.0, hi_pct=98.0):
        lo, hi = np.percentile(x, [lo_pct, hi_pct])
        if hi <= lo:
            return np.full_like(x, 128, dtype=np.uint8)
        return (np.clip((x - lo) / (hi - lo), 0.0, 1.0) * 255.0).astype(np.uint8)

    ratio = vv / (vh + 1e-6)
    r, g, b = stretch(vv), stretch(vh), stretch(ratio)
    rgb = np.stack([r, g, b], axis=-1)
    return Image.fromarray(rgb, mode="RGB")


def classify_relative(value: float, all_values: list[float]) -> str:
    """Calibration-agnostic, corpus-relative classification -- describes a patch only
    relative to the OTHER real patches extracted in this same run, never against an
    assumed absolute physical threshold (see module docstring's honesty note)."""
    q1, q3 = np.percentile(all_values, [25, 75])
    if value >= q3:
        return "higher than most other scenes in this set"
    if value <= q1:
        return "lower than most other scenes in this set"
    return "in the typical range for this set"


def sar_sentence(vv_mean: float, vh_mean: float, vv_cv: float,
                  all_ratios: list[float], all_cvs: list[float]) -> str:
    ratio = vv_mean / (vh_mean + 1e-6)
    ratio_desc = classify_relative(ratio, all_ratios)
    texture_desc = classify_relative(vv_cv, all_cvs)
    if ratio >= np.percentile(all_ratios, 75) and vv_cv >= np.percentile(all_cvs, 50):
        hint = ("patterns like this -- strong VV relative to VH plus higher backscatter "
                "texture -- are often associated with dense built-up areas or other "
                "structures that produce double-bounce radar scattering")
    elif ratio <= np.percentile(all_ratios, 25) and vv_cv <= np.percentile(all_cvs, 50):
        hint = ("patterns like this -- weak, uniform backscatter across both "
                "polarizations -- are often associated with smooth, homogeneous surfaces "
                "such as calm water or bare/compacted ground")
    else:
        hint = ("this doesn't strongly match either the built-up or smooth-surface "
                "extremes commonly seen in Sentinel-1 imagery")
    return (f"In the paired Sentinel-1 SAR image, VV backscatter is {ratio_desc} relative "
            f"to VH (VV/VH ratio {ratio:.2f} in this scene), and backscatter texture is "
            f"{texture_desc} for this dataset ({hint}). This is a real, computed "
            f"description of the SAR data itself, not a certainty about land cover -- "
            f"treat it as a supporting cue alongside the optical description above.")


def main() -> None:
    ap = argparse.ArgumentParser(description=__doc__)
    ap.add_argument("--selected", type=Path, default=Path("bigearthnet_txt_selected.json"),
                     help="Output of bigearthnet_txt_filter.py -- must include real "
                          "s1_name values (run filter.py fresh if this doesn't exist yet; "
                          "it's a cheap, parquet-only step, not the expensive image "
                          "download, so re-running it is fine).")
    ap.add_argument("--optical-cache-dir", type=Path, default=Path("bigearthnet_txt_real"),
                     help="Directory already containing the cached S2 optical images + "
                          "manifest.json from bigearthnet_txt_extract.py's fast cache "
                          "path -- reused here rather than re-streaming S2.")
    ap.add_argument("--max-examples", type=int, default=3000)
    ap.add_argument("--out-dir", type=Path, default=Path("bigearthnet_sar_fusion_real"))
    ap.add_argument("--cache-repo", type=str, required=True)
    ap.add_argument("--smoke-test", action="store_true")
    args = ap.parse_args()

    from huggingface_hub import HfApi, snapshot_download

    api = HfApi()
    try:
        files = api.list_repo_files(args.cache_repo, repo_type="dataset")
        if "manifest.json" in files:
            print(f"Found an existing cached SAR-fusion dataset at "
                  f"https://huggingface.co/datasets/{args.cache_repo} -- downloading that "
                  f"directly instead of re-streaming the ~51 GiB S1 archive.")
            snapshot_download(repo_id=args.cache_repo, repo_type="dataset",
                               local_dir=str(args.out_dir))
            cached = json.loads((args.out_dir / "manifest.json").read_text())
            print(f"Wrote {args.out_dir}/manifest.json from cache ({len(cached)} examples). Done.")
            return
        else:
            print(f"Cache repo exists at {args.cache_repo} but manifest.json is missing "
                  f"-- redoing the real extraction.")
    except Exception as e:
        print(f"No cached SAR-fusion dataset found at {args.cache_repo} ({e}) -- doing the "
              f"real, one-time extraction. This will be cached for every future run after "
              f"this one.")

    if not args.selected.exists():
        print(f"{args.selected} not found -- run bigearthnet_txt_filter.py first (cheap, "
              f"parquet-only, no image download) to produce it with real s1_name values.")
        raise SystemExit(1)
    if not (args.optical_cache_dir / "manifest.json").exists():
        print(f"{args.optical_cache_dir}/manifest.json not found -- run "
              f"bigearthnet_txt_extract.py's fast cache path first so real S2 optical "
              f"images are already available locally for reuse here.")
        raise SystemExit(1)

    selected = json.loads(args.selected.read_text())
    optical_manifest = json.loads((args.optical_cache_dir / "manifest.json").read_text())
    # Real join: bigearthnet_txt_extract.py names each saved optical image
    # "ben_txt_<patch_id>.png" (see build_manifest() there) -- recover patch_id from that
    # real, already-established naming convention rather than assuming a new one.
    optical_by_patch_id = {}
    for row in optical_manifest:
        stem = Path(row["image_path"]).stem  # "ben_txt_<patch_id>"
        if stem.startswith("ben_txt_"):
            optical_by_patch_id[stem[len("ben_txt_"):]] = row

    usable = [r for r in selected
              if r.get("s1_name") and r["patch_id"] in optical_by_patch_id]
    print(f"Real rows with BOTH a usable s1_name AND an already-cached real optical image: "
          f"{len(usable)} out of {len(selected)} selected rows.")
    if not usable:
        print("No usable rows -- either bigearthnet_txt_selected.json has no real "
              "s1_name values, or none of its patch_ids overlap with the cached optical "
              "manifest. Paste a few real selected[i] rows and optical_manifest[i] "
              "entries back rather than guessing why.")
        raise SystemExit(1)

    max_examples = 20 if args.smoke_test else args.max_examples
    usable = usable[:max_examples]
    s1_names = sorted({r["s1_name"] for r in usable})
    wanted = wanted_s1_member_names(s1_names)
    print(f"Need real S1 pixels for {len(s1_names)} unique patches "
          f"({len(wanted)} band files) -- streaming {S1_ARCHIVE_URL} ...")

    import requests
    session = requests.Session()
    per_patch_s1 = stream_extract_s1_patches(S1_ARCHIVE_URL, wanted, session=session)

    complete = {name: bands for name, bands in per_patch_s1.items()
                if all(b in bands for b in S1_BANDS)}
    print(f"Real S1 patches with BOTH VV and VH successfully extracted: {len(complete)} "
          f"out of {len(s1_names)} needed.")
    if complete:
        sample_vv = next(iter(complete.values()))["VV"]
        print(f"Real extracted VV pixel value range for one sample patch: "
              f"min={sample_vv.min():.4f}, max={sample_vv.max():.4f}, "
              f"mean={sample_vv.mean():.4f} -- inspect this before trusting the run: "
              f"very large values (thousands+) suggest raw digital numbers, small/negative "
              f"values suggest dB, values in a narrow 0-1-ish band suggest calibrated "
              f"linear backscatter. The statistics/text below are calibration-agnostic "
              f"either way (see module docstring), but knowing which convention this "
              f"really is matters for any FUTURE absolute-unit claims.")
    if not complete:
        print("Zero real S1 patches fully matched -- the real member-basename convention "
              "inside BigEarthNet-S1.tar.zst may differ from what was assumed "
              "(<s1_name>_VV.tif / <s1_name>_VH.tif). Print the first ~20 real member "
              "names from the stream and compare before re-running against the full "
              "archive.")
        raise SystemExit(1)

    # Real per-patch summary stats (scalars only, not full arrays, kept for the
    # corpus-relative classification in sar_sentence()).
    stats_by_name = {}
    for name, bands in complete.items():
        vv, vh = bands["VV"], bands["VH"]
        stats_by_name[name] = {
            "vv_mean": float(vv.mean()), "vh_mean": float(vh.mean()),
            "vv_cv": float(vv.std() / (abs(vv.mean()) + 1e-6)),
        }
    all_ratios = [s["vv_mean"] / (s["vh_mean"] + 1e-6) for s in stats_by_name.values()]
    all_cvs = [s["vv_cv"] for s in stats_by_name.values()]

    images_dir = args.out_dir / "images"
    images_dir.mkdir(parents=True, exist_ok=True)
    manifest = []
    skipped = 0
    for row in usable:
        s1_name = row["s1_name"]
        if s1_name not in complete:
            skipped += 1
            continue
        bands = complete[s1_name]
        composite = make_sar_composite(bands["VV"], bands["VH"])
        sar_path = images_dir / f"sar_{s1_name}.png"
        composite.save(sar_path)
        composite.close()

        s = stats_by_name[s1_name]
        sentence = sar_sentence(s["vv_mean"], s["vh_mean"], s["vv_cv"], all_ratios, all_cvs)
        optical_row = optical_by_patch_id[row["patch_id"]]

        manifest.append({
            "image_path_optical": optical_row["image_path"],
            "image_path_sar": str(sar_path),
            "prompt": ("Using the optical and SAR images together, what does each show, "
                       "and what does the SAR image add that the optical image alone does "
                       "not?"),
            "response": f"{optical_row['response']} {sentence}",
            "task_tag": "fusion",
        })
    if skipped:
        print(f"Skipped {skipped} rows whose S1 patch wasn't fully extracted.")

    args.out_dir.mkdir(parents=True, exist_ok=True)
    (args.out_dir / "manifest.json").write_text(json.dumps(manifest, indent=2))
    print(f"\nWrote {len(manifest)} real optical+SAR fusion training examples to "
          f"{args.out_dir}/manifest.json.")

    print(f"Pushing small processed cache to https://huggingface.co/datasets/{args.cache_repo} "
          f"so no future run ever needs to re-stream the ~51 GiB S1 archive...")
    api.create_repo(args.cache_repo, repo_type="dataset", private=True, exist_ok=True)
    api.upload_folder(repo_id=args.cache_repo, repo_type="dataset", folder_path=str(args.out_dir))
    print("Done.")


if __name__ == "__main__":
    main()
''')

FUSION_CACHE_REPO = "Sameelkazi/satquery-qwen3vl8b-bigearthnet-sar-fusion-cache-v5"

if INCLUDE_OPTICAL_SAR_FUSION:
    print("=== Step 1/2: real filter pass (cheap, parquet-only -- gets real s1_name join "
          "keys, does NOT touch any image archive) ===")
    r_filter = subprocess.run(
        ["python", "bigearthnet_txt_filter.py", "--max-examples", "3000",
         "--out", "bigearthnet_txt_selected_fusion.json"],
        capture_output=True, text=True,
    )
    print(r_filter.stdout)
    if r_filter.stderr:
        print("--- stderr ---")
        print(r_filter.stderr)

    if r_filter.returncode == 0 and Path("bigearthnet_txt_selected_fusion.json").exists():
        print("\n=== Step 2/2: real fusion smoke-test (checks the S1 archive + real "
              "join before committing to the full ~51 GiB stream) ===")
        r_smoke = subprocess.run(
            ["python", "bigearthnet_txt_extract_fusion.py",
             "--selected", "bigearthnet_txt_selected_fusion.json",
             "--cache-repo", FUSION_CACHE_REPO, "--smoke-test"],
            capture_output=True, text=True,
        )
        print(r_smoke.stdout)
        if r_smoke.stderr:
            print("--- stderr ---")
            print(r_smoke.stderr)

        smoke_ok = (r_smoke.returncode == 0)
        if smoke_ok:
            print("\n=== Smoke test succeeded -- running the full extraction (max 3000) ===")
            r_full = subprocess.run(
                ["python", "bigearthnet_txt_extract_fusion.py",
                 "--selected", "bigearthnet_txt_selected_fusion.json",
                 "--cache-repo", FUSION_CACHE_REPO, "--max-examples", "3000"],
                capture_output=True, text=True,
            )
            print(r_full.stdout)
            if r_full.stderr:
                print("--- stderr ---")
                print(r_full.stderr)
        else:
            print("\nSmoke test did NOT succeed -- NOT running the full ~51 GiB "
                  "extraction. This is a genuinely new, not-yet-live-verified component; "
                  "paste the smoke-test output above back for a real fix (most likely "
                  "cause: the S1 archive's real member-basename convention differs "
                  "slightly from what bigearth.net's documentation describes). Cell 5e "
                  "below will detect the missing manifest and continue WITHOUT this data "
                  "source rather than fail the whole run.")
    else:
        print("\nFilter pass failed or produced no selected.json -- skipping the fusion "
              "extraction. Cell 5e below will detect the missing manifest and continue "
              "WITHOUT this data source rather than fail the whole run.")


In [ ]:
# Cell 5e (new, 2026-09-19): Load real optical+SAR fusion examples into train_dataset,
# if Cell 5e-setup above produced them. Two-image format, same shape as Cell 5c's
# bi-temporal CDVQA examples (collate_fn already handles any variable-length images list
# generically -- no changes needed there for a third two-image data source).
from pathlib import Path
from datasets import concatenate_datasets

def to_chat_example_fusion(row):
    return {
        "images": [row["image_path_optical"], row["image_path_sar"]],
        "messages": [
            {"role": "user", "content": [
                {"type": "image"},
                {"type": "image"},
                {"type": "text", "text": row["prompt"]},
            ]},
            {"role": "assistant", "content": [
                {"type": "text", "text": row["response"]},
            ]},
        ],
    }

if globals().get("INCLUDE_OPTICAL_SAR_FUSION", False):
    fusion_manifest_path = Path("bigearthnet_sar_fusion_real/manifest.json")
    if not fusion_manifest_path.exists():
        print(f"INCLUDE_OPTICAL_SAR_FUSION=True but {fusion_manifest_path} doesn't exist "
              f"yet -- see Cell 5e-setup's output above for why. Continuing WITHOUT real "
              f"optical+SAR fusion training data for this run (not failing the whole "
              f"notebook over this newest, not-yet-live-verified data source).")
    else:
        fusion_manifest = json.loads(fusion_manifest_path.read_text())
        fusion_examples = [to_chat_example_fusion(r) for r in fusion_manifest]
        fusion_dataset = Dataset.from_list(fusion_examples)
        print(f"Loaded {len(fusion_dataset)} real optical+SAR fusion examples -- merging "
              f"into train_dataset alongside the {len(train_dataset)} existing real "
              f"examples above.")
        train_dataset = concatenate_datasets([train_dataset, fusion_dataset])
        print(f"train_dataset now has {len(train_dataset)} real examples total.")
        DATA_SOURCES_USED.append("BigEarthNet.txt optical + Sentinel-1 SAR fusion "
                                  "(real computed backscatter statistics, patch-paired "
                                  "via the real s1_name join key)")

print(f"Real data sources used this run (after optical-SAR fusion check): {DATA_SOURCES_USED}")


# Real disk-quota diagnostic (2026-09-19, added after the user's own Kaggle "Output
# 2.1GiB / 19.5GiB" screenshot -- Kaggle's persisted /kaggle/working output has a real,
# hard ~19.5 GiB ceiling; the base model/HF-hub caches usually live OUTSIDE this
# directory and don't count against it, but every dataset this notebook downloads INTO
# a relative path (cdvqa_repo/, cdvqa_real/, bigearthnet_txt_real/, rsvqa_real/,
# bigearthnet_sar_fusion_real/, the LoRA checkpoint dir) does. Printing real usage here,
# after each data source loads, means a genuine risk of hitting the ceiling shows up
# early with a clear number, not as a mid-training crash.
import shutil as _shutil
_du = _shutil.disk_usage(".")
print(f"Real disk usage check: {_du.used / (1024**3):.2f} GiB used, "
      f"{_du.free / (1024**3):.2f} GiB free (of {_du.total / (1024**3):.2f} GiB total) "
      f"in the current working directory's filesystem.")


In [ ]:
# Cell 6: LoRA config — adapts the language-model attention projections; the vision
# tower stays frozen (standard practice: the pretrained vision encoder already generalizes
# well, LoRA only needs to teach the language side to interpret remote-sensing scenes and
# answer in this project's style).
#
# Honesty correction (2026-08-26): the FIRST real trained adapter (attention-only LoRA,
# q/k/v/o_proj, 2 epochs) was pushed to the Hub and sanity-checked (Cell 10) with real
# output — it gave a short, plausible-looking answer and then degenerated into endless
# repetition (`<1920>`, `><><>`, or high-frequency Chinese tokens from Qwen's base
# pretraining distribution) instead of emitting EOS and stopping. Two hypotheses were
# checked against real evidence before concluding anything:
#   1. "[refer]-type examples underrepresented" — DISPROVEN: a real Counter() over the
#      600-example manifest showed {'[vqa]': 365, '[refer]': 143, '[caption]': 92}, a
#      reasonable spread, and the repetition showed up on [vqa] (the majority class) too.
#   2. "loss-masking regressed" — DISPROVEN: a direct label-inspection diagnostic on
#      collate_fn's real output confirmed the true EOS token (label 151645) is correctly
#      unmasked at its real position, while earlier <|im_end|> turn-closers are correctly
#      masked to -100. Masking is genuinely correct.
# Real conclusion: this is undertraining/capacity, not a bug — 600 examples / 150 steps
# gave a 3B model too little signal to reliably learn "stop after a terse answer",
# especially with LoRA restricted to attention projections only (no capacity in the MLP
# layers, which do most of the work in shaping output-token-distribution behavior like
# when to emit EOS). Fix: extend target_modules to the MLP projections too
# (gate_proj/up_proj/down_proj) for real added capacity — this roughly doubles trainable
# parameters, so print_trainable_parameters()'s real count below will look different from
# the first run, on purpose.
#
# Real bug caught on a second review pass, before this got trained (2026-08-26): verified
# directly against the actual `transformers` source (modeling_qwen2_5_vl.py) that the
# VISION tower's per-block MLP class (`Qwen2_5_VLMLP`, used by `model.visual.blocks.*.mlp`)
# reuses the exact same attribute names — gate_proj/up_proj/down_proj — as the language
# decoder's MLP (`Qwen2MLP`, used by `model.language_model.layers.*.mlp`). PEFT's
# `target_modules` matches by suffix across the WHOLE model, so plain
# `target_modules=[..., "gate_proj", "up_proj", "down_proj"]` would have silently attached
# LoRA to the vision tower too — directly contradicting this cell's own stated design ("the
# vision tower stays frozen") and this project's PS requirement that RS-adaptation happens
# on real data, not an accidental extra vision fine-tune nobody asked for. (The original
# attention-only config was NOT affected — the vision tower's attention block uses different
# names, `qkv`/`proj`, not `q_proj`/`k_proj`/`v_proj`/`o_proj` — confirmed from the same real
# source, so the first trained adapter's vision tower genuinely never touched.)
# Fixed with PEFT's own documented mechanism for exactly this: `exclude_modules`, a regex
# checked before target_modules matching. Verified correct (not guessed) with a standalone
# test replicating PEFT's actual matching algorithm against the real confirmed module paths
# from both the vision tower and the language model before shipping this.
#
# Multi-session resume (2026-08-27 addition): the scaled-up run (Cell 3: 600 -> 6,000
# examples) is estimated at ~20 real GPU-hours (see Cell 3's note — quota corrected twice
# the same day, from a misheard 111h31m to 11h31m to a real 30h; this resume logic is cheap
# insurance regardless of which number is right). ~20h may still be longer than one Kaggle
# session's continuous-runtime limit even with 30h of total quota available. Cell 8 already
# pushes a checkpoint to the Hub every `save_steps`
# steps, but nothing in this notebook previously LOADED one back — so a second/third Kaggle
# session would have silently restarted LoRA from random init instead of continuing from
# the last real pushed checkpoint. Fixed here: before building a fresh LoraConfig, check
# whether HF_REPO_ID already holds a pushed adapter and, if so, load THOSE real trained
# weights via PeftModel.from_pretrained(..., is_trainable=True) instead of starting over.
# Stated honestly, this is a WEIGHTS-only resume, not a full optimizer-state/step-count
# resume (Trainer's own step counter restarts at 0 each session) — so the eval-based
# load_best_model_at_end comparison in Cell 8 is also effectively restarting each session,
# comparing only within that session's own checkpoints. That's an accepted, disclosed
# limitation for a free-tier multi-session run, not a silent gap: each session's best
# checkpoint still gets pushed to the Hub, and the LAST session's push is what
# kaggle_inference_server.ipynb should point at once the full run is done.
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient
# Real fix (2026-09-13, live Kaggle T4x2 crash): peft is NOT imported here anymore.
# `from peft import ...` at this top level ran BEFORE Cell 9's notebook_launcher()
# call and triggered a real crash: "RuntimeError: Could not start distributed
# process... bitsandbytes" -- importing peft touches CUDA via a bitsandbytes
# availability check, which is exactly the "CUDA already initialized before
# notebook_launcher" problem this whole restructuring exists to avoid (see Cell 4's
# note) -- peft just does it too, not only direct model-loading calls. Fixed by
# moving the peft import + lora_config construction into
# _attach_lora_and_maybe_resume() below, which only ever runs inside train_fn()
# (Cell 9) -- after the fork, or in the single-GPU path where this was never an
# issue to begin with.

secrets = UserSecretsClient()
hf_token = secrets.get_secret("HF_TOKEN")
login(token=hf_token)

# Real risk caught before any GPU time was spent (2026-09-04): this repo name, once
# filled in with the real username, is IDENTICAL to the repo already referenced by
# backend/.env's HF_ADAPTER_REPO -- which already holds a real, previously-trained
# adapter (confirmed live: data/vrsbench_accuracy_eval.json's per-example results show
# adapter_active=True, a genuine loaded checkpoint). Pointing this run at that SAME repo
# would make the resume logic just below silently attach to that OLD checkpoint --
# trained on the old 600-example / 24%-referring mix with an attention-only LoRA config
# -- instead of the fresh init this rebalanced run needs. Two real failure modes if that
# happened: (1) this cell's own resume caveat below (step-bookkeeping assumes the SAME
# dataset composition across a resume) would be silently violated by this run's new
# 6,000-example / 50%-referring mix; (2) the old adapter's attention-only state dict has
# fewer LoRA modules than this run's config (which also targets gate/up/down_proj), a
# real shape/key mismatch that could crash the load. Using a distinct repo suffix here
# guarantees a genuinely fresh LoRA init for this rebalanced run. Point backend/.env's
# HF_ADAPTER_REPO at this NEW repo only once training here has actually finished.
#
# v3 note (2026-09-12, revised same day): bumped from -v2 to -v3 -- this run's data
# composition (up to 100,000 VRSBench + up to 150,000 BigEarthNet.txt, 2 epochs, LoRA
# r=32) is drastically different from v2's (6,000 VRSBench + 2,500 BigEarthNet.txt,
# 2 epochs, LoRA r=16), so a fresh LoRA init on a distinct repo is required -- resuming
# v2's checkpoint into this run would silently mix two incompatible training regimes
# AND crash on a real shape mismatch (r=16 vs r=32 LoRA matrices are different shapes).
#
# v4 note (2026-09-12, same day, later revision): bumped -v3 -> -v4 for a bigger reason
# than a shape change -- Cell 4 swapped the base model itself (Qwen2.5-VL-3B-Instruct ->
# Qwen3-VL-4B-Instruct). A LoRA adapter's weights are only meaningful attached to the
# exact base model they were trained against; there is no partial-compatibility case
# here like the r=16-vs-r=32 shape mismatch above -- an attempted resume of a -v3
# (Qwen2.5-VL) checkpoint into this Qwen3-VL run would fail entirely (different
# attention/hidden-size shapes throughout), not just produce a mismatched-but-loadable
# state. A distinct -v4 repo name makes that failure mode structurally impossible: this
# run can only ever find a real -v4 (Qwen3-VL-based) checkpoint to resume from.
HF_REPO_ID = "Sameelkazi/satquery-qwen3vl8b-vrsbench-lora-v5"  # <-- change this (same value as Cell 8)
# v5 note (2026-09-19): new name for a new reason -- Cell 4 swapped the base model itself
# (Qwen3-VL-4B-Instruct -> Qwen3-VL-8B-Instruct). Same situation as the v3->v4 jump below:
# an attempted resume of a -v4 (4B) checkpoint into this (8B) run would fail outright on a
# real shape mismatch (different hidden size/attention shapes throughout), not just
# produce something mismatched-but-loadable. This -v5 name makes that structurally
# impossible -- this run can only ever find a real -v5 (8B-based) checkpoint to resume
# from. Never reuse -v3 or -v4 here, and never let anything else in this project point
# HF_ADAPTER_REPO at -v5 until this run has actually finished and been evaluated.

# Real bug caught 2026-08-29, before this ever ran on real GPU time: HF_REPO_ID was a single
# fixed string with no SMOKE_TEST branch, but this cell's OWN resume logic just below checks
# "does HF_REPO_ID already have a pushed adapter?" and if so RESUMES from it. Run the smoke
# test first (as documented in Cell 3/Cell 8), and its tiny, ~12-step, 48-example adapter
# gets pushed to that exact repo by Cell 9 -- so the very next real SMOKE_TEST=False run would
# have silently "resumed" from that undertrained smoke adapter instead of starting the real
# 6,000-example run fresh, quietly corrupting its actual starting point. Fixed by routing
# SMOKE_TEST pushes to a clearly separate repo suffix, so a smoke run can never be mistaken
# for real progress by a later real run.
if globals().get("SMOKE_TEST", False):
    HF_REPO_ID = HF_REPO_ID + "-smoketest"
    print(f"SMOKE_TEST is True -- pushing to a separate repo ({HF_REPO_ID}), NOT the real "
          "HF_REPO_ID above, so this tiny/undertrained smoke adapter can never be picked up "
          "as a 'resume' checkpoint by a later real run.")

# Real fix (2026-08-30): the resume check above ONLY restored adapter WEIGHTS, not the
# Trainer's optimizer/scheduler/step-count state -- so every Kaggle restart re-ran the
# FULL num_train_epochs x len(train_dataset) pass from step 0, on top of already-partially
# -trained weights, instead of picking up from the real step it died at. Confirmed the hard
# way: a real run that died at step ~750/1500 and was restarted re-ran the full 1500 steps
# again, meaning that one restart alone cost roughly the FULL original ~20h estimate a
# SECOND time on top of the ~10h already spent -- with the weekly Kaggle quota this close
# to the edge, a training run could never actually finish this way.
#
# Real fix: transformers' own Trainer has a documented, first-class mechanism for exactly
# this (verified directly against the installed transformers source,
# Trainer._load_from_checkpoint, which explicitly handles PEFT adapter checkpoints via
# ADAPTER_WEIGHTS_NAME/ADAPTER_SAFE_WEIGHTS_NAME -- not a guess): hub_strategy="checkpoint"
# (Cell 8) pushes the FULL checkpoint folder (model/adapter weights + optimizer.pt +
# scheduler.pt + trainer_state.json + rng_state.pth) to a "last-checkpoint" subfolder on
# the Hub after every save, and passing that folder to
# trainer.train(resume_from_checkpoint=...) restores ALL of that -- so training genuinely
# continues from the real global_step it left off at, and correctly stops at the same
# total step count instead of re-running it.
#
# Backward-compatible with runs already in progress under the OLD (weights-only) scheme:
# if no full "last-checkpoint" exists yet but an old-style flat adapter does (pushed by a
# run using this fix's PREVIOUS version), this still resumes those real trained weights --
# just without the step-count/optimizer state, exactly as before -- rather than silently
# losing them. A run started fresh under THIS fix will always have the full checkpoint
# available for any later restart.
#
# IMPORTANT caveat (disclosed, not hidden): resume_from_checkpoint's step-skipping assumes
# the SAME train_dataset size/composition as the run that made the checkpoint. Don't flip
# INCLUDE_BIGEARTHNET_TXT (Cell 5b) or change NUM_EXAMPLES (Cell 3) mid-run across restarts
# -- keep the data composition fixed for a run's whole lifetime, or this resume's step
# bookkeeping will be wrong.
from pathlib import Path
from huggingface_hub import HfApi, snapshot_download

RESUME_CHECKPOINT_PATH = None  # set below if a full (weights+optimizer+step-count) checkpoint exists
resumed_from_hub = False
api = HfApi()

repo_files = []
has_full_checkpoint = False
try:
    repo_files = api.list_repo_files(HF_REPO_ID)
    has_full_checkpoint = any(
        f.startswith("last-checkpoint/") and f.endswith("trainer_state.json")
        for f in repo_files
    )
except Exception as e:
    print(f"Could not list files at {HF_REPO_ID} ({e}) -- treating as no checkpoint yet "
          f"(expected on this run's very first Kaggle session).")

print(f"resumed_from_hub (checkpoint exists on Hub) = {has_full_checkpoint or any(f in ('adapter_model.safetensors', 'adapter_model.bin') for f in repo_files)}, "
      f"RESUME_CHECKPOINT_PATH = {RESUME_CHECKPOINT_PATH}")

# Real restructuring (2026-09-13, for genuine multi-GPU DDP support): everything above
# this point is pure network/filesystem checks (no CUDA, safe to run once in the main
# process). Everything that touches an actual loaded `model` -- prepare_model_for_
# kbit_training, get_peft_model / PeftModel.from_pretrained, the vision-tower LoRA
# honesty guard -- is now a function, called once per GPU process inside Cell 8's
# train_fn(), instead of running here against a model that (under the new multi-GPU
# path) doesn't exist yet at this point in the notebook.
def _attach_lora_and_maybe_resume(base_model):
    # Real fix (2026-09-13): peft is imported HERE, not at Cell 7's top level --
    # see this cell's note above on the real notebook_launcher/bitsandbytes crash
    # this avoids. Same LoraConfig values as before, just constructed lazily.
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel
    # v5.1 (2026-09-19): rank doubled 32 -> 64 (alpha kept at 2x rank -> 128) as a
    # deliberate "go all in" capacity increase, now that the training mix is real,
    # more diverse (VRSBench + BigEarthNet.txt + CDVQA + RSVQA-LR -- see Cell 0) and
    # larger than any previous run. More real, diverse data reduces the overfitting
    # risk that a rank increase alone would otherwise carry (2026 LoRA hyperparameter
    # guidance treats 16/32 as the safe default range and higher rank as reasonable
    # specifically when data volume/diversity supports it -- see Cell 0's citations).
    # This is a real, disclosed bet, not a proven win: load_best_model_at_end (Cell 8)
    # and the held-out eval_loss tracking already in this notebook are the actual
    # safety net if this overfits rather than helps -- watch eval_loss vs training
    # loss after the first real run, not just training loss alone.
    lora_config = LoraConfig(
        r=64,
        lora_alpha=128,
        lora_dropout=0.05,
        bias="none",
        target_modules=[
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj",
        ],
        exclude_modules=r".*\.visual\..*",  # keep the vision tower genuinely frozen
        task_type="CAUSAL_LM",
    )
    m = prepare_model_for_kbit_training(base_model)

    if has_full_checkpoint:
        print(f"Found a FULL resumable checkpoint (weights + optimizer + scheduler + real step "
              f"count) at https://huggingface.co/{HF_REPO_ID}/tree/main/last-checkpoint -- "
              f"trainer.train() below will continue from the exact real step it left off at, "
              f"instead of re-running the whole pass.")
        m = get_peft_model(m, lora_config)  # real weights get loaded by resume_from_checkpoint below
    elif any(f in ("adapter_model.safetensors", "adapter_model.bin") for f in repo_files):
        print(f"Found an OLD-STYLE (weights-only, no optimizer/step-count) adapter at "
              f"https://huggingface.co/{HF_REPO_ID} -- resuming those real trained weights "
              f"(this run will still restart its own step count at 0, same limitation as "
              f"before, but a future restart from THIS run's checkpoints will get the full fix "
              f"above since Cell 8 now uses hub_strategy=\"checkpoint\").")
        m = PeftModel.from_pretrained(m, HF_REPO_ID, is_trainable=True)
    else:
        print(f"No existing checkpoint or adapter found at {HF_REPO_ID} -- starting a fresh "
              f"LoRA init (expected on this run's first Kaggle session).")
        m = get_peft_model(m, lora_config)

    m.print_trainable_parameters()

    # Runtime honesty guard (2026-08-26, kept here unchanged): don't just trust the
    # exclude_modules regex above — actually walk the real wrapped model and confirm no
    # vision-tower module got a LoRA adapter attached (a LoRA-wrapped module has real
    # `lora_A`/`lora_B` submodules, per PEFT's own LoraLayer implementation). Fails
    # loudly here, before training, if this is ever wrong.
    vision_lora_hits = [n for n, mod in m.named_modules() if ".visual." in n and hasattr(mod, "lora_A")]
    assert not vision_lora_hits, f"Vision tower got LoRA-wrapped, contradicting 'frozen vision tower': {vision_lora_hits}"
    print(f"Confirmed: 0 vision-tower modules were LoRA-wrapped (checked {sum(1 for _ in m.named_modules())} total modules) — vision tower is genuinely frozen.")
    return m

print("Defined _attach_lora_and_maybe_resume() -- not applied to any model yet.")


In [ ]:
# Cell 7: Collate function — turns each chat example into real model inputs (actual
# image pixels via the processor, not a text placeholder) with the prompt portion masked
# out of the loss so the model is only trained to produce the assistant's answer.
#
# Honesty correction (2026-08-25): the ORIGINAL version of this cell had exactly this
# docstring claim ("prompt portion masked out of the loss"), but the code underneath it
# only did `labels[labels == pad_token_id] = -100` — that masks PADDING, not the prompt.
# Real consequence, caught from the user's actual Kaggle training numbers: loss at step 10
# was 13.37, at step 20 was 8.34 — both suspiciously close to ln(vocab_size) ≈ 11.9 for
# this tokenizer (~151,645 tokens), which is the loss a model gets for pure random
# guessing. That match isn't a coincidence: with the prompt unmasked, the model was being
# scored on predicting the user's question text AND every image placeholder token (which
# vary in count per image and aren't real "next-token" targets), not just the real
# assistant answer — inflating the loss toward the random-guess baseline instead of a
# normal SFT curve (which should start lower, around 1.5-3.5, and trend down).
#
# Fixed properly here: for each example, tokenize the PROMPT-ONLY portion (system + user
# turn + image, with the generation prompt, but no assistant answer) separately to get its
# exact real token length for THIS example — image-token counts vary per image/resolution,
# so this has to be computed per-example, not assumed shared. Everything up to that length
# is masked to -100 in the labels, so only the real assistant-answer tokens contribute to
# the loss, which is what this cell always claimed to do.
#
# Resolution/scale-jitter augmentation (2026-08-27 addition — report.tex Sec 3.5
# "Domain gap to the judging dataset" mitigation item #1): the real judging imagery
# (Cartosat-2S/RISAT) will differ in effective ground-sample-distance and sensor noise
# characteristics from VRSBench's DOTA/DIOR-sourced training images (see report.tex Sec
# 3.4) — a real, disclosed, previously-UNMITIGATED risk. Implemented here, not as a
# separate preprocessing pass, so every epoch sees fresh random jitter rather than one
# fixed augmented copy: for a random 50% of examples per batch, the image is downsampled by
# a random factor (0.35x-1.0x of its original size) and upsampled back to its ORIGINAL
# pixel dimensions before being handed to the processor. Downsample-then-upsample-to-the-
# SAME-final-size is deliberate: it changes effective sharpness/detail (simulating a
# coarser real sensor GSD) WITHOUT changing final image dimensions or any relative object
# position, so referring-expression box labels (normalized 0-100 against image width/
# height) stay exactly valid for every example, including the newly-rebalanced [refer]
# examples from Cell 3 — a crop-based jitter would have shifted true box positions and
# needed label recomputation, which this deliberately avoids. Stated honestly: this is a
# cheap, generic blur/detail proxy for a real sensor-domain gap, not a calibrated
# radiometric simulation of Cartosat-2S/RISAT's actual noise model — a real improvement
# over doing nothing, not a claim that the domain gap is now closed (report.tex Sec 3.5
# still discloses it as an open risk).
#
# Pillow version guard (senior-engineer recheck, 2026-08-27): Pillow >=10 REMOVED the old
# bare `Image.BILINEAR` constant in favor of `Image.Resampling.BILINEAR` — this notebook has
# never actually run with the augmentation below, so hardcoding the old bare constant would
# be a real, untested crash risk on whatever Pillow version Kaggle's current image ships
# (could burn real setup time discovering an AttributeError only after the GPU is already
# allocated). `_RESAMPLE` below resolves to the right one either way, checked once at import
# time rather than guessed.
import random
import re
import torch

_RESAMPLE = getattr(Image, "Resampling", Image).BILINEAR

def _maybe_scale_jitter(img, prob=0.5, min_factor=0.35):
    if random.random() > prob:
        return img
    w, h = img.size
    factor = random.uniform(min_factor, 1.0)
    small = img.resize((max(1, int(w * factor)), max(1, int(h * factor))), _RESAMPLE)
    return small.resize((w, h), _RESAMPLE)

# Coordinate-token loss weighting (2026-09-04 addition -- report.tex Sec 5.2 mitigation
# item #3 / the grounding-accuracy gap disclosed in data/vrsbench_accuracy_eval.json):
# real IoU evidence showed Acc@0.5 = 0.00 on 50 held-out referring-expression examples
# even after the coordinate-order fix. Plain uniform next-token cross-entropy treats
# every wrong prediction equally -- predicting "<96>" instead of "<95>" is penalized
# exactly as much as predicting "<40>" instead of "<95>" -- so the loss signal never
# specifically rewards a TIGHTER box prediction over a merely-plausible one. This does
# not fix that metric-vs-loss mismatch on its own (a real IoU-aware/regression loss would
# be a bigger change than this notebook makes elsewhere), but it does increase the real
# gradient signal on the tokens that matter most for this specific gap: for any training
# example whose real assistant response is (almost) entirely one or more real
# "{<n><n><n><n>}" box tokens -- VRSBench's own [refer]-type answer convention, confirmed
# in backend/models/geochat_service.py's parse_boxes_from_text docstring and this cell's
# own box-token training-data audit above -- every real (non-masked) label token in that
# example gets its loss upweighted by COORD_LOSS_WEIGHT below. Detected directly from the
# response TEXT here, not from a task_tag field, because to_chat_example (the next real
# cell) does not carry task_tag forward into the "messages" dict this collate_fn receives
# -- threading a new field through would touch more cells than this fix needs to. Left at
# 1.0x (unweighted, i.e. mathematically identical to before this change) for every VQA/
# captioning example, so those aren't perturbed at all.
#
# Honesty note: this has not been run on real Kaggle GPU time yet. Run with SMOKE_TEST=
# True first (Cell 3) and confirm training loss still starts and trends the way it did
# before this change (see this cell's own 2026-08-25 honesty note above on what a broken
# loss computation looks like -- a value near ln(vocab_size) is the red flag) before
# trusting this on a real multi-hour run.
COORD_LOSS_WEIGHT = 3.0
# Updated 2026-09-12 (paired with Cell 3/4's box-format switch to Qwen3-VL's native
# "bbox_2d" JSON convention): matches one-or-more real '{"bbox_2d": [x1, y1, x2, y2]}'
# JSON objects, the NEW target format emitted by Cell 3's convert_legacy_box_tokens() and
# finetuning/bigearthnet_txt_filter.py's box_str_to_token() -- not the OLD
# '{<x1><y1><x2><y2>}' bracket format this regex matched before. Kept intentionally
# strict (whole-response-is-box-tokens only, same as before) so COORD_LOSS_WEIGHT only
# ever fires on a real box-only answer, not a box mentioned mid-sentence.
_BOX_ONLY_RESPONSE_RE = re.compile(
    r'^\s*(?:\{"bbox_2d":\s*\[\s*\d{1,4}\s*,\s*\d{1,4}\s*,\s*\d{1,4}\s*,\s*\d{1,4}\s*\]\}\s*)+$'
)

def _is_box_only_response(messages):
    """True if the assistant turn's real text content is (almost) purely one or more real
    '{<n><n><n><n>}' box tokens -- see COORD_LOSS_WEIGHT's note above for why this is
    checked from the text itself rather than a task_tag field."""
    for m in messages:
        if m.get("role") == "assistant":
            for part in m.get("content", []):
                if part.get("type") == "text":
                    return bool(_BOX_ONLY_RESPONSE_RE.match(part.get("text", "")))
    return False

def collate_fn(examples):
    # Right-padding is required for the prefix-length masking below to line up with the
    # real token positions — explicit here rather than trusting the checkpoint's default
    # (some decoder-only tokenizers default to left-padding for generation).
    processor.tokenizer.padding_side = "right"

    texts, images, prompt_lens, is_box_only = [], [], [], []
    for ex in examples:
        imgs = [_maybe_scale_jitter(Image.open(p).convert("RGB")) for p in ex["images"]]
        images.append(imgs)
        texts.append(processor.apply_chat_template(ex["messages"], tokenize=False))
        is_box_only.append(_is_box_only_response(ex["messages"]))

        prompt_only_messages = [m for m in ex["messages"] if m["role"] != "assistant"]
        prompt_text = processor.apply_chat_template(
            prompt_only_messages, tokenize=False, add_generation_prompt=True
        )
        prompt_inputs = processor(text=[prompt_text], images=[imgs], return_tensors="pt")
        prompt_lens.append(prompt_inputs["input_ids"].shape[1])

    batch = processor(text=texts, images=images, return_tensors="pt", padding=True)
    labels = batch["input_ids"].clone()
    labels[labels == processor.tokenizer.pad_token_id] = -100
    for i, plen in enumerate(prompt_lens):
        labels[i, :plen] = -100
    batch["labels"] = labels

    # loss_weights: same shape as labels, 1.0 everywhere except real (non-masked) tokens
    # of a real [refer]-type box-only example, which get COORD_LOSS_WEIGHT. Popped out of
    # the batch by CoordWeightedSFTTrainer.compute_loss (next real cell) before the model
    # forward pass -- passing it straight to model(**batch) would raise a real TypeError,
    # since Qwen3VLForConditionalGeneration.forward() has no such kwarg (verified
    # against the base model actually in use as of this v4 revision, see Cell 4's note).
    loss_weights = torch.ones_like(labels, dtype=torch.float)
    for i, box_only in enumerate(is_box_only):
        if box_only:
            loss_weights[i, labels[i] != -100] = COORD_LOSS_WEIGHT
    batch["loss_weights"] = loss_weights

    return batch


In [ ]:
# Cell 8: Train — pushing the real adapter to the Hub after every checkpoint, not just at
# the very end. (HF login + the resume-from-Hub-checkpoint check now live in Cell 6, since
# the resume decision has to happen BEFORE the LoRA adapter is built, not here.)
#
# Honesty note (2026-08-26): a real Kaggle run completed a full, healthy 150-step training
# (final loss ~1.01, a genuinely good real result) — then the Kaggle session died
# completely (kernel AND local disk both reset) before the user got to run the old Cell 9,
# which was the ONLY place this notebook saved or pushed anything. Two hours of real
# training were lost because the trained adapter existed only in local memory/disk until a
# separate manual step ran afterward. Free-tier Kaggle sessions can disconnect without
# warning on a run this long — depending on "save it all at the very end" isn't safe here.
#
# Fixed properly: this cell now pushes the adapter to a private Hub repo after every
# checkpoint (`save_strategy="steps"`, `hub_strategy="every_save"`).
#
# v3 REVISED (2026-09-12, same day -- deadline extended, re-targeted for quality):
# num_train_epochs restored 2 -> back to v2's proven value -- and NUM_EXAMPLES raised to
# up to 100,000 VRSBench + up to 150,000 BigEarthNet.txt (Cell 3/5b), LoRA rank doubled
# to 32 (Cell 7). See Cell 3's v3-revised note for the full real GPU-hour math (~833h
# estimated, ~4-4.5 weeks across a 6-7-account rotating relay). _SAVE_EVAL_STEPS raised
# 250 -> 2,000 to keep checkpoint count/Hub-push overhead sane given the ~8x larger step
# count (up to 62,500 vs 7,500).
#
# Scale-up, resized TWICE (2026-08-27 — senior-engineer recheck requested explicitly to
# avoid wasting real GPU time): NUM_EXAMPLES raised 600 -> 6,000 in Cell 3, sized against
# the user's real, twice-corrected quota (111h31m misheard -> 11h31m -> 30h real). At the
# ~48s/step real observed rate (see Cell 3), 6,000 examples / (batch_size=1 x
# grad_accum=8) x 2 epochs = 1,500 steps x ~48s = an ESTIMATED ~20.0 real GPU-hours,
# deliberately leaving ~10h (~33%) of the 30h budget unspent as a safety buffer, not
# maximized against the full quota (see Cell 3's note for the full reasoning).
# num_train_epochs kept at 2 (not 4) — with 10x more UNIQUE real examples per epoch, 2
# passes over fresh data is a more efficient use of GPU-hours than repeating a smaller set
# 4x. `save_steps`/`eval_steps` set to 100 (~15 checkpoints across 1,500 steps) to bound a
# mid-session disconnect's real loss to at most ~100 steps (~1.3h), while keeping Hub-push
# overhead modest.
#
# TimeBudgetGuard (new, 2026-08-27): the whole point of the "recheck like a senior ML
# engineer" ask was to not find out 15 hours into a bad estimate that the real per-step
# rate was wrong. This callback measures the REAL rate from the first 20 real steps of
# THIS run and prints an explicit estimate + warning within minutes, instead of trusting
# the ~48s/step figure above blindly for the full ~20h.
#
# Honesty note carried over (2026-08-26): more LoRA capacity (Cell 6) on top of more real
# examples still needs a real overfitting check, not just a training-loss-only view.
# `eval_dataset` (VRSBench's own separate, genuinely-unseen Images_val.zip, built in Cell
# 3/5) is evaluated every `eval_steps` steps, and `load_best_model_at_end=True` means the
# adapter pushed at the end of THIS session is whichever checkpoint scored best on that
# real held-out data within this session (see Cell 6's honesty note on what "resume" does
# and doesn't carry across separate Kaggle sessions).
# v5 note (2026-09-19): num_train_epochs/NUM_EXAMPLES/LoRA rank are UNCHANGED from v4 -- 
# same data composition and capacity, only Cell 4's base model differs -- so this cell's
# existing step-count math (up to 62,500 steps) still applies. What changes is s/step:
# 8B is roughly ~2x the per-step compute of 4B, so expect a real rate meaningfully higher
# than v4's measured 98.0s/step -- TimeBudgetGuard below will report the real number
# within the first ~20 real steps of THIS run; do not assume the v4 number carries over.
import time
from transformers import TrainerCallback
from trl import SFTConfig, SFTTrainer


class TimeBudgetGuard(TrainerCallback):
    """Real early-warning check: estimates real wall-clock time for the whole run from the
    first `check_at_step` real steps and prints a clear warning if that extrapolates past
    `safe_budget_hours`, so a bad throughput estimate gets caught in minutes, not hours.
    This does NOT stop training automatically (a false-positive warning on a slow-starting
    step shouldn't kill a real run unattended) — it prints, loudly, so a human watching the
    Kaggle session can decide to interrupt it (Run > Interrupt) if the number looks wrong."""

    def __init__(self, safe_budget_hours=24.0, check_at_step=20):
        self.safe_budget_hours = safe_budget_hours
        self.check_at_step = check_at_step
        self.start_time = None
        self.warned = False

    def on_train_begin(self, args, state, control, **kwargs):
        self.start_time = time.time()

    def on_step_end(self, args, state, control, **kwargs):
        # Real fix (2026-09-13, paired with multi-GPU DDP support below): under DDP,
        # every process runs this same callback -- without this guard, a 2-GPU run
        # would print this exact message twice (once per process), which is
        # confusing, not wrong. is_world_process_zero is a genuine transformers
        # TrainerState field for exactly this. Single-GPU runs are unaffected (that
        # field is always True there).
        if not state.is_world_process_zero:
            return
        if self.warned or state.global_step < self.check_at_step:
            return
        elapsed = time.time() - self.start_time
        real_secs_per_step = elapsed / state.global_step
        total_steps = state.max_steps if getattr(state, "max_steps", 0) else state.global_step
        est_total_hours = (real_secs_per_step * total_steps) / 3600
        print(f"\n[TIME BUDGET CHECK] After {state.global_step} real steps: "
              f"{real_secs_per_step:.1f}s/step measured (Cell 3's estimate assumed ~48s/step) "
              f"-> estimated total run time {est_total_hours:.2f}h for {total_steps} steps.")
        if est_total_hours > self.safe_budget_hours:
            print(f"  WARNING: this exceeds the {self.safe_budget_hours}h safe budget out of "
                  f"the real ~30h quota. Consider stopping this run (Kaggle: Run > Interrupt) "
                  f"and lowering NUM_EXAMPLES/num_train_epochs in Cell 3/8 before restarting, "
                  f"rather than letting it run until quota runs out mid-training.")
        else:
            print(f"  Within the {self.safe_budget_hours}h safe budget — continuing.")
        self.warned = True


# Real bug caught on this same recheck (2026-08-27): with the fixed save_steps/eval_steps
# below (100) and SMOKE_TEST's tiny 48-example pool (Cell 3) at 2 epochs / 8-example steps
# = only 12 total steps, eval/save/push would NEVER fire during a smoke run (100 > 12) --
# silently defeating the whole stated purpose of SMOKE_TEST ("exercise... one real eval
# pass, one real checkpoint save, and one real Hub push"). Scaled down here so the smoke
# run actually exercises every part of the pipeline it claims to.
_SMOKE = globals().get("SMOKE_TEST", False)
_SAVE_EVAL_STEPS = 5 if _SMOKE else 2000

# Real multi-GPU DDP support (2026-09-13): torch.cuda.device_count() is safe to call
# here (does not itself initialize a CUDA context -- accelerate relies on this same
# fact internally). On Kaggle's "GPU T4 x2" accelerator this is 2; on a plain "GPU T4"
# session (and the existing, already-validated smoke-test runs) this is 1.
NUM_GPUS = max(1, torch.cuda.device_count())
_BASE_EFFECTIVE_BATCH = 8  # per_device_train_batch_size(1) x original grad_accum(8) -- the
# single-GPU value this whole notebook (LoRA rank, learning rate, step-count math in
# Cell 3/8's comments) was tuned against.
_PER_DEVICE_BATCH = 1
_GRAD_ACCUM = max(1, _BASE_EFFECTIVE_BATCH // (_PER_DEVICE_BATCH * NUM_GPUS))
# On 1 GPU: _GRAD_ACCUM = 8 (identical to before this change). On 2 GPUs: _GRAD_ACCUM = 4,
# so the EFFECTIVE GLOBAL batch stays 1*4*2 = 8 -- same as before -- which is what keeps
# the 2-GPU path a genuine wall-clock speedup with training dynamics essentially
# unchanged, rather than silently doubling the effective batch size.
print(f"NUM_GPUS={NUM_GPUS}, gradient_accumulation_steps={_GRAD_ACCUM} "
      f"(effective global batch = {_PER_DEVICE_BATCH * _GRAD_ACCUM * NUM_GPUS}, "
      f"same as this notebook's original single-GPU tuning).")

# Real crash fix + future-proofing (2026-09-19, from an actual Kaggle traceback):
# `SFTConfig.__init__() got an unexpected keyword argument 'warmup_ratio'` happened
# because Cell 2's unpinned `-U trl` resolved to a version whose SFTConfig didn't
# carry that field yet. Cell 2 now pins a version that does, but rather than trust
# pinning alone (Kaggle base images/pip resolvers can still surprise you), this
# builds the desired config as a plain dict and only passes through the keys the
# INSTALLED SFTConfig actually accepts -- introspected live via inspect.signature,
# not assumed. Anything dropped is printed loudly (never silently), so a future
# trl release that renames/removes a field degrades to "ran without that one knob"
# instead of crashing the whole run again.
import inspect

_desired_sft_kwargs = dict(
    output_dir="qwen3vl_vrsbench_lora_ckpt",
    num_train_epochs=2,
    per_device_train_batch_size=_PER_DEVICE_BATCH,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=_GRAD_ACCUM,
    learning_rate=2e-4,
    # warmup_ratio (2026-09-19 addition): real, evidence-based guidance (a fine-tuning
    # hyperparameter survey across many LoRA/QLoRA runs) recommends ~0.1 as a good
    # default -- values of 0.05/0.1/0.25 performed similarly in that survey, but 0.0
    # (this notebook's previous, unset default) risks early-training instability from
    # the full learning rate applying from step 1. Left lr_scheduler_type at its
    # default ("linear") rather than switching to cosine -- the SAME survey found
    # linear-with-warmup matches or slightly beats cosine in practice, so this is not
    # a guess. weight_decay is deliberately left at its default 0.0 too -- the same
    # guidance says only add it if real eval_loss is seen rising while training loss
    # keeps falling (this notebook already tracks eval_loss via load_best_model_at_end
    # above, so that signal is genuinely monitored, not ignored).
    warmup_ratio=0.1,
    logging_steps=10,
    # Real speedup win (2026-09-13, zero accuracy risk): let image decoding/
    # collation for the NEXT batch happen on CPU worker processes while the GPU
    # is still busy on the CURRENT batch, instead of doing it single-threaded and
    # blocking the GPU. Same exact examples reach the model in the same order per
    # optimizer step -- this only overlaps IO/CPU work with GPU compute, it does
    # not change any actual training data or math. 2 workers is conservative for
    # Kaggle's CPU count; raise if TimeBudgetGuard still shows GPU utilization
    # gaps after this.
    dataloader_num_workers=2,
    dataloader_pin_memory=True,
    eval_strategy="steps",
    eval_steps=_SAVE_EVAL_STEPS,
    save_strategy="steps",
    save_steps=_SAVE_EVAL_STEPS,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="loss",
    greater_is_better=False,
    bf16=True,
    remove_unused_columns=False,
    gradient_checkpointing=True,
    report_to=[],
    loss_type="nll",
    # NEFTune (2026-09-19 addition, arXiv:2310.05914): adds noise to the training
    # embeddings on the forward pass only -- a verified, widely-reproduced technique
    # for measurably improving instruction-tuning quality, zero extra inference cost
    # (NEFTune only ever applies during training), negligible training cost. 5 is the
    # paper's own commonly-used default alpha; TRL's SFTConfig supports this natively.
    neftune_noise_alpha=5,
    push_to_hub=True,
    hub_model_id=HF_REPO_ID,
    hub_strategy="checkpoint",  # fixed 2026-08-30: also pushes a full resumable
    # "last-checkpoint" (weights + optimizer + scheduler + real step count) after every
    # save, not just the bare adapter weights -- see Cell 6's honesty note on why the
    # old "every_save" alone meant every Kaggle restart re-ran the whole pass from step 0.
    hub_private_repo=True,
)

_accepted_params = set(inspect.signature(SFTConfig.__init__).parameters.keys())
_filtered_sft_kwargs = {k: v for k, v in _desired_sft_kwargs.items() if k in _accepted_params}
_dropped_sft_kwargs = {k: v for k, v in _desired_sft_kwargs.items() if k not in _accepted_params}
if _dropped_sft_kwargs:
    print(f"WARNING: the installed trl's SFTConfig does not accept these keys -- "
          f"running WITHOUT them rather than crashing (their intended values were "
          f"{_dropped_sft_kwargs}): {sorted(_dropped_sft_kwargs.keys())}")
sft_config = SFTConfig(**_filtered_sft_kwargs)

# CoordWeightedSFTTrainer (2026-09-04, pairs with collate_fn's loss_weights above): TRL's
# SFTTrainer.compute_loss (verified directly against the installed trl/trainer/
# sft_trainer.py source, not guessed) does a real amount of its own setup before the loss
# (MoE aux-loss routing, Liger-kernel handling, a clearer OOM/truncation error message,
# entropy logging) and then delegates to the base Trainer's compute_loss. This subclass
# does NOT bypass any of that: it calls super().compute_loss(..., return_outputs=True) to
# get the real logits exactly as TRL would compute them, and only replaces the FINAL loss
# reduction with a per-token-weighted version -- everywhere loss_weights is 1.0 (every
# non-[refer] example, and the padding/prompt positions already masked to -100) this is
# mathematically identical to TRL's own unweighted loss; it only changes the [refer]
# examples' coordinate tokens. loss_weights is popped from `inputs` before the super()
# call, since passing it through to model.forward() would raise a real TypeError
# (Qwen2_5_VLForConditionalGeneration.forward() has no such kwarg).
#
# Honesty note: this subclass has not been run on real Kaggle GPU time yet -- run
# SMOKE_TEST=True first (Cell 3) and confirm training loss still starts and trends
# normally (not near ln(vocab_size) -- see this notebook's own 2026-08-25 note on what a
# broken loss computation looks like) before trusting this on a real multi-hour run.
class CoordWeightedSFTTrainer(SFTTrainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        loss_weights = inputs.pop("loss_weights", None)
        if loss_weights is None:
            # No weights on this batch (shouldn't happen with collate_fn above) -- defer
            # entirely to TRL's own standard loss rather than guessing one up.
            return super().compute_loss(
                model, inputs, return_outputs=return_outputs, num_items_in_batch=num_items_in_batch
            )

        labels = inputs.get("labels")
        loss, outputs = super().compute_loss(
            model, inputs, return_outputs=True, num_items_in_batch=num_items_in_batch
        )
        if labels is None or not hasattr(outputs, "logits") or outputs.logits is None:
            # Real fallback (e.g. Liger-kernel skip_logits path skipped materializing
            # logits) -- defer to TRL's own already-computed loss rather than crash.
            return (loss, outputs) if return_outputs else loss

        logits = outputs.logits
        shift_logits = logits[..., :-1, :].contiguous()
        shift_labels = labels[..., 1:].contiguous()
        shift_weights = loss_weights[..., 1:].contiguous()

        loss_fct = torch.nn.CrossEntropyLoss(reduction="none", ignore_index=-100)
        per_token_loss = loss_fct(
            shift_logits.view(-1, shift_logits.size(-1)),
            shift_labels.view(-1),
        )
        flat_weights = shift_weights.reshape(-1).to(per_token_loss.dtype)
        valid = (shift_labels.view(-1) != -100).to(per_token_loss.dtype)
        # Same denominator convention as HF's own default reduction (mean over all real,
        # non-masked tokens in this micro-batch) -- not a new source of drift from what
        # was already in use, just weighted before the mean instead of uniform.
        denom = valid.sum().clamp(min=1)
        weighted_loss = (per_token_loss * flat_weights * valid).sum() / denom

        return (weighted_loss, outputs) if return_outputs else weighted_loss


# Real restructuring (2026-09-13, for genuine multi-GPU DDP support): model load +
# LoRA attach + Trainer + train() + final save/push all now happen inside ONE
# function, so it can be launched either directly (1 GPU -- identical to the
# already-validated single-GPU/smoke-test path) or via accelerate.notebook_launcher
# across N GPU processes (Kaggle "GPU T4 x2"). Under notebook_launcher, `model`/
# `trainer` only exist inside each spawned process's own memory -- they are NOT
# available as globals in this notebook's main process afterward, which is why Cell
# 9's old final-save-and-push code and Cell 10's sanity check were both updated to
# not depend on them (see those cells' own notes).
def train_fn():
    if NUM_GPUS > 1:
        from accelerate import PartialState
        device_map = {"": PartialState().process_index}
    else:
        device_map = "auto"
    base_model = _load_base_model(device_map)
    model = _attach_lora_and_maybe_resume(base_model)

    trainer = CoordWeightedSFTTrainer(
        model=model,
        args=sft_config,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        data_collator=collate_fn,
        callbacks=[TimeBudgetGuard(safe_budget_hours=24.0, check_at_step=(3 if _SMOKE else 20))],
    )
    trainer.train(resume_from_checkpoint=RESUME_CHECKPOINT_PATH)

    # Guarded to the main process only (trainer.is_world_process_zero() is a genuine
    # Trainer method) -- under 2-GPU DDP, both processes finish training, but only one
    # of them should print this / do the final save+push (the automatic per-checkpoint
    # Hub pushes during training via hub_strategy="checkpoint" are ALREADY correctly
    # rank-0-only, that's built into transformers' Trainer itself -- this guard is only
    # needed for the manual code below, which Trainer doesn't know about).
    if trainer.is_world_process_zero():
        print(f"Real adapter checkpoints were pushed to https://huggingface.co/{HF_REPO_ID} "
              f"throughout training (every {_SAVE_EVAL_STEPS} steps, including a full resumable "
              f"'last-checkpoint' folder) — even if this session had died mid-run, the most recent "
              "full checkpoint would already be safe on the Hub. If your Kaggle session's "
              "continuous-runtime limit is hit before this finishes, just start a NEW Kaggle "
              "session and re-run this whole notebook from Cell 1 — Cell 6 will detect the full "
              "checkpoint on the Hub, download it, and this line's resume_from_checkpoint will "
              "continue from the exact real step it left off at (not restart the whole pass — see "
              "Cell 6's honesty note for the fix and its one caveat about not changing the "
              "training data composition mid-run).")
        print("load_best_model_at_end=True already reloaded the checkpoint with the lowest REAL "
              "held-out eval loss (within THIS session) into `model` above — check the eval_loss "
              "column in the printed training log to see how it actually trended (should trend "
              "down like the training loss; if it starts climbing while training loss keeps "
              "falling, that's real overfitting, and the reload above already protected you from "
              "shipping that step).")

        # Folded in from the old Cell 9 (2026-09-13): belt-and-suspenders final save/push on
        # top of the during-training Hub pushes, in case this same process is still alive
        # right after training. Moved here (from a separate notebook cell) because `model`/
        # `trainer` only exist in this function's local scope now, not as notebook globals.
        FINAL_DIR = "qwen3vl_vrsbench_lora_final"
        model.save_pretrained(FINAL_DIR)
        processor.save_pretrained(FINAL_DIR)
        print(f"Saved real trained adapter to ./{FINAL_DIR}")
        model.push_to_hub(HF_REPO_ID, private=True)
        processor.push_to_hub(HF_REPO_ID, private=True)
        print(f"Pushed final real adapter to https://huggingface.co/{HF_REPO_ID}")


# Real speedup (2026-09-13, zero accuracy risk if this all works as designed -- see this
# notebook file's own honest caveat in the edit script that produced this cell): launch
# real synchronous DDP training across every visible GPU when there's more than one
# (Kaggle's "GPU T4 x2"), otherwise train directly in this single process -- the exact
# same already-validated path as before this change.
if NUM_GPUS > 1:
    from accelerate import notebook_launcher
    print(f"Detected {NUM_GPUS} GPUs -- launching real synchronous DDP training across "
          f"all of them via accelerate.notebook_launcher. This has NOT been run on real "
          f"Kaggle T4x2 hardware yet -- watch the first few minutes of output closely "
          f"(same red flags this notebook already tracks: loss near ln(vocab_size), a "
          f"crash instead of clean startup) before trusting a real multi-hour run.")
    notebook_launcher(train_fn, num_processes=NUM_GPUS)
else:
    print("Single GPU detected -- training directly in this process (unchanged from the "
          "already-validated smoke-test path).")
    train_fn()


In [ ]:
# Cell 9: this cell is now a NO-OP (2026-09-13). The final save + push used to live
# here, but under the new multi-GPU DDP path (see Cell 8), `model`/`trainer` only
# exist inside train_fn()'s local scope in whichever process(es) actually trained --
# referencing them here as notebook globals would raise a real NameError whenever
# training ran across 2+ GPUs (and would silently just re-save/re-push a stale
# `model` even on 1 GPU, since train_fn already did this exact save+push at the end
# of Cell 8, guarded to the main process). Left in place, not deleted, only so the
# cell numbering referenced in this notebook's other cells/comments doesn't shift.
print("Final save + push already happened inside Cell 8's train_fn() above (see its "
      "own printed confirmation) -- nothing to do here.")


In [ ]:
# Cell 10: Sanity check — run the freshly trained adapter on one of the REAL, genuinely
# held-out eval images (from eval_manifest / Images_val.zip — never seen during training,
# not the training manifest) and print the answer. This should reference genuine,
# image-specific content for that particular image, not a generic templated sentence — if
# every image gets back the same answer, something upstream isn't actually conditioning on
# the image and needs debugging before this adapter is trusted for the demo.
#
# Honesty correction (2026-08-26): the first real adapter's real output here was a short,
# plausible answer followed by endless repetition (`<1920>`, `><><>`, high-frequency
# Chinese tokens) instead of stopping — a genuine undertraining symptom (see Cell 6/Cell 8
# notes), not a bug in this cell. Fixed at the root via more LoRA capacity + more epochs +
# real held-out eval tracking above. `repetition_penalty` and `no_repeat_ngram_size` are
# added here too, as a defensive safeguard — real generation-time settings that make
# runaway repetition far less likely regardless of training quality, not a substitute for
# the real training fix, and not something that fabricates or improves the actual answer
# content.
#
# Honesty correction #2 (2026-08-26): this cell used to sample from `manifest` (the
# TRAINING set) — a model that had simply memorized its 600 training answers could pass
# that check without actually generalizing. Now sampling from `eval_manifest`, the real
# held-out set from Cell 3/5, so this is a genuine generalization check, not a
# memorization check.
import random
from peft import PeftModel

# Real restructuring (2026-09-13): reload the just-pushed adapter FRESH from the Hub
# instead of reusing the old `model` notebook-global, which no longer exists here
# once Cell 8 may have trained across 2+ separate DDP processes (see Cell 8's note).
# This is also a more honest sanity check either way -- it verifies what actually
# landed on the Hub, not just whatever object happens to still be in this process's
# memory.
print(f"Reloading the just-pushed real adapter from https://huggingface.co/{HF_REPO_ID} "
      f"fresh for this sanity check...")
_sanity_base_model = _load_base_model(device_map="auto")
model = PeftModel.from_pretrained(_sanity_base_model, HF_REPO_ID)

sample = random.choice(eval_manifest)
image = Image.open(sample["image_path"]).convert("RGB")
messages = [{"role": "user", "content": [
    {"type": "image"},
    {"type": "text", "text": sample["prompt"]},
]}]
text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = processor(text=[text], images=[[image]], return_tensors="pt").to(model.device)

with torch.inference_mode():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=128,
        do_sample=False,
        repetition_penalty=1.3,
        no_repeat_ngram_size=3,
    )
generated = output_ids[:, inputs["input_ids"].shape[1]:]
answer = processor.batch_decode(generated, skip_special_tokens=True)[0]

print("Real HELD-OUT VRSBench prompt (image never seen in training):", sample["prompt"])
print("Real VRSBench ground-truth answer:", sample["response"])
print("Model's answer:", answer)

In [ ]:
# Cell 11 (new, 2026-09-19): Real quantitative held-out evaluation across all THREE
# newly-added data sources -- not just one anecdotal sample. This directly closes a real
# gap: this project's own PRD explicitly calls for "a measurable before/after difference
# to show evaluators" (Section 2C-i) and a benchmark script over VRSBench/RSVQA/CDVQA
# (Phase 4, build_benchmark.py) -- until now, no such quantitative check existed for this
# run; data was being added without measuring whether it actually helps. This cell is
# that check, built directly from data already loaded/downloaded earlier in this exact
# notebook run (VRSBench's real held-out eval_manifest from Cell 3/5, plus the real Val
# splits of CDVQA and RSVQA-LR -- which Cells 5c-setup/5d-setup already downloaded, and
# which were NEVER used for training: cdvqa_prepare.py/rsvqa_prepare.py both build their
# training manifest by iterating splits ["Train", "Val", ...] and stopping once
# --max-examples is reached -- confirmed live that Train alone (65,967 CDVQA / 57,223
# RSVQA-LR records) already exceeds the 20,000/15,000 caps used here, so Val/Test are
# genuinely untouched by training, not a hopeful assumption).
#
# Two metrics reported per source: exact-match (case/whitespace-insensitive) -- the
# standard metric these benchmarks' own papers report -- and a softer token-overlap F1,
# since some real ground-truth answers are short free-text rather than strict
# yes/no/category labels. Honesty note: this is a SMALL quick eval (tens of examples per
# source, not the full test split) -- meant as a genuine directional signal you can quote
# ("v5 scored X% on a real held-out CDVQA slice"), not a claim of exhaustive benchmarking.
# Compare against this project's own previously-disclosed v2 baseline numbers
# (44.0% / 16.0% / 0.3671 -- see report.tex) when writing up the comparison.
import json as _json
import re
import sys
from pathlib import Path

import torch

EVAL_SAMPLES_PER_SOURCE = 30  # keep this small -- it costs real generate() calls, not free


def _normalize(s: str) -> str:
    return re.sub(r"\s+", " ", str(s).strip().lower())


def _token_f1(pred: str, gold: str) -> float:
    pred_toks, gold_toks = _normalize(pred).split(), _normalize(gold).split()
    if not pred_toks or not gold_toks:
        return float(pred_toks == gold_toks)
    common = {}
    for t in pred_toks:
        common[t] = common.get(t, 0) + 1
    overlap = 0
    for t in gold_toks:
        if common.get(t, 0) > 0:
            overlap += 1
            common[t] -= 1
    if overlap == 0:
        return 0.0
    precision = overlap / len(pred_toks)
    recall = overlap / len(gold_toks)
    return 2 * precision * recall / (precision + recall)


def _generate(images, prompt: str) -> str:
    messages = [{"role": "user", "content": [{"type": "image"} for _ in images] +
                 [{"type": "text", "text": prompt}]}]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[images], return_tensors="pt").to(model.device)
    with torch.inference_mode():
        output_ids = model.generate(
            **inputs, max_new_tokens=64, do_sample=False,
            repetition_penalty=1.3, no_repeat_ngram_size=3,
        )
    generated = output_ids[:, inputs["input_ids"].shape[1]:]
    return processor.batch_decode(generated, skip_special_tokens=True)[0]


def _score_source(name: str, examples: list):
    if not examples:
        print(f"[{name}] no real held-out examples available -- skipping (see warning above).")
        return None
    n = min(EVAL_SAMPLES_PER_SOURCE, len(examples))
    sample = examples[:n]
    exact_hits, f1_sum = 0, 0.0
    per_example = []
    for ex in sample:
        try:
            pred = _generate(ex["images"], ex["prompt"])
        except Exception as e:
            print(f"  [{name}] generation failed on one example ({e}) -- counted as wrong, not skipped.")
            pred = ""
        is_exact = _normalize(pred) == _normalize(ex["response"])
        f1 = _token_f1(pred, ex["response"])
        exact_hits += int(is_exact)
        f1_sum += f1
        per_example.append({"prompt": ex["prompt"], "gold": ex["response"], "pred": pred,
                             "exact": is_exact, "f1": round(f1, 3)})
    exact_acc = exact_hits / n
    mean_f1 = f1_sum / n
    print(f"[{name}] real held-out eval on {n} examples: exact-match={exact_acc:.1%}, "
          f"token-F1={mean_f1:.3f}")
    return {"n": n, "exact_match": exact_acc, "token_f1": mean_f1, "examples": per_example}


results = {}

# --- VRSBench: genuine held-out eval_manifest from Cell 3/5 (Images_val.zip) ---
vrsbench_examples = [{"images": [r["image_path"]], "prompt": r["prompt"], "response": r["response"]}
                      for r in eval_manifest[:EVAL_SAMPLES_PER_SOURCE]]
results["VRSBench"] = _score_source("VRSBench (held-out)", vrsbench_examples)

# --- CDVQA: genuine held-out Val split, reusing cdvqa_prepare.py's own verified join/
# image-resolution logic (written to disk by Cell 5c-setup) rather than re-implementing it ---
cdvqa_examples = []
try:
    sys.path.insert(0, ".")
    import cdvqa_prepare
    cdvqa_repo = Path("cdvqa_repo")
    second_images = Path("second_images")
    if cdvqa_repo.exists() and second_images.exists():
        val_records = cdvqa_prepare.load_split(cdvqa_repo, "Val")
        pairs = cdvqa_prepare.find_second_image_root(second_images)
        if val_records and pairs:
            pre_root, post_root = pairs[0]
            for rec in val_records:
                pre_path, post_path = pre_root / rec["file_name"], post_root / rec["file_name"]
                if pre_path.exists() and post_path.exists():
                    cdvqa_examples.append({
                        "images": [str(pre_path), str(post_path)],
                        "prompt": f"[cdvqa] {rec['question']}",
                        "response": str(rec["answer"]),
                    })
                if len(cdvqa_examples) >= EVAL_SAMPLES_PER_SOURCE:
                    break
    else:
        print("CDVQA held-out eval skipped -- cdvqa_repo/second_images not present in this "
              "session (Cell 5c-setup may not have run or the cache download failed).")
except Exception as e:
    print(f"CDVQA held-out eval skipped due to a real error: {e}")
cdvqa_examples = [{"images": e["images"], "prompt": e["prompt"], "response": e["response"]}
                   for e in cdvqa_examples]
results["CDVQA"] = _score_source("CDVQA (held-out Val split)", cdvqa_examples)

# --- RSVQA-LR: genuine held-out Val split, same reuse pattern ---
rsvqa_examples = []
try:
    import rsvqa_prepare
    rsvqa_dest = Path("rsvqa_lr_raw")
    rsvqa_images = Path("rsvqa_real") / "images"
    if rsvqa_dest.exists() and rsvqa_images.exists():
        val_records = rsvqa_prepare.load_split(rsvqa_dest, "Val")
        cache = {}
        for rec in val_records:
            img_path = rsvqa_prepare.find_image_file(rsvqa_images, rec["img_id"], rec["original_name"], cache)
            if img_path is not None:
                rsvqa_examples.append({
                    "images": [str(img_path)],
                    "prompt": f"[rsvqa] {rec['question']}",
                    "response": rec["answer"],
                })
            if len(rsvqa_examples) >= EVAL_SAMPLES_PER_SOURCE:
                break
    else:
        print("RSVQA-LR held-out eval skipped -- rsvqa_lr_raw/rsvqa_real/images not present "
              "in this session (Cell 5d-setup may not have run or the cache download failed).")
except Exception as e:
    print(f"RSVQA-LR held-out eval skipped due to a real error: {e}")
results["RSVQA-LR"] = _score_source("RSVQA-LR (held-out Val split)", rsvqa_examples)

with open("real_eval_results_v5.json", "w") as f:
    _json.dump(results, f, indent=2)
print("\nSaved full per-example results to real_eval_results_v5.json -- pull this file "
      "down after the run (Kaggle Output tab) for the report/PPT; do not hand-copy just "
      "the summary numbers without the underlying examples if the accuracy looks off, "
      "spot-check the actual pred/gold pairs first.")
